# Dataset Exploration — ASSISTments Skill Builder

## Student Personalization Memory Layer

### Purpose

This notebook documents the initial exploration of the raw ASSISTments Skill Builder dataset used for the Student Personalization Memory Layer.

The goal is to fully understand the dataset before performing any cleaning, preprocessing, feature engineering, or model training.

We investigate:

- Dataset size and structure
- Important memory-related fields
- Missing values
- Student, assignment, problem, and skill counts
- Skill distribution
- Correctness distribution
- Attempts, hints, and response-time behaviour
- Invalid and extreme values
- Exact duplicates
- Missing skill information
- Skill ID/name consistency
- Student interaction history
- Assignment history as a possible session proxy
- Interaction ordering using `order_id`
- Repeated opportunity representations
- Multiple skills linked to one interaction
- Availability of natural-language question text
- Candidate behavioural variables such as `first_action`

> Important: The raw dataset is not modified in this notebook. This stage is only exploratory analysis.

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/raw/skill_builder_data.csv")

df = pd.read_csv(
    DATA_PATH,
    encoding="latin1",
    low_memory=False
)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully.
Rows: 525,534
Columns: 30


## 1. Dataset Structure

First, we inspect all columns available in the dataset.

This helps determine which fields are useful for:

- Student identification
- Session/activity grouping
- Problem identification
- Concept-Based Memory
- Correctness tracking
- Attempt behaviour
- Hint behaviour
- Response-time behaviour
- Future learning-state prediction

In [2]:
print(f"Total columns: {len(df.columns)}\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Total columns: 30

1. order_id
2. assignment_id
3. user_id
4. assistment_id
5. problem_id
6. original
7. correct
8. attempt_count
9. ms_first_response
10. tutor_mode
11. answer_type
12. sequence_id
13. student_class_id
14. position
15. type
16. base_sequence_id
17. skill_id
18. skill_name
19. teacher_id
20. school_id
21. hint_count
22. hint_total
23. overlap_time
24. template_id
25. answer_id
26. answer_text
27. first_action
28. bottom_hint
29. opportunity
30. opportunity_original


## 2. Important Memory-Related Columns

We initially focus on fields that may be useful for the Student Personalization Memory Layer.

These include student identity, assignment/activity identity, problem identity, concept information, correctness, attempts, hints, response time, and opportunity information.

In [3]:
important_cols = [
    "user_id",
    "assignment_id",
    "problem_id",
    "skill_id",
    "skill_name",
    "correct",
    "attempt_count",
    "hint_count",
    "hint_total",
    "ms_first_response",
    "opportunity"
]

df[important_cols].info()

print("\nMissing values:")
print(df[important_cols].isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 525534 entries, 0 to 525533
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   user_id            525534 non-null  int64  
 1   assignment_id      525534 non-null  int64  
 2   problem_id         525534 non-null  int64  
 3   skill_id           459208 non-null  float64
 4   skill_name         446844 non-null  str    
 5   correct            525534 non-null  int64  
 6   attempt_count      525534 non-null  int64  
 7   hint_count         525534 non-null  int64  
 8   hint_total         525534 non-null  int64  
 9   ms_first_response  525534 non-null  int64  
 10  opportunity        525534 non-null  int64  
dtypes: float64(1), int64(9), str(1)
memory usage: 44.1 MB

Missing values:
user_id                  0
assignment_id            0
problem_id               0
skill_id             66326
skill_name           78690
correct                  0
attempt_count            

### Initial Missing-Value Observation

The important behavioural fields such as correctness, attempts, hints, and response time are complete.

However, concept information contains missing values.

This issue must be investigated before Concept-Based Memory is created.

No rows are removed at this stage.

## 3. Dataset Population Counts

We inspect the number of unique:

- Students
- Assignments
- Problems
- Skill IDs
- Skill names

In [4]:
print("Unique students:", df["user_id"].nunique())
print("Unique assignments:", df["assignment_id"].nunique())
print("Unique problems:", df["problem_id"].nunique())
print("Unique skill IDs:", df["skill_id"].nunique())
print("Unique skill names:", df["skill_name"].nunique())

Unique students: 4217
Unique assignments: 3521
Unique problems: 26688
Unique skill IDs: 123
Unique skill names: 110


## 4. Available Mathematics Skills

The dataset contains named mathematics concepts.

These concepts are important for Concept-Based Memory and may later help define the canonical topic list used by the Memory Component.

In [5]:
skills = sorted(
    df["skill_name"]
    .dropna()
    .astype(str)
    .unique()
)

print(f"Total named skills: {len(skills)}\n")

for i, skill in enumerate(skills, start=1):
    print(f"{i}. {skill}")

Total named skills: 110

1. Absolute Value
2. Addition Whole Numbers
3. Addition and Subtraction Fractions
4. Addition and Subtraction Integers
5. Addition and Subtraction Positive Decimals
6. Algebraic Simplification
7. Algebraic Solving
8. Angles - Obtuse, Acute, and Right
9. Angles on Parallel Lines Cut by a Transversal
10. Area Circle
11. Area Irregular Figure
12. Area Parallelogram
13. Area Rectangle
14. Area Trapezoid
15. Area Triangle
16. Box and Whisker
17. Calculations with Similar Figures
18. Choose an Equation from Given Information
19. Circle Graph
20. Circumference 
21. Complementary and Supplementary Angles
22. Computation with Real Numbers
23. Congruence
24. Conversion of Fraction Decimals Percents
25. Counting Methods
26. D.4.8-understanding-concept-of-probabilities
27. Distributive Property
28. Divisibility Rules
29. Division Fractions
30. Effect of Changing Dimensions of a Shape Prportionally
31. Equation Solving More Than Two Steps
32. Equation Solving Two or Fewer S

## 5. Skill Distribution

We now check how many interactions are available for each skill.

This is important because highly imbalanced skill frequencies can affect future machine-learning experiments.

In [6]:
skill_counts = df["skill_name"].value_counts()

print(skill_counts.to_string())

print("\nMinimum interactions:", skill_counts.min())
print("Maximum interactions:", skill_counts.max())

skill_name
Equation Solving Two or Fewer Steps                                       24253
Percent Of                                                                22931
Addition and Subtraction Integers                                         22895
Conversion of Fraction Decimals Percents                                  20992
Volume Rectangular Prism                                                  19489
Venn Diagram                                                              18275
Volume Cylinder                                                           13734
Multiplication and Division Integers                                      13159
Area Rectangle                                                            12934
Addition and Subtraction Fractions                                        11334
Area Triangle                                                             10839
Area Trapezoid                                                             9288
Proportion                   

### Skill Imbalance Observation

The dataset is strongly imbalanced.

Some concepts contain tens of thousands of interactions, while a few concepts contain only a very small number.

We will not remove rare skills yet.

The imbalance will be handled later when designing the preprocessing and model-training strategy.

## 6. Correctness Distribution

Correctness is one of the most important learning-behaviour variables.

The dataset represents:

- `0` = incorrect
- `1` = correct

In [7]:
correct_counts = df["correct"].value_counts(dropna=False).sort_index()

print(correct_counts)

print("\nPercentages:")
print(
    (
        df["correct"]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
        * 100
    ).round(2)
)

correct
0    168449
1    357085
Name: count, dtype: int64

Percentages:
correct
0    32.05
1    67.95
Name: proportion, dtype: float64


### Correctness Finding

The dataset contains more correct interactions than incorrect interactions.

Correctness will later contribute to:

- Short-Term Memory
- Long-Term Memory
- Concept-Based Memory
- Learning-state feature engineering
- Learning signal generation

## 7. Attempts, Hints, and Response Time

We inspect descriptive statistics for behavioural variables that may later be used by the Memory Component and Student Learning State Predictor.

In [8]:
behaviour_cols = [
    "attempt_count",
    "hint_count",
    "hint_total",
    "ms_first_response"
]

df[behaviour_cols].describe().round(2)

,attempt_count,hint_count,hint_total,ms_first_response
count,525534.00,525534.00,525534.00,525534.00
mean,1.50,0.44,2.34,47992.17
std,10.58,1.13,1.65,318698.28
min,0.00,0.00,0.00,-7759575.00
25%,1.00,0.00,0.00,7687.00
50%,1.00,0.00,3.00,20791.00
75%,1.00,0.00,3.00,50000.00
max,3824.00,10.00,10.00,84076920.00


### Initial Data-Quality Observation

Some extreme values are visible.

Examples include:

- Very high attempt counts
- Negative response times
- Extremely large response times

These values must be investigated before being used for machine learning.

No values are corrected yet.

## 8. Invalid Response Times

Negative response time is logically impossible, so we count negative, zero, and positive values separately.

In [9]:
response_time = df["ms_first_response"]

print("Negative response times:", (response_time < 0).sum())
print("Zero response times:", (response_time == 0).sum())
print("Positive response times:", (response_time > 0).sum())

Negative response times: 8
Zero response times: 4678
Positive response times: 520848


## 9. Attempt Count Outliers

Attempt count may be useful for measuring how much effort a student needs.

However, very large values may represent unusual behaviour, logging artefacts, or other dataset issues.

In [10]:
attempts = df["attempt_count"]

print("Attempts > 10:", (attempts > 10).sum())
print("Attempts > 50:", (attempts > 50).sum())
print("Attempts > 100:", (attempts > 100).sum())

print("\nLargest values:")
print(attempts.nlargest(20).to_list())

Attempts > 10: 2807
Attempts > 50: 449
Attempts > 100: 148

Largest values:
[3824, 3740, 1853, 1853, 1853, 1631, 1503, 1426, 1265, 1180, 656, 608, 564, 491, 486, 455, 453, 412, 401, 397]


## 10. Response-Time Outliers

We inspect unusually long response times.

These values may later require clipping, transformation, filtering, or another justified preprocessing strategy.

In [11]:
response_time = df["ms_first_response"]

print("> 5 minutes:", (response_time > 300_000).sum())
print("> 10 minutes:", (response_time > 600_000).sum())
print("> 30 minutes:", (response_time > 1_800_000).sum())
print("> 1 hour:", (response_time > 3_600_000).sum())

print("\nLargest values:")
print(response_time.nlargest(10).to_list())

> 5 minutes: 6544
> 10 minutes: 1834
> 30 minutes: 476
> 1 hour: 200

Largest values:
[84076920, 84076920, 74563450, 66158290, 59446217, 44637627, 44637627, 28814407, 25877940, 25877940]


## 11. Exact Duplicate Rows

Before designing a cleaning strategy, we verify whether completely identical rows exist.

In [12]:
print("Total rows:", len(df))
print("Exact duplicate rows:", df.duplicated().sum())

Total rows: 525534


Exact duplicate rows: 0


## 12. Missing Skill ID and Skill Name Relationship

Concept information is essential for Concept-Based Memory.

We investigate whether missing `skill_name` values can be recovered using `skill_id`.

In [13]:
both_missing = (
    df["skill_id"].isna()
    & df["skill_name"].isna()
).sum()

id_exists_name_missing = (
    df["skill_id"].notna()
    & df["skill_name"].isna()
).sum()

name_exists_id_missing = (
    df["skill_id"].isna()
    & df["skill_name"].notna()
).sum()

both_available = (
    df["skill_id"].notna()
    & df["skill_name"].notna()
).sum()

print("Both missing:", both_missing)
print("Skill ID exists, name missing:", id_exists_name_missing)
print("Skill name exists, ID missing:", name_exists_id_missing)
print("Both available:", both_available)

Both missing: 66326
Skill ID exists, name missing: 12364
Skill name exists, ID missing: 0
Both available: 446844


## 13. Can Missing Skill Names Be Recovered?

If a missing-name skill ID appears elsewhere with one clear name, it could potentially be recovered.

We check whether this is possible without guessing.

In [14]:
known_skill_mapping = (
    df.dropna(subset=["skill_id", "skill_name"])
    .groupby("skill_id")["skill_name"]
    .nunique()
)

missing_name_ids = (
    df.loc[
        df["skill_id"].notna()
        & df["skill_name"].isna(),
        "skill_id"
    ]
    .unique()
)

recoverable = sum(
    (
        skill_id in known_skill_mapping.index
        and known_skill_mapping.loc[skill_id] == 1
    )
    for skill_id in missing_name_ids
)

print("Unique skill IDs with missing names:", len(missing_name_ids))
print("Recoverable IDs with one clear name:", recoverable)
print(
    "Unrecoverable/ambiguous IDs:",
    len(missing_name_ids) - recoverable
)

Unique skill IDs with missing names: 12
Recoverable IDs with one clear name: 0
Unrecoverable/ambiguous IDs: 12


### Missing Skill Finding

The missing skill names cannot currently be recovered reliably from the available dataset mappings.

Therefore, no skill names should be guessed or manually invented during preprocessing.

## 14. Skill ID ↔ Skill Name Consistency

For available values, we verify whether one `skill_id` always maps to one `skill_name`.

In [15]:
skill_mapping_counts = (
    df.dropna(subset=["skill_id", "skill_name"])
    .groupby("skill_id")["skill_name"]
    .nunique()
)

print(
    "Skill IDs with exactly one name:",
    (skill_mapping_counts == 1).sum()
)

print(
    "Skill IDs with multiple names:",
    (skill_mapping_counts > 1).sum()
)

print(
    "\nMaximum names for one skill ID:",
    skill_mapping_counts.max()
)

Skill IDs with exactly one name: 111
Skill IDs with multiple names: 0

Maximum names for one skill ID: 1


## 15. Student Interaction History

The Memory Component depends on historical behaviour.

We therefore inspect how many interactions each student has.

In [16]:
student_interaction_counts = df.groupby("user_id").size()

print(student_interaction_counts.describe().round(2))

print(
    "\nStudents with < 5 interactions:",
    (student_interaction_counts < 5).sum()
)

print(
    "Students with < 10 interactions:",
    (student_interaction_counts < 10).sum()
)

print(
    "Students with >= 50 interactions:",
    (student_interaction_counts >= 50).sum()
)

count    4217.00
mean      124.62
std       307.59
min         1.00
25%        10.00
50%        27.00
75%        95.00
max      8214.00
dtype: float64

Students with < 5 interactions: 441
Students with < 10 interactions: 973
Students with >= 50 interactions: 1519


### Student History Observation

Some students have very limited history.

This will be important when building the Student Learning State Predictor.

A state prediction based on only one or two interactions should not be treated with the same confidence as a prediction based on substantial history.

## 16. Assignment History

The existing project documents map `assignment_id` to `session_id`.

However, the raw dataset calls this field an assignment, not a session.

Therefore, we currently treat it only as a **proxy session/activity identifier** for prototype memory experiments.

In [17]:
assignments_per_student = (
    df.groupby("user_id")["assignment_id"]
    .nunique()
)

print(assignments_per_student.describe().round(2))

print(
    "\nStudents with 1 assignment:",
    (assignments_per_student == 1).sum()
)

print(
    "Students with >= 5 assignments:",
    (assignments_per_student >= 5).sum()
)

print(
    "Students with >= 10 assignments:",
    (assignments_per_student >= 10).sum()
)

count    4217.00
mean       15.71
std        34.02
min         1.00
25%         2.00
50%         5.00
75%        12.00
max       369.00
Name: assignment_id, dtype: float64

Students with 1 assignment: 681
Students with >= 5 assignments: 2161
Students with >= 10 assignments: 1296


## 17. Investigating Interaction Ordering

Short-Term Memory and performance-trend modelling require some form of interaction ordering.

The dataset contains `order_id`, so we first investigate whether it can safely represent an interaction.

In [18]:
print("order_id dtype:", df["order_id"].dtype)
print("Missing order_id:", df["order_id"].isna().sum())
print("Unique order_id:", df["order_id"].nunique())

print("\nFirst 10 order_id values:")
print(df["order_id"].head(10).to_list())

order_id dtype: int64
Missing order_id: 0
Unique order_id: 346860

First 10 order_id values:
[33022537, 33022709, 35450204, 35450295, 35450311, 35450555, 35450573, 35480603, 33140811, 33140919]


## 18. Repeated `order_id` Values

`order_id` is not unique across the dataset.

We investigate how often it repeats.

In [19]:
order_counts = df["order_id"].value_counts()

print(
    "Order IDs appearing once:",
    (order_counts == 1).sum()
)

print(
    "Order IDs appearing more than once:",
    (order_counts > 1).sum()
)

print(
    "Maximum rows sharing one order_id:",
    order_counts.max()
)

print("\nMost repeated order IDs:")
print(order_counts.head(10).to_string())

Order IDs appearing once: 296624
Order IDs appearing more than once: 50236
Maximum rows sharing one order_id: 215

Most repeated order IDs:
order_id
36386020    215
36386028    215
36386036    215
36386043    215
36386049    215
36395758    215
36395765    215
36395771    215
36369512    213
36369521    213


## 19. Inspecting a Highly Repeated `order_id`

We inspect one highly repeated order ID to determine why it appears many times.

In [20]:
sample_order_id = 36386020

sample_order = df[
    df["order_id"] == sample_order_id
]

sample_cols = [
    "order_id",
    "user_id",
    "assignment_id",
    "problem_id",
    "skill_id",
    "skill_name",
    "correct",
    "attempt_count",
    "opportunity"
]

sample_order[sample_cols].head(20)

,order_id,user_id,assignment_id,problem_id,skill_id,skill_name,correct,attempt_count,opportunity
295327,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,841
295328,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,842
295329,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,843
295330,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,844
295331,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,845
295332,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,846
295333,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,847
295334,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,848
295335,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,849
295336,36386020,77118,285857,90237,279.0,Multiplication and Division Integers,0,2,850


### Observation

For the inspected `order_id`, student, assignment, problem, skill, correctness, and attempt count remain the same while `opportunity` changes.

This suggests that some repeated rows represent opportunity-level expansions rather than fully independent student interactions.

## 20. Which Columns Change Within the Repeated Order?

We verify exactly which fields vary within the selected repeated `order_id`.

In [21]:
changing_columns = [
    column
    for column in df.columns
    if sample_order[column].nunique(dropna=False) > 1
]

print("Rows:", len(sample_order))

print("\nColumns that change within this order_id:")
print(changing_columns)

Rows: 215

Columns that change within this order_id:
['opportunity', 'opportunity_original']


## 21. Measuring Opportunity-Related Repetition

We compare the raw row count with unique interaction groups based on important interaction fields.

This is only investigation.

No records are removed.

In [22]:
interaction_keys = [
    "order_id",
    "user_id",
    "assignment_id",
    "problem_id",
    "skill_id",
    "correct",
    "attempt_count"
]

interaction_groups = df.drop_duplicates(
    subset=interaction_keys
)

print("Original rows:", len(df))
print("Unique interaction groups:", len(interaction_groups))
print(
    "Extra repeated rows:",
    len(df) - len(interaction_groups)
)

Original rows:

 525534
Unique interaction groups: 401756
Extra repeated rows: 123778


## 22. Is `order_id` Stable for the Base Interaction?

We verify whether one `order_id` ever points to multiple:

- Students
- Assignments
- Problems
- Correctness values
- Attempt counts

In [23]:
order_group = df.groupby("order_id")

print(
    "order_id linked to multiple students:",
    (order_group["user_id"].nunique() > 1).sum()
)

print(
    "order_id linked to multiple assignments:",
    (order_group["assignment_id"].nunique() > 1).sum()
)

print(
    "order_id linked to multiple problems:",
    (order_group["problem_id"].nunique() > 1).sum()
)

print(
    "order_id linked to multiple correctness values:",
    (order_group["correct"].nunique() > 1).sum()
)

print(
    "order_id linked to multiple attempt counts:",
    (order_group["attempt_count"].nunique() > 1).sum()
)

order_id linked to multiple students: 0
order_id linked to multiple assignments: 0
order_id linked to multiple problems: 0
order_id linked to multiple correctness values: 0
order_id linked to multiple attempt counts: 0


### Base Interaction Finding

For each `order_id`, student, assignment, problem, correctness, and attempt count remain stable.

This suggests that `order_id` is useful as a base interaction identifier.

However, we still need to check skill relationships before using it for deduplication.

## 23. Can One `order_id` Contain Multiple Skills?

This is critical for Concept-Based Memory.

If a problem is associated with multiple skills, keeping only one row per `order_id` would destroy valid concept information.

In [24]:
order_group = df.groupby("order_id")

print(
    "order_id with multiple skill IDs:",
    (
        order_group["skill_id"]
        .nunique(dropna=False)
        > 1
    ).sum()
)

print(
    "order_id with multiple skill names:",
    (
        order_group["skill_name"]
        .nunique(dropna=False)
        > 1
    ).sum()
)

print(
    "Maximum skill IDs for one order_id:",
    order_group["skill_id"].nunique().max()
)

print(
    "Maximum skill names for one order_id:",
    order_group["skill_name"].nunique().max()
)

order_id with multiple skill IDs: 47037


order_id with multiple skill names: 46890
Maximum skill IDs for one order_id: 4
Maximum skill names for one order_id: 4


### Multi-Skill Finding

One interaction can be associated with multiple mathematics skills.

Up to four skills were observed for a single `order_id`.

Therefore:

> We must not deduplicate the dataset using `order_id` alone.

Concept associations must be preserved.

## 24. Unique `order_id + skill_id` Combinations

We investigate whether `order_id + skill_id` provides a safer representation for concept-level interaction analysis.

In [25]:
order_skill_unique = df.drop_duplicates(
    subset=["order_id", "skill_id"]
)

print("Original rows:", len(df))

print(
    "Unique order_id + skill_id rows:",
    len(order_skill_unique)
)

print(
    "Repeated opportunity rows removed:",
    len(df) - len(order_skill_unique)
)

print(
    "\nRows with missing skill_id after deduplication:",
    order_skill_unique["skill_id"].isna().sum()
)

Original rows: 525534
Unique order_id + skill_id rows: 401756
Repeated opportunity rows removed: 123778

Rows with missing skill_id after deduplication: 63755


### Concept-Level Interaction Finding

`order_id + skill_id` preserves multi-skill associations while removing many opportunity-level repetitions.

This currently appears more appropriate for Concept-Based Memory than `order_id` alone.

A final preprocessing decision will be made only after all dataset investigation is complete.

## 25. Unique Interactions With No Concept Information

We now inspect interactions where neither skill ID nor skill name is available.

In [26]:
missing_skill_interactions = order_skill_unique[
    order_skill_unique["skill_id"].isna()
]

print(
    "Unique rows with missing skill_id:",
    len(missing_skill_interactions)
)

print(
    "Unique order_ids among them:",
    missing_skill_interactions["order_id"].nunique()
)

print(
    "Skill name available:",
    missing_skill_interactions["skill_name"]
    .notna()
    .sum()
)

print(
    "Skill name also missing:",
    missing_skill_interactions["skill_name"]
    .isna()
    .sum()
)

Unique rows with missing skill_id: 63755
Unique order_ids among them: 63755
Skill name available: 0
Skill name also missing: 63755


### Missing Concept Interpretation

These interactions cannot directly contribute to Concept-Based Memory because the corresponding concept is unknown.

However, they may still contain useful general behavioural evidence for Long-Term Memory.

Therefore, they should not automatically be deleted from the complete interaction history.

## 26. Investigating `answer_text`

The future Topic Classifier requires natural-language mathematics questions.

We inspect `answer_text` to determine whether it contains the original question or only the student's answer.

In [27]:
print(
    "Missing answer_text:",
    df["answer_text"].isna().sum()
)

print(
    "Non-missing answer_text:",
    df["answer_text"].notna().sum()
)

print("\nSample values:")

print(
    df["answer_text"]
    .dropna()
    .astype(str)
    .head(20)
    .to_string(index=False)
)

Missing answer_text: 94784
Non-missing answer_text: 430750

Sample values:
26
55
88
41
65
12
 6
55
12
36
79
18
66
46
47
70
55
82
14
 6


### Question-Text Limitation

The inspected `answer_text` values mainly contain student responses such as numerical answers.

They do not provide the original natural-language mathematics question.

Therefore:

- This ASSISTments CSV is suitable for interaction-memory research.
- It is useful for Student Learning State feature development.
- It is not sufficient by itself for training the free-text Topic Classifier.

A separate suitable question-text dataset/source will be required for the Topic Classification research.

## 27. Inspecting Other Text-Like Columns

We check several other fields to determine whether any contains usable natural-language mathematics questions.

In [28]:
text_like_cols = [
    "tutor_mode",
    "answer_type",
    "type",
    "answer_text",
    "first_action"
]

for column in text_like_cols:
    print(f"\n--- {column} ---")
    
    print(
        df[column]
        .dropna()
        .astype(str)
        .head(15)
        .to_string(index=False)
    )


--- tutor_mode ---
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor
tutor

--- answer_type ---
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra
algebra

--- type ---
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection
MasterySection

--- answer_text ---
26
55
88
41
65
12
 6
55
12
36
79
18
66
46
47

--- first_action ---
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


### Text Availability Finding

The inspected fields do not provide the original natural-language mathematics problem text required for question-to-topic classification.

Topic-classifier dataset preparation must therefore be handled separately from the ASSISTments interaction-memory dataset.

## 28. `first_action` Distribution

`first_action` may potentially describe some aspect of the student's initial behaviour.

Before using it as a feature, we first inspect its distribution.

In [29]:
first_action_counts = (
    df["first_action"]
    .value_counts(dropna=False)
    .sort_index()
)

print(first_action_counts)

print("\nPercentages:")

print(
    (
        df["first_action"]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .sort_index()
        * 100
    ).round(2)
)

first_action
0    479427
1     37419
2      8688
Name: count, dtype: int64

Percentages:
first_action
0    91.23
1     7.12
2     1.65
Name: proportion, dtype: float64


### `first_action` Caution

The dataset contains three `first_action` values: 0, 1, and 2.

However, the exact semantic meaning of these codes has not yet been verified from authoritative dataset documentation.

Therefore, this feature will not be used in modelling until its meaning is confirmed.

# Dataset Exploration Summary

The initial exploration identified several important characteristics of the ASSISTments Skill Builder dataset.

## Confirmed Strengths

The dataset provides useful student-learning interaction variables including:

- Student identity
- Assignment/activity identity
- Problem identity
- Correctness
- Attempt count
- Hint usage
- Response time
- Mathematics skill information for many interactions
- Interaction ordering information

These fields are suitable for developing:

- Short-Term Memory
- Long-Term Memory
- Concept-Based Memory
- Student behavioural features
- Student Learning State modelling

## Important Data-Quality Issues

The exploration also identified:

- Missing skill IDs and names
- Very large attempt-count outliers
- Negative response times
- Zero response times
- Extremely large response times
- Strong skill-frequency imbalance
- Students with very little historical data
- Opportunity-related repeated row representations
- Multiple skill associations for one base interaction

## Important Interaction-Structure Finding

`order_id` appears to identify a base interaction, but one `order_id` may belong to multiple skills.

Therefore, deduplicating only by `order_id` would remove valid concept information.

`order_id + skill_id` currently appears to be a safer concept-level representation.

## Important Topic-Classifier Finding

The current ASSISTments CSV does not provide suitable natural-language mathematics question text.

Therefore, the system will use:

**ASSISTments**
→ Memory development and Student Learning State research

and a **separate question-text dataset/source**
→ Topic Classifier training

## Current Research Stage

No cleaning or preprocessing has yet been applied.

The raw dataset remains unchanged.

The next stage will continue validating the dataset structure and then design a justified preprocessing pipeline before any model training begins.

## 29. Verification Against ASSISTments Dataset Documentation

The exploratory findings were compared with the original ASSISTments Skill Builder dataset documentation.

The documentation confirms that:

- A student-problem interaction may have multiple skill tags.
- In the Skill Builder dataset, those skills can be represented across separate rows.
- The corresponding `opportunity` value is skill-specific.
- Therefore, repeated rows should not automatically be treated as ordinary duplicate interactions.
- Deduplicating only by `order_id` would incorrectly remove valid multi-skill information.

Our raw dataset contains 346,860 unique `order_id` values, which also matches the commonly reported number of base interactions for ASSISTments 2009–2010.

### Current Decision

We will preserve the original raw dataset.

During preprocessing, we will create different representations depending on the purpose:

- Base interaction representation for general student history.
- Skill-aware interaction representation for Concept-Based Memory.

The exact preprocessing keys will be finalized only after the remaining dataset fields and official definitions are verified.

## 30. Investigating `first_action`

The `first_action` field contains three values:

- 0
- 1
- 2

Before using this field as a behavioural feature, we need to understand how it relates to student correctness and hint usage.

We compare:

- `first_action` vs `correct`
- `first_action` vs whether hints were used

At this stage, we do not assign a final meaning to the codes 0, 1, and 2.

In [30]:
print("first_action vs correct:")

print(
    pd.crosstab(
        df["first_action"],
        df["correct"]
    )
)

print("\nfirst_action vs hint usage:")

print(
    pd.crosstab(
        df["first_action"],
        df["hint_count"] > 0
    )
)

first_action vs correct:
correct            0       1
first_action                
0             122342  357085
1              37419       0
2               8688       0

first_action vs hint usage:


hint_count     False  True 
first_action               
0             436943  42484
1                  0  37419
2               8688      0


### Findings

Observed relationships:

- `first_action = 0` contains all correct responses, but also some incorrect responses.
- `first_action = 1` is always associated with hint usage in this dataset.
- `first_action = 2` is always incorrect and has no recorded hint usage.

However, these relationships alone are not enough to define the official meaning of the three codes.

### Decision

`first_action` will not yet be used as a Student Learning State feature.

Its official definition must first be verified from the dataset documentation or another reliable source.

## 31. `first_action` vs `bottom_hint`

We continue investigating `first_action` before deciding whether it is useful for the Student Learning State model.

This check compares `first_action` with `bottom_hint`.

The goal is to better understand whether the different `first_action` codes are related to hint behaviour.

No feature meaning is assumed yet.

In [31]:
print(
    pd.crosstab(
        df["first_action"],
        df["bottom_hint"],
        dropna=False
    )
)

print("\nbottom_hint values:")

print(
    df["bottom_hint"]
    .value_counts(dropna=False)
)

bottom_hint     0.0    1.0     NaN
first_action                      
0             10568  31920  436939
1              9330  28089       0
2                 0      0    8688

bottom_hint values:
bottom_hint
NaN    445627
1.0     60009
0.0     19898
Name: count, dtype: int64


### Interpretation

This output is used only to investigate the relationship between `first_action` and `bottom_hint`.

The final meaning of the `first_action` codes will not be assigned until the official dataset definition is verified.

Therefore, `first_action` is still excluded from the learning-state feature set for now.

## 32. Investigating `hint_count` and `hint_total`

Hint usage is potentially an important behavioural signal for the Student Learning State model.

The dataset contains two related fields:

- `hint_count`
- `hint_total`

Before selecting a feature, we investigate their relationship and distributions.

At this stage, we do not assume that these two fields represent the same information.

In [32]:
print(
    "Rows where hint_count > hint_total:",
    (df["hint_count"] > df["hint_total"]).sum()
)

print(
    "Rows where hint_count == hint_total:",
    (df["hint_count"] == df["hint_total"]).sum()
)

print(
    "Rows where hint_count < hint_total:",
    (df["hint_count"] < df["hint_total"]).sum()
)

print("\nhint_count values:")
print(df["hint_count"].value_counts().sort_index())

print("\nhint_total values:")
print(df["hint_total"].value_counts().sort_index())

Rows where hint_count > hint_total: 0
Rows where hint_count == hint_total: 198308
Rows where hint_count < hint_total: 327226

hint_count values:
hint_count
0     445631
1       8542
2      18065
3      33193
4      14935
5       3678
6       1106
7        383
10         1
Name: count, dtype: int64

hint_total values:
hint_total
0     138551
1       5791
2      91301
3     162596
4      91323
5      28495
6       5209
7       1982
8        280
10         6
Name: count, dtype: int64


### Interpretation

This comparison helps determine whether `hint_count` and `hint_total` contain different behavioural information.

A final feature-selection decision will be made only after examining the output and confirming the meaning of these fields.

No hint-related values are modified during this investigation.

## 33. Investigating Zero Attempt Counts

`attempt_count` is expected to describe student attempt behaviour, but the dataset contains records where:

`attempt_count = 0`

Before using attempt count in the Student Learning State model, we investigate these records.

We examine:

- Number and percentage of zero-attempt records
- Correctness
- `first_action`
- Hint usage

No records are removed or modified.

In [33]:
zero_attempts = df[df["attempt_count"] == 0]

print(
    "Rows with attempt_count = 0:",
    len(zero_attempts)
)

print(
    "Percentage:",
    round(len(zero_attempts) / len(df) * 100, 2)
)

print("\nCorrectness:")
print(
    zero_attempts["correct"]
    .value_counts(dropna=False)
)

print("\nfirst_action:")
print(
    zero_attempts["first_action"]
    .value_counts(dropna=False)
)

print("\nhint_count:")
print(
    zero_attempts["hint_count"]
    .value_counts(dropna=False)
    .sort_index()
)

Rows with attempt_count = 0: 22153
Percentage: 4.22

Correctness:
correct
1    12971
0     9182
Name: count, dtype: int64

first_action:
first_action
0    13204
2     8644
1      305
Name: count, dtype: int64

hint_count:
hint_count
0    21848
1      157
2       56
3       49
4       24
5       13
6        5
7        1
Name: count, dtype: int64


### Interpretation

Zero attempt counts require investigation before preprocessing because they may represent a particular interaction behaviour rather than ordinary answered problems.

The relationship with correctness, hint usage, and `first_action` will help determine how these records should be handled.

No preprocessing decision is made at this stage.

## 34. Are Extreme Attempt Counts Caused by Opportunity Repetition?

Earlier exploration found unusually large `attempt_count` values, including values above 100 and a maximum of 3,824.

However, the dataset also contains opportunity-related repeated rows.

Before treating large attempt counts as outliers, we compare:

- Raw rows with `attempt_count > 100`
- Unique base `order_id` values
- Unique `order_id + skill_id` combinations

We also inspect the largest unique interactions.

No outlier correction is performed.

In [34]:
extreme_attempts = df[df["attempt_count"] > 100]

print(
    "Raw rows with attempts > 100:",
    len(extreme_attempts)
)

print(
    "Unique order_ids:",
    extreme_attempts["order_id"].nunique()
)

print(
    "Unique order_id + skill_id:",
    len(
        extreme_attempts.drop_duplicates(
            subset=["order_id", "skill_id"]
        )
    )
)

print("\nTop extreme base interactions:")

extreme_cols = [
    "order_id",
    "user_id",
    "problem_id",
    "skill_id",
    "skill_name",
    "attempt_count",
    "correct",
    "hint_count"
]

display(
    extreme_attempts
    .drop_duplicates(subset=["order_id", "skill_id"])
    [extreme_cols]
    .sort_values("attempt_count", ascending=False)
    .head(15)
)

Raw rows with attempts > 100: 148
Unique order_ids: 140
Unique order_id + skill_id: 148

Top extreme base interactions:


,order_id,user_id,problem_id,skill_id,skill_name,attempt_count,correct,hint_count
119136,27334469,83208,85958,47.0,Conversion of Fraction Decimals Percents,3824,0,3
264441,27332470,83209,88497,276.0,Multiplication and Division Positive Decimals,3740,0,0
8613,33353256,84309,93402,2.0,Circle Graph,1853,0,0
96917,33353256,84309,93402,37.0,NaN,1853,0,0
192063,33353256,84309,93402,70.0,Percent Of,1853,0,0
191842,33428018,83197,94881,70.0,Percent Of,1631,0,0
282118,35619174,90548,89664,277.0,Addition and Subtraction Integers,1503,0,0
160214,32020677,88029,87118,61.0,Division Fractions,1426,0,4
210978,32086266,83199,60075,76.0,NaN,1265,0,6
403583,24669567,78164,97586,311.0,Equation Solving Two or Fewer Steps,1180,0,1


### Interpretation

This check determines whether the previously observed extreme attempt counts are mainly an artefact of repeated opportunity rows or whether genuinely extreme `attempt_count` values exist at the base interaction level.

The result will help us design a justified attempt-count preprocessing strategy later.

No values are removed, capped, or transformed during this stage.

## 35. Extreme Response Times at the Base Interaction Level

Earlier exploration identified response times greater than one hour, with the maximum reaching many hours.

Because the dataset contains opportunity-related repeated rows, we need to determine whether these extreme response times disappear when interactions are considered at the base level.

We compare:

- Raw rows with response time greater than one hour
- Unique `order_id` values
- Unique `order_id + skill_id` combinations
- The largest base interaction response times

No response-time values are modified.

In [35]:
extreme_response = df[
    df["ms_first_response"] > 3_600_000
]

print(
    "Raw rows > 1 hour:",
    len(extreme_response)
)

print(
    "Unique order_ids:",
    extreme_response["order_id"].nunique()
)

print(
    "Unique order_id + skill_id:",
    len(
        extreme_response.drop_duplicates(
            subset=["order_id", "skill_id"]
        )
    )
)

extreme_response_cols = [
    "order_id",
    "user_id",
    "problem_id",
    "skill_id",
    "skill_name",
    "ms_first_response",
    "correct",
    "attempt_count",
    "hint_count"
]

print("\nLargest base-interaction response times:")

display(
    extreme_response
    .drop_duplicates(subset=["order_id", "skill_id"])
    [extreme_response_cols]
    .sort_values(
        "ms_first_response",
        ascending=False
    )
    .head(15)
)

Raw rows > 1 hour: 200
Unique order_ids: 151
Unique order_id + skill_id: 200

Largest base-interaction response times:


,order_id,user_id,problem_id,skill_id,skill_name,ms_first_response,correct,attempt_count,hint_count
24535,35187187,88478,49012,10.0,Table,84076920,0,1,0
46926,35187187,88478,49012,12.0,Mean,84076920,0,1,0
235238,32190591,79678,87423,83.0,Divisibility Rules,74563450,1,1,0
309326,23086044,79430,53786,280.0,Addition and Subtraction Fractions,66158290,1,1,0
206113,31640268,79632,89632,75.0,Square Root,59446217,1,1,0
64482,31789579,78903,138095,17.0,Probability of Two Distinct Events,44637627,1,1,0
69945,38221066,96211,138095,17.0,Probability of Two Distinct Events,44637627,1,1,0
314119,31344659,87069,86348,280.0,Addition and Subtraction Fractions,28814407,1,1,0
486131,34680132,79071,135893,NaN,NaN,25877940,1,1,0
524865,38252008,96293,135893,NaN,NaN,25877940,1,1,0


### Interpretation

This analysis checks whether extreme response times are caused mainly by opportunity-level row repetition or remain present at the base interaction level.

If the extreme values remain after considering unique interactions, they will require a separate preprocessing strategy.

Possible treatments such as filtering, clipping, transformation, or robust feature construction will only be selected after the data behaviour has been fully investigated.

No preprocessing decision is made at this stage.

## 36. Investigating Negative Response Times

Earlier exploration identified 8 rows where `ms_first_response` is negative.

A negative response time is not logically valid as an elapsed student response duration.

Before deciding how to handle these records during preprocessing, we inspect the affected interactions and determine whether they are repeated representations of the same base interaction.

We examine:

- `order_id`
- Student
- Assignment
- Problem
- Skill
- Response time
- Correctness
- Attempt count
- Hint usage
- `first_action`

No values are modified.

In [36]:
negative_response = df[
    df["ms_first_response"] < 0
]

negative_cols = [
    "order_id",
    "user_id",
    "assignment_id",
    "problem_id",
    "skill_id",
    "skill_name",
    "ms_first_response",
    "correct",
    "attempt_count",
    "hint_count",
    "first_action"
]

print(
    "Negative response-time rows:",
    len(negative_response)
)

print(
    "Unique order_ids:",
    negative_response["order_id"].nunique()
)

print("\nAffected interactions:")

display(
    negative_response[
        negative_cols
    ].sort_values("ms_first_response")
)

Negative response-time rows: 8
Unique order_ids: 7

Affected interactions:


,order_id,user_id,assignment_id,problem_id,skill_id,skill_name,ms_first_response,correct,attempt_count,hint_count,first_action
10859,35397005,79026,281860,51323,4.0,Histogram as Table or Graph,-7759575,1,1,0,0
313173,31480404,86708,273180,86735,280.0,Addition and Subtraction Fractions,-842373,1,1,0,0
313160,31480044,86708,273179,86342,280.0,Addition and Subtraction Fractions,-800125,0,6,3,0
111496,29732401,78895,271445,85928,47.0,Conversion of Fraction Decimals Percents,-314441,1,1,0,0
512579,32915077,89175,278361,54081,NaN,NaN,-6576,1,1,0,0
499554,26392381,83240,266602,90438,NaN,NaN,-6206,0,2,0,0
249565,31272249,70746,272490,49325,92.0,Pattern Finding,-4078,1,1,0,0
195498,31272249,70746,272490,49325,74.0,Subtraction Whole Numbers,-4078,1,1,0,0


### Interpretation

Negative response times are considered a data-quality issue because elapsed response duration cannot normally be negative.

This inspection determines whether the 8 affected rows correspond to 8 separate interactions or whether some are repeated skill/opportunity representations.

The final preprocessing strategy will be decided after examining these records.

No rows are deleted and no response times are replaced at this stage.

## 37. Investigating Zero Response Times

The dataset contains interactions where:

`ms_first_response = 0`

Unlike negative response times, a zero value should not automatically be considered invalid without further investigation.

It could represent:

- A special logging behaviour
- A missing/unknown duration encoded as zero
- An immediate recorded action
- Another dataset-specific condition


We therefore examine:

- Raw zero-response rows
- Unique base interactions
- Skill-aware interactions
- Correctness
- Attempt counts
- Hint usage

No zero response-time records are removed or changed.

In [37]:
zero_response = df[
    df["ms_first_response"] == 0
]

print(
    "Raw zero-response rows:",
    len(zero_response)
)

print(
    "Unique order_ids:",
    zero_response["order_id"].nunique()
)

print(
    "Unique order_id + skill_id:",
    len(
        zero_response.drop_duplicates(
            subset=["order_id", "skill_id"]
        )
    )
)

print("\nCorrectness:")
print(
    zero_response["correct"].value_counts()
)

print("\nAttempt count summary:")
print(
    zero_response["attempt_count"]
    .describe()
    .round(2)
)

print("\nHint usage:")
print(
    (zero_response["hint_count"] > 0)
    .value_counts()
)

Raw zero-response rows: 4678
Unique order_ids: 310
Unique order_id + skill_id: 317

Correctness:
correct
1    2931
0    1747
Name: count, dtype: int64

Attempt count summary:
count    4678.00
mean        0.98
std         1.59
min         0.00
25%         0.00
50%         0.00
75%         2.00
max        13.00
Name: attempt_count, dtype: float64

Hint usage:
hint_count
False    4180
True      498
Name: count, dtype: int64


### Interpretation

This analysis determines whether zero response times behave like ordinary student interactions or appear to represent a special data condition.

A zero response time will not be automatically replaced, removed, or treated as a valid instantaneous response until its behaviour and dataset meaning are sufficiently understood.

The final response-time preprocessing strategy will be defined after completing the remaining data-quality investigation.

## 38. Investigating `opportunity` and `opportunity_original`

Earlier analysis showed that repeated representations of the same interaction often differ only in:

- `opportunity`
- `opportunity_original`

Because these fields are responsible for an important part of the dataset structure, we investigate them before designing the preprocessing pipeline.

We examine:

- Their distributions
- Missing values
- How often the two fields are equal
- How often they differ

No opportunity values are modified.

In [38]:
print("opportunity summary:")
print(
    df["opportunity"]
    .describe()
    .round(2)
)

print("\nopportunity_original summary:")
print(
    df["opportunity_original"]
    .describe()
    .round(2)
)

print(
    "\nSame value:",
    (
        df["opportunity"]
        == df["opportunity_original"]
    ).sum()
)

print(
    "Different value:",
    (
        df["opportunity"]
        != df["opportunity_original"]
    ).sum()
)

print(
    "\nMissing opportunity:",
    df["opportunity"].isna().sum()
)

print(
    "Missing opportunity_original:",
    df["opportunity_original"].isna().sum()
)

opportunity summary:
count    525534.00
mean         79.02
std         271.38
min           1.00
25%           4.00
50%          12.00
75%          46.00
max        3585.00
Name: opportunity, dtype: float64

opportunity_original summary:
count    449220.00
mean         79.27
std         281.53
min           1.00
25%           4.00
50%          10.00
75%          39.00
max        3585.00
Name: opportunity_original, dtype: float64

Same value: 408650
Different value: 116884

Missing opportunity: 0
Missing opportunity_original: 76314


### Interpretation

`opportunity` and `opportunity_original` are being investigated because they are connected to the repeated skill/opportunity representation found earlier.

Understanding these fields is necessary before deciding how to construct:

1. A base interaction table for general student memory.
2. A skill-aware interaction table for Concept-Based Memory.

No deduplication or preprocessing decision is made from this check alone.

## 39. Investigating Interaction Ordering Within Students

Short-Term Memory and learning-state trend calculations require interactions to be processed in the correct order.

The dataset contains `order_id`, but an identifier should not automatically be assumed to be a timestamp.

We therefore test whether each student's base interactions appear in non-decreasing `order_id` order in the raw dataset.

For this check, only one row per `order_id` is used because multiple skill/opportunity rows can represent the same base interaction.

No rows are reordered or modified.

In [39]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

student_order_results = []
unordered_examples = []

for user_id, group in base_interactions.groupby(
    "user_id",
    sort=False
):
    values = group["order_id"].to_numpy()

    is_non_decreasing = (
        values[1:] >= values[:-1]
    ).all()

    student_order_results.append(
        is_non_decreasing
    )

    if (
        not is_non_decreasing
        and len(unordered_examples) < 10
    ):
        unordered_examples.append(
            (
                user_id,
                values[:15].tolist()
            )
        )

print(
    "Students checked:",
    len(student_order_results)
)

print(
    "Students already non-decreasing by order_id:",
    sum(student_order_results)
)

print(
    "Students not non-decreasing:",
    len(student_order_results)
    - sum(student_order_results)
)

print("\nExample unordered students:")

for example in unordered_examples:
    print(example)

Students checked: 4217
Students already non-decreasing by order_id: 1310
Students not non-decreasing: 2907

Example unordered students:
(64525, [33022537, 33022709, 28186893, 28187093, 32413158, 33022751, 33023039, 33023131, 33023183, 33023245, 33023588, 33023618, 33023666, 33023719, 33023842])
(70363, [35450204, 35450295, 35450311, 35450555, 35450573, 35480603, 35305403, 35305686, 35306351, 35306712, 35307058, 35384917, 35385665, 35397317, 35397465])
(70677, [33140811, 33140919, 32839630, 33140432, 32834431, 32834439, 32835372, 32835380, 32835398, 32835475, 32835482, 33745219, 33745405, 33746385, 33746501])
(70695, [33275447, 33275494, 33276570, 33280266, 27839599, 27839886, 27839905, 27839943, 32815212, 32815273, 32815341, 32815355, 32815381, 32815394, 33280320])
(70699, [35222057, 35222063, 32415856, 32416156, 32416348, 32261065, 32261192, 32261410, 32274580, 32274652, 32274751, 32275236, 32275276, 32275305, 32273125])
(70707, [32871676, 32871682, 28192679, 32870939, 32871700, 33155

### Interpretation

This test checks whether the raw row sequence and `order_id` provide consistent interaction ordering within each student.

Even if `order_id` is monotonic for many students, this alone does not prove that it represents an exact timestamp.

The result will determine whether additional ordering evidence is needed before using `order_id` for:

- Short-Term Memory
- Recent-interaction windows
- Improvement/decline calculations
- Learning-state feature generation

No chronological assumption is finalized at this stage.

## 40. Investigating `order_id` Within Student Assignments

The previous analysis showed that the complete raw interaction sequence for most students is not already arranged in increasing `order_id` order.

One possible reason is that interactions from different assignments are mixed in the dataset.

We therefore perform a more focused test.

For each:

`student + assignment`

we check whether the base interactions appear in non-decreasing `order_id` order.

Only one row per `order_id` is considered so that multi-skill and opportunity representations do not distort this test.

No rows are reordered or modified.

In [40]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

total_groups = 0
ordered_groups = 0
unordered_groups = 0
unordered_examples = []

for (user_id, assignment_id), group in (
    base_interactions.groupby(
        ["user_id", "assignment_id"],
        sort=False
    )
):
    values = group["order_id"].to_numpy()

    total_groups += 1

    is_non_decreasing = (
        values[1:] >= values[:-1]
    ).all()

    if is_non_decreasing:
        ordered_groups += 1
    else:
        unordered_groups += 1

        if len(unordered_examples) < 10:
            unordered_examples.append(
                (
                    user_id,
                    assignment_id,
                    values[:15].tolist()
                )
            )

print(
    "Student-assignment groups checked:",
    total_groups
)

print(
    "Groups non-decreasing by order_id:",
    ordered_groups
)

print(
    "Groups not non-decreasing:",
    unordered_groups
)

print(
    "Percentage ordered:",
    round(
        ordered_groups / total_groups * 100,
        2
    )
)

print("\nExample unordered groups:")

for example in unordered_examples:
    print(example)

Student-assignment groups checked: 66239
Groups non-decreasing by order_id: 56981
Groups not non-decreasing: 9258
Percentage ordered: 86.02

Example unordered groups:
(14, 263600, [21617692, 21617731, 21617749, 21617805, 21617825, 21617718, 21617722, 21617780, 21617787, 21617828, 21617834])
(64525, 270591, [28186893, 28187093, 28186971, 28187151, 28187752, 28187806, 28187858])
(64525, 277547, [32413158, 32413105, 32413123])
(64525, 277620, [33023245, 33023588, 33023618, 33023666, 33023523, 33023551])
(70363, 220655, [35305403, 35305686, 35306351, 35306712, 35307058, 35384917, 35385665, 35305924, 35306168, 35307194, 35307280])
(70702, 277547, [32623543, 32623608, 32623585, 32623602])
(70704, 270591, [29429404, 29429410, 29429397])
(70707, 277547, [32870939, 32871023, 32871037, 32870930, 32870975, 32871051])
(70707, 277620, [33155607, 33155651, 33155774, 33155781, 33155630, 33155642])
(70708, 270591, [29483296, 29483274, 29483304, 29483258, 29483265, 29483281, 29483289])


### Interpretation

This analysis checks whether `order_id` behaves more consistently when interactions are examined within individual student assignments.

If most student-assignment groups are internally ordered, it would provide stronger evidence that `order_id` contains useful local ordering information even though the complete raw student sequence is mixed.

However, monotonic behaviour alone still does not prove that `order_id` is an exact timestamp.

The result will help determine how interaction history should be constructed for:

- Short-Term Memory
- Recent performance windows
- Improvement/decline features
- Student Learning State prediction

No final chronological ordering rule is selected yet.

## 41. Investigating `position` as an Ordering Feature

The dataset contains a field named `position`.

Because `order_id` alone has not yet been proven to represent exact chronological order, we investigate whether `position` may describe the position of a problem or interaction within an assignment.

We examine:

- Missing values
- Number of unique values
- Distribution
- Availability at the base interaction level
- Example relationships between student, assignment, `order_id`, and `position`

No ordering rule is created yet.

In [41]:
print(
    "Missing position:",
    df["position"].isna().sum()
)

print(
    "Unique position values:",
    df["position"].nunique()
)

print("\nPosition summary:")

print(
    df["position"]
    .describe()
    .round(2)
)

base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

print(
    "\nBase interactions missing position:",
    base_interactions["position"].isna().sum()
)

print("\nFirst 20 base interaction examples:")

display(
    base_interactions[
        [
            "user_id",
            "assignment_id",
            "order_id",
            "position"
        ]
    ].head(20)
)

Missing position: 0
Unique position values: 283

Position summary:
count    525534.00
mean         46.11
std          60.59
min           1.00
25%           6.00
50%          18.00
75%          57.00
max         295.00
Name: position, dtype: float64

Base interactions missing position: 0

First 20 base interaction examples:


,user_id,assignment_id,order_id,position
0,64525,277618,33022537,126
1,64525,277618,33022709,126
2,70363,220674,35450204,22
3,70363,220674,35450295,22
4,70363,220674,35450311,22
5,70363,220674,35450555,22
6,70363,220674,35450573,22
7,70363,220674,35480603,22
8,70677,220674,33140811,22
9,70677,220674,33140919,22


### Interpretation

The `position` field is being investigated as a possible source of within-assignment ordering information.

If it consistently represents the sequence of problems inside an assignment, it may be more appropriate than relying only on raw CSV row order.

However, the column name alone is not sufficient evidence.

Its behaviour and relationship with assignments and interactions must be investigated before it is used for Short-Term Memory or trend calculations.

No final ordering strategy is selected at this stage.

## 42. Understanding What `position` Represents

The previous analysis showed that many interactions can share the same `position`.

We now investigate whether `position` is primarily associated with:

- An assignment
- A problem
- An assignment-problem combination

This is important because the field should not be used for chronological ordering unless its structural meaning is understood.

No data is modified.

In [42]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

assignments_multiple_positions = (
    base_interactions
    .groupby("assignment_id")["position"]
    .nunique()
    .gt(1)
    .sum()
)

problems_multiple_positions = (
    base_interactions
    .groupby("problem_id")["position"]
    .nunique()
    .gt(1)
    .sum()
)

assignment_problem_multiple_positions = (
    base_interactions
    .groupby(
        ["assignment_id", "problem_id"]
    )["position"]
    .nunique()
    .gt(1)
    .sum()
)

print(
    "Assignments with multiple positions:",
    assignments_multiple_positions
)

print(
    "Problems with multiple positions:",
    problems_multiple_positions
)

print(
    "Assignment+problem pairs with multiple positions:",
    assignment_problem_multiple_positions
)

print(
    "\nUnique assignments:",
    base_interactions["assignment_id"].nunique()
)

print(
    "Unique problems:",
    base_interactions["problem_id"].nunique()
)

print(
    "Unique assignment+problem pairs:",
    base_interactions[
        ["assignment_id", "problem_id"]
    ].drop_duplicates().shape[0]
)

Assignments with multiple positions: 0
Problems with multiple positions: 21273
Assignment+problem pairs with multiple positions: 0

Unique assignments: 3521
Unique problems: 26688


Unique assignment+problem pairs: 155455


### Interpretation

This analysis helps determine the structural meaning of `position`.

If an assignment-problem pair consistently has one position, this would suggest that `position` describes where a problem is placed within an assignment rather than the chronological order in which an individual student answered it.

That distinction is important:

**Problem position**
is not necessarily the same as
**student interaction time/order**.

Therefore, `position` will not be used for Short-Term Memory sequencing until this relationship is confirmed.

## 43. Investigating `sequence_id` and `base_sequence_id`

The `position` field does not provide student-level chronological ordering.

The dataset also contains:

- `sequence_id`
- `base_sequence_id`

These fields may describe the instructional sequence or activity structure.

We therefore investigate:

- Missing values
- Number of unique sequences
- Relationship between assignments and sequences
- Whether sequences are shared across assignments

The goal is to understand dataset structure before deciding how Short-Term Memory should reconstruct student history.

No data is modified.

In [43]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

print("--- sequence_id ---")
print(
    "Missing:",
    base_interactions["sequence_id"].isna().sum()
)
print(
    "Unique:",
    base_interactions["sequence_id"].nunique()
)

print("\n--- base_sequence_id ---")
print(
    "Missing:",
    base_interactions["base_sequence_id"].isna().sum()
)
print(
    "Unique:",
    base_interactions["base_sequence_id"].nunique()
)

print(
    "\nAssignments linked to multiple sequence_ids:",
    (
        base_interactions
        .groupby("assignment_id")["sequence_id"]
        .nunique(dropna=False)
        > 1
    ).sum()
)

print(
    "Assignments linked to multiple base_sequence_ids:",
    (
        base_interactions
        .groupby("assignment_id")["base_sequence_id"]
        .nunique(dropna=False)
        > 1
    ).sum()
)

print(
    "\nsequence_ids linked to multiple assignments:",
    (
        base_interactions
        .groupby("sequence_id")["assignment_id"]
        .nunique()
        > 1
    ).sum()
)

print(
    "base_sequence_ids linked to multiple assignments:",
    (
        base_interactions
        .groupby("base_sequence_id")["assignment_id"]
        .nunique()
        > 1
    ).sum()
)

--- sequence_id ---
Missing: 0
Unique: 677

--- base_sequence_id ---
Missing: 0
Unique: 366

Assignments linked to multiple sequence_ids: 0
Assignments linked to multiple base_sequence_ids: 0

sequence_ids linked to multiple assignments: 347
base_sequence_ids linked to multiple assignments: 272


### Interpretation

`sequence_id` and `base_sequence_id` are being investigated as structural fields rather than automatically treating them as timestamps.

Their relationships with assignments will help determine whether they represent:

- Instructional sequences
- Reusable activity structures
- Assignment organization
- Or information useful for reconstructing student interaction history

No chronological ordering rule is selected yet.

## 44. Searching for an Explicit Timestamp or Date Field

Short-Term Memory requires a reliable definition of "recent" student interactions.

So far:

- Raw CSV row order is not reliably chronological across students.
- `order_id` has some ordering behaviour but is not proven to be a timestamp.
- `position` represents assignment structure rather than student interaction time.
- `sequence_id` and `base_sequence_id` represent structural grouping.

Before designing an alternative ordering strategy, we systematically inspect every raw dataset column and its data type/sample value to determine whether an explicit timestamp or date field exists.

No data is modified.

In [44]:
print("All columns with dtypes:\n")

for column in df.columns:

    non_missing = df[column].dropna()

    sample_value = (
        non_missing.iloc[0]
        if len(non_missing) > 0
        else None
    )

    print(
        f"{column:25} "
        f"{str(df[column].dtype):10} "
        f"Sample: {sample_value}"
    )

All columns with dtypes:

order_id                  int64      Sample: 33022537
assignment_id             int64      Sample: 277618
user_id                   int64      Sample: 64525
assistment_id             int64      Sample: 33139
problem_id                int64      Sample: 51424
original                  int64      Sample: 1
correct                   int64      Sample: 1
attempt_count             int64      Sample: 1
ms_first_response         int64      Sample: 32454
tutor_mode                str        Sample: tutor
answer_type               str        Sample: algebra
sequence_id               int64      Sample: 5948
student_class_id          int64      Sample: 13241
position                  int64      Sample: 126
type                      str        Sample: MasterySection
base_sequence_id          int64      Sample: 5948
skill_id                  float64    Sample: 1.0
skill_name                str        Sample: Box and Whisker
teacher_id                int64      Sample: 2276

### Interpretation

This inspection determines whether the raw dataset contains an explicit timestamp/date variable that can establish true student interaction chronology.

If no such field exists, the project must clearly document this dataset limitation.

Any proxy ordering strategy used later must then be justified and must not be described as exact chronological time unless supported by evidence.

## 45. Investigating `overlap_time`

The dataset contains another time-related field called `overlap_time`.

The first inspected value was identical to `ms_first_response`, so we need to determine whether these two fields contain the same information or represent different timing behaviour.

We investigate:

- Missing `overlap_time` values
- Negative values
- Zero values
- Exact equality with `ms_first_response`
- Distribution of `overlap_time`
- Distribution of the difference between the two timing fields

No timing values are modified.

In [45]:
print(
    "Missing overlap_time:",
    df["overlap_time"].isna().sum()
)

print(
    "Negative overlap_time:",
    (df["overlap_time"] < 0).sum()
)

print(
    "Zero overlap_time:",
    (df["overlap_time"] == 0).sum()
)

print(
    "\nExactly equal to ms_first_response:",
    (
        df["overlap_time"]
        == df["ms_first_response"]
    ).sum()
)

print(
    "Different from ms_first_response:",
    (
        df["overlap_time"]
        != df["ms_first_response"]
    ).sum()
)

print("\noverlap_time summary:")

print(
    df["overlap_time"]
    .describe()
    .round(2)
)

time_difference = (
    df["overlap_time"]
    - df["ms_first_response"]
)

print(
    "\nDifference summary "
    "(overlap_time - ms_first_response):"
)

print(
    time_difference
    .describe()
    .round(2)
)

Missing overlap_time: 0
Negative overlap_time: 11
Zero overlap_time: 2982

Exactly equal to ms_first_response: 366920
Different from ms_first_response: 158614

overlap_time summary:
count      525534.00
mean        59588.73
std        342412.89
min      -7759575.00
25%          9904.00
50%         25735.00
75%         62323.00
max      84076925.00
Name: overlap_time, dtype: float64

Difference summary (overlap_time - ms_first_response):
count      525534.00
mean        11596.56
std        111330.63
min      -7267727.00
25%             0.00
50%             0.00
75%           110.00
max      46520416.00
dtype: float64


### Interpretation

This analysis determines whether `overlap_time` provides additional behavioural timing information beyond `ms_first_response`.

If the two fields are largely identical, including both as learning-state features could introduce redundant information.

If they differ meaningfully, the relationship must be understood before either field is selected.

No feature-selection or preprocessing decision is made at this stage.

## 46. Investigating `original`

The dataset contains a field named `original`.

Before deciding whether this field is relevant to student memory or learning-state prediction, we investigate its basic behaviour.

We examine:

- Missing values
- Available values and their frequencies
- Relationship with correctness
- Relationship with attempt count

The field name alone is not used to infer its official meaning.

No data is modified.

In [46]:
print(
    "Missing original:",
    df["original"].isna().sum()
)

print("\noriginal values:")

print(
    df["original"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPercentages:")

print(
    (
        df["original"]
        .value_counts(
            normalize=True,
            dropna=False
        )
        .sort_index()
        * 100
    ).round(2)
)

print("\noriginal vs correct:")

print(
    pd.crosstab(
        df["original"],
        df["correct"]
    )
)

print("\noriginal vs attempt_count summary:")

print(
    df.groupby("original")["attempt_count"]
    .agg([
        "count",
        "mean",
        "median",
        "max"
    ])
    .round(2)
)

Missing original: 0

original values:
original
0     76314
1    449220
Name: count, dtype: int64

Percentages:
original
0    14.52
1    85.48
Name: proportion, dtype: float64

original vs correct:
correct        0       1
original                
0          30179   46135
1         138270  310950

original vs attempt_count summary:


           count  mean  median   max
original                            
0          76314  1.84     1.0  1180
1         449220  1.45     1.0  3824


### Interpretation

This analysis explores whether `original` separates interactions with noticeably different correctness or attempt behaviour.

However, statistical relationships alone are not enough to determine what `original = 0` and `original = 1` officially mean.

The field will not be used as a learning-state feature until its meaning and relevance are sufficiently understood.

## 47. Investigating `assistment_id` and `problem_id`

The dataset contains both:

- `assistment_id`
- `problem_id`

Before designing the memory schema, we need to understand whether these fields identify the same learning object or different levels of the ASSISTments content structure.

We investigate:

- Missing values
- Number of unique IDs
- Whether one assistment can contain multiple problems
- Whether one problem can belong to multiple assistments

No data is modified.

In [47]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

print(
    "Missing assistment_id:",
    base_interactions["assistment_id"].isna().sum()
)

print(
    "Unique assistment_id:",
    base_interactions["assistment_id"].nunique()
)

print(
    "Unique problem_id:",
    base_interactions["problem_id"].nunique()
)

assistment_problem_counts = (
    base_interactions
    .groupby("assistment_id")["problem_id"]
    .nunique()
)

problem_assistment_counts = (
    base_interactions
    .groupby("problem_id")["assistment_id"]
    .nunique()
)

print(
    "\nassistment_ids linked to multiple problems:",
    (assistment_problem_counts > 1).sum()
)

print(
    "problem_ids linked to multiple assistments:",
    (problem_assistment_counts > 1).sum()
)

print(
    "\nMaximum problems per assistment:",
    assistment_problem_counts.max()
)

print(
    "Maximum assistments per problem:",
    problem_assistment_counts.max()
)

Missing assistment_id: 0
Unique assistment_id: 17725
Unique problem_id: 26688

assistment_ids linked to multiple problems: 2990
problem_ids linked to multiple assistments: 0

Maximum problems per assistment: 11
Maximum assistments per problem: 1


### Interpretation

This analysis determines the structural relationship between `assistment_id` and `problem_id`.

Understanding this distinction is important when deciding what identifier should be stored in the Student Personalization Memory Layer for each student interaction.

The final identifier design will be based on the observed relationship and verified dataset semantics rather than column names alone.

## 48. Investigating Class, Teacher, and School Structure

The dataset also contains contextual identifiers:

- `student_class_id`
- `teacher_id`
- `school_id`

These fields describe the educational environment rather than the student's direct learning behaviour.

Before deciding whether they are relevant to the Student Personalization Memory Layer or Student Learning State model, we investigate:

- Number of unique classes, teachers, and schools
- Missing values
- Whether students appear in multiple classes
- Whether classes are associated with multiple teachers
- Whether teachers or classes are associated with multiple schools

No contextual identifiers are modified.

In [48]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

print(
    "Unique classes:",
    base_interactions["student_class_id"].nunique()
)

print(
    "Unique teachers:",
    base_interactions["teacher_id"].nunique()
)

print(
    "Unique schools:",
    base_interactions["school_id"].nunique()
)

print(
    "\nMissing classes:",
    base_interactions["student_class_id"].isna().sum()
)

print(
    "Missing teachers:",
    base_interactions["teacher_id"].isna().sum()
)

print(
    "Missing schools:",
    base_interactions["school_id"].isna().sum()
)

print(
    "\nStudents linked to multiple classes:",
    (
        base_interactions
        .groupby("user_id")["student_class_id"]
        .nunique()
        > 1
    ).sum()
)

print(
    "Classes linked to multiple teachers:",
    (
        base_interactions
        .groupby("student_class_id")["teacher_id"]
        .nunique()
        > 1
    ).sum()
)

print(
    "Teachers linked to multiple schools:",
    (
        base_interactions
        .groupby("teacher_id")["school_id"]
        .nunique()
        > 1
    ).sum()
)

print(
    "Classes linked to multiple schools:",
    (
        base_interactions
        .groupby("student_class_id")["school_id"]
        .nunique()
        > 1
    ).sum()
)

Unique classes: 250
Unique teachers: 153
Unique schools: 75

Missing classes: 0
Missing teachers: 0
Missing schools: 0

Students linked to multiple classes: 437


Classes linked to multiple teachers: 0
Teachers linked to multiple schools: 21
Classes linked to multiple schools: 25


### Interpretation

These identifiers provide educational context but are not automatically suitable learning-state features.

In particular, school, teacher, or class identity could cause the model to learn institution-specific patterns instead of the student's actual learning behaviour.

The relationships are therefore documented first.

A later feature-selection stage will decide whether these contextual identifiers should be:

- Stored only as metadata
- Excluded from machine-learning features
- Or used only for specific research analysis

No feature-selection decision is finalized yet.

## 49. Investigating Interaction-Type Categorical Fields

The dataset contains several categorical fields describing the interaction:

- `tutor_mode`
- `answer_type`
- `type`

These fields may provide useful context about how a student interacted with a problem.

Before deciding whether they should be stored or used as Student Learning State features, we investigate:

- Missing values
- Number of unique categories
- Frequency of each category

No categorical values are modified.

In [49]:
categorical_interaction_cols = [
    "tutor_mode",
    "answer_type",
    "type"
]

for column in categorical_interaction_cols:

    print(f"--- {column} ---")

    print(
        "Missing:",
        df[column].isna().sum()
    )

    print(
        "Unique:",
        df[column].nunique()
    )

    print(
        df[column]
        .value_counts(dropna=False)
        .to_string()
    )

    print()

--- tutor_mode ---


Missing: 0
Unique: 2
tutor_mode
tutor    525201
test        333

--- answer_type ---
Missing: 0
Unique: 5
answer_type
algebra          436214
choose_1          48923
fill_in_1         39320
choose_n           1069
open_response         8

--- type ---
Missing: 0


Unique: 1
type
MasterySection    525534



### Interpretation

These categorical fields are being examined to determine whether they provide meaningful interaction context or contain little useful variation.

Fields with almost no variation are unlikely to provide useful predictive information.

Fields with meaningful categories may still require further investigation before being selected for the Student Learning State model.

No feature-selection decision is made at this stage.

## 50. Behaviour Across `answer_type`

`answer_type` contains several different response formats and has more variation than fields such as `type`.

We investigate whether student behaviour differs across answer types.

For each `answer_type`, we calculate:

- Number of rows
- Accuracy
- Average attempt count
- Average hint count
- Median first-response time

This helps determine whether answer format may influence behavioural measurements used by the Student Learning State model.

No feature-selection decision is made yet.

In [50]:
answer_type_behaviour = (
    df.groupby("answer_type")
    .agg(
        rows=("order_id", "size"),
        accuracy=("correct", "mean"),
        avg_attempts=("attempt_count", "mean"),
        avg_hints=("hint_count", "mean"),
        median_response_ms=("ms_first_response", "median")
    )
    .round(3)
)

display(answer_type_behaviour)

,rows,accuracy,avg_attempts,avg_hints,median_response_ms
answer_type,,,,,
algebra,436214,0.677,1.500,0.453,22751.5
choose_1,48923,0.760,1.346,0.069,11227.0
choose_n,1069,0.656,1.284,0.110,8253.0
fill_in_1,39320,0.605,1.730,0.782,19407.0
open_response,8,0.000,1.000,2.750,63929.0


### Interpretation

Different answer formats may naturally produce different levels of:

- Correctness
- Attempts
- Hint usage
- Response time

Therefore, behavioural differences should not automatically be interpreted as differences in student ability.

If `answer_type` has a meaningful relationship with these variables, it may later be useful as contextual information when constructing learning-state features.

However, rare categories must also be treated carefully because very small groups may not provide reliable evidence.

No preprocessing or feature-selection decision is finalized at this stage.

## 51. Behaviour Across `tutor_mode`

The dataset contains two tutor modes:

- `tutor`
- `test`

However, the `test` category is extremely rare.

We investigate whether student behaviour differs between these modes by comparing:

- Number of interactions
- Number of unique students
- Accuracy
- Average attempt count
- Average hint usage
- Median first-response time

This analysis helps determine whether `tutor_mode` provides useful contextual information.

No feature-selection decision is made yet.

In [51]:
tutor_mode_behaviour = (
    df.groupby("tutor_mode")
    .agg(
        rows=("order_id", "size"),
        unique_students=("user_id", "nunique"),
        accuracy=("correct", "mean"),
        avg_attempts=("attempt_count", "mean"),
        avg_hints=("hint_count", "mean"),
        median_response_ms=("ms_first_response", "median")
    )
    .round(3)
)

display(tutor_mode_behaviour)

,rows,unique_students,accuracy,avg_attempts,avg_hints,median_response_ms
tutor_mode,,,,,,
test,333,30,0.751,1.000,0.000,14914.0
tutor,525201,4214,0.679,1.503,0.441,20797.0


### Interpretation

`tutor_mode` may represent different interaction contexts, and student behaviour may naturally differ between tutor and test conditions.

However, the `test` category contains very few interactions compared with `tutor`.

Therefore:

- Differences involving `test` must be interpreted cautiously.
- The field should not automatically be treated as an important predictive feature.
- Its usefulness will be reconsidered during feature selection.

No preprocessing decision is made at this stage.

## 52. Investigating `bottom_hint`

The dataset contains a field named `bottom_hint`.

Earlier analysis showed that it has a relationship with `first_action`, but the field itself has not yet been investigated in detail.

We examine:

- Missing values
- Distribution of recorded values
- Relationship with actual hint usage
- Accuracy
- Attempt behaviour
- Average hint usage

The field name alone is not used to assign an official semantic meaning.

No data is modified.

In [52]:
print(
    "Missing bottom_hint:",
    df["bottom_hint"].isna().sum()
)

print(
    "Percentage missing:",
    round(
        df["bottom_hint"].isna().mean() * 100,
        2
    )
)

print("\nValues:")

print(
    df["bottom_hint"]
    .value_counts(dropna=False)
)

print("\nbottom_hint vs hint_count > 0:")

print(
    pd.crosstab(
        df["bottom_hint"],
        df["hint_count"] > 0,
        dropna=False
    )
)

print("\nBehaviour by bottom_hint:")

bottom_hint_behaviour = (
    df.groupby(
        "bottom_hint",
        dropna=False
    )
    .agg(
        rows=("order_id", "size"),
        accuracy=("correct", "mean"),
        avg_attempts=("attempt_count", "mean"),
        avg_hints=("hint_count", "mean")
    )
    .round(3)
)

display(bottom_hint_behaviour)

Missing bottom_hint: 445627
Percentage missing: 84.8

Values:
bottom_hint
NaN    445627
1.0     60009
0.0     19898
Name: count, dtype: int64

bottom_hint vs hint_count > 0:
hint_count    False  True 
bottom_hint               
0.0               4  19894
1.0               0  60009
NaN          445627      0

Behaviour by bottom_hint:


,rows,accuracy,avg_attempts,avg_hints
bottom_hint,,,,
0.0,19898,0.001,3.300,2.048
1.0,60009,0.000,3.051,3.182
NaN,445627,0.801,1.214,0.000


### Interpretation

`bottom_hint` is being investigated as a possible hint-related contextual field.

A missing value must not automatically be interpreted as `0`, because missingness may itself correspond to a different interaction condition.

The relationship between `bottom_hint`, `hint_count`, correctness, and attempts will help determine whether this field provides additional information beyond ordinary hint usage.

Its official semantic meaning must still be verified before it is used in the Student Learning State model.

No feature-selection or missing-value treatment is finalized at this stage.

## 53. Does `bottom_hint` Add Information Beyond `hint_count`?

The previous analysis showed a very strong relationship between `bottom_hint` and hint usage.

Because `bottom_hint` is missing for 84.8% of rows, we should determine whether it provides useful information beyond the simpler `hint_count` variable.

For interactions where at least one hint was used, we examine:

- `bottom_hint` distribution
- Relationship between `hint_count` and `bottom_hint`
- Correctness for different combinations

This helps identify whether the two fields are redundant or capture different aspects of hint behaviour.

No feature is removed at this stage.

In [53]:
hinted_interactions = df[
    df["hint_count"] > 0
]

print(
    "Rows with hint usage:",
    len(hinted_interactions)
)

print(
    "\nbottom_hint distribution among hinted rows:"
)

print(
    hinted_interactions["bottom_hint"]
    .value_counts(dropna=False)
)

print("\nBy hint_count:")

print(
    pd.crosstab(
        hinted_interactions["hint_count"],
        hinted_interactions["bottom_hint"],
        dropna=False
    )
)

print(
    "\nAccuracy by hint_count and bottom_hint:"
)

hint_bottom_accuracy = (
    hinted_interactions
    .groupby(
        ["hint_count", "bottom_hint"],
        dropna=False
    )["correct"]
    .agg(["count", "mean"])
    .round(3)
)

display(hint_bottom_accuracy)

Rows with hint usage: 79903

bottom_hint distribution among hinted rows:
bottom_hint
1.0    60009
0.0    19894
Name: count, dtype: int64

By hint_count:
bottom_hint   0.0    1.0
hint_count              
1            8101    441
2            5870  12195
3            3480  29713
4            1909  13026
5             372   3306
6             161    945
7               0    383
10              1      0

Accuracy by hint_count and bottom_hint:


count   mean
hint_count bottom_hint              
1          0.0           8101  0.001
           1.0            441  0.000
2          0.0           5870  0.000
           1.0          12195  0.000
3          0.0           3480  0.000
           1.0          29713  0.000
4          0.0           1909  0.001
           1.0          13026  0.000
5          0.0            372  0.000
           1.0           3306  0.000
6          0.0            161  0.000
           1.0            945  0.000
7          1.0            383  0.000
10         0.0              1  0.000

### Interpretation

This analysis evaluates whether `bottom_hint` captures additional information about hint behaviour beyond the number of hints used.

If `bottom_hint` separates different behaviours among interactions with similar `hint_count`, it may contain useful contextual information.

If it is almost completely determined by `hint_count`, using both fields may introduce unnecessary redundancy.

Because the official meaning of `bottom_hint` has not yet been verified, no final feature-selection decision is made from statistical association alone.

## 54. Investigating Hint Usage Ratio

Raw `hint_count` tells us how many hints were used, but different problems contain different numbers of available hints.

For example:

- Using 2 of 2 available hints
- Using 2 of 7 available hints

may represent different levels of student support need.

We therefore investigate a possible derived behavioural measure:

`hint_usage_ratio = hint_count / hint_total`

The ratio is calculated only when `hint_total > 0`.

We examine:

- Number of eligible interactions
- Distribution of the ratio
- Whether any ratio exceeds 1
- Frequency of no hint usage
- Frequency of using all available hints
- Relationship between hint usage ratio and correctness

This is exploratory analysis only. The feature is not yet added to the final model.

In [54]:
hint_ratio_data = df[
    df["hint_total"] > 0
].copy()

hint_ratio_data["hint_usage_ratio"] = (
    hint_ratio_data["hint_count"]
    / hint_ratio_data["hint_total"]
)

print(
    "Rows with hint_total > 0:",
    len(hint_ratio_data)
)

print(
    "Rows with hint_total = 0:",
    (df["hint_total"] == 0).sum()
)

print("\nRatio summary:")

print(
    hint_ratio_data["hint_usage_ratio"]
    .describe()
    .round(3)
)

print(
    "\nRatio > 1:",
    (hint_ratio_data["hint_usage_ratio"] > 1).sum()
)

print(
    "Ratio = 0:",
    (hint_ratio_data["hint_usage_ratio"] == 0).sum()
)

print(
    "Ratio = 1:",
    (hint_ratio_data["hint_usage_ratio"] == 1).sum()
)

ratio_bins = [
    -0.001,
    0,
    0.25,
    0.50,
    0.75,
    0.999999,
    1
]

ratio_labels = [
    "0",
    "0-25%",
    "25-50%",
    "50-75%",
    "75-<100%",
    "100%"
]

hint_ratio_data["ratio_group"] = pd.cut(
    hint_ratio_data["hint_usage_ratio"],
    bins=ratio_bins,
    labels=ratio_labels,
    include_lowest=True
)

print("\nAccuracy by ratio group:")

display(
    hint_ratio_data
    .groupby(
        "ratio_group",
        observed=True
    )["correct"]
    .agg(["count", "mean"])
    .round(3)
)

Rows with hint_total > 0: 386983
Rows with hint_total = 0: 138551

Ratio summary:
count    386983.000
mean          0.182
std           0.372
min           0.000
25%           0.000
50%           0.000
75%           0.000
max           1.000
Name: hint_usage_ratio, dtype: float64

Ratio > 1: 0
Ratio = 0: 307080
Ratio = 1: 59757

Accuracy by ratio group:


,count,mean
ratio_group,,
0,307080,0.845
0-25%,3047,0.000
25-50%,7054,0.000
50-75%,8172,0.000
75-<100%,1873,0.000
100%,59757,0.000


### Interpretation

The hint usage ratio is being investigated as a possible normalized measure of student support-seeking behaviour.

Unlike raw `hint_count`, the ratio considers how many hints were available for the problem.

This may eventually help distinguish between students who:

- Needed no hints
- Used only part of the available support
- Used all available hints

However, `hint_total = 0` requires separate treatment because the ratio is undefined when no hints are available.

The relationship between hint usage and correctness must also be interpreted carefully because the ASSISTments interaction mechanics may influence how correctness is recorded after hints are requested.

No final feature-engineering decision is made at this stage.

## 55. Attempt Count and Student Performance

`attempt_count` is one of the proposed behavioural signals for identifying a student's learning state.

However, the dataset contains:

- Zero attempts
- Normal attempt counts
- Extremely large attempt counts

Instead of relying only on the raw mean, we group interactions into:

- 0 attempts
- 1 attempt
- 2 attempts
- 3 attempts
- 4–5 attempts
- 6–10 attempts
- More than 10 attempts

For each group, we examine:

- Number of interactions
- Accuracy
- Average hint usage
- Median first-response time

This helps determine whether attempt behaviour provides meaningful information about student difficulty.

No attempt values are modified.

In [55]:
attempt_bins = [
    -1,
    0,
    1,
    2,
    3,
    5,
    10,
    float("inf")
]

attempt_labels = [
    "0",
    "1",
    "2",
    "3",
    "4-5",
    "6-10",
    ">10"
]

attempt_analysis = df.copy()

attempt_analysis["attempt_group"] = pd.cut(
    attempt_analysis["attempt_count"],
    bins=attempt_bins,
    labels=attempt_labels
)

attempt_group_behaviour = (
    attempt_analysis
    .groupby(
        "attempt_group",
        observed=True
    )
    .agg(
        rows=("order_id", "size"),
        accuracy=("correct", "mean"),
        avg_hints=("hint_count", "mean"),
        median_response_ms=(
            "ms_first_response",
            "median"
        )
    )
    .round(3)
)

display(attempt_group_behaviour)

,rows,accuracy,avg_hints,median_response_ms
attempt_group,,,,
0,22153,0.586,0.028,1.0
1,403814,0.848,0.219,21269.0
2,58585,0.020,1.194,24865.0
3,18917,0.010,1.552,25027.0
4-5,13515,0.003,1.794,22243.0
6-10,5743,0.004,2.252,27994.0
>10,2807,0.007,2.167,27597.0


### Interpretation

This analysis investigates whether attempt count is a meaningful indicator of student difficulty.

If higher attempt groups show systematically different correctness, hint usage, or response-time behaviour, attempt count may be useful when constructing learning-state features.

However, raw attempt count should not automatically be used without preprocessing because extreme values have already been identified.

Also, `attempt_count = 0` is treated separately because earlier analysis showed that zero-attempt records can still contain correct and incorrect interactions.

No clipping, transformation, or final feature-engineering decision is made at this stage.

## 56. Response Time and Student Performance

Response time is one of the proposed behavioural signals for Student Learning State identification.

Previous exploration identified:

- Negative response times
- Zero response times
- Extremely large response times

Therefore, this analysis considers only interactions where:

`ms_first_response > 0`

Positive response times are grouped into:

- Up to 5 seconds
- 5–10 seconds
- 10–30 seconds
- 30–60 seconds
- 1–5 minutes
- 5–10 minutes
- 10–60 minutes
- More than 1 hour

For each group, we examine:

- Number of interactions
- Accuracy
- Average attempts
- Average hint usage

This is exploratory analysis only. No response-time values are modified.

In [56]:
positive_response = df[
    df["ms_first_response"] > 0
].copy()

response_bins = [
    0,
    5_000,
    10_000,
    30_000,
    60_000,
    300_000,
    600_000,
    3_600_000,
    float("inf")
]

response_labels = [
    "<=5s",
    "5-10s",
    "10-30s",
    "30-60s",
    "1-5m",
    "5-10m",
    "10-60m",
    ">1h"
]

positive_response["response_group"] = pd.cut(
    positive_response["ms_first_response"],
    bins=response_bins,
    labels=response_labels,
    include_lowest=True
)

response_behaviour = (
    positive_response
    .groupby(
        "response_group",
        observed=True
    )
    .agg(
        rows=("order_id", "size"),
        accuracy=("correct", "mean"),
        avg_attempts=("attempt_count", "mean"),
        avg_hints=("hint_count", "mean")
    )
    .round(3)
)

display(response_behaviour)

,rows,accuracy,avg_attempts,avg_hints
response_group,,,,
<=5s,79379,0.599,1.167,0.737
5-10s,78847,0.728,1.471,0.314
10-30s,155808,0.717,1.547,0.328
30-60s,97971,0.697,1.481,0.378
1-5m,102299,0.639,1.692,0.538
5-10m,4710,0.564,2.259,0.614
10-60m,1634,0.573,3.781,0.539
>1h,200,0.690,1.490,0.175


### Interpretation

This analysis investigates whether response duration contains useful behavioural information for learning-state identification.

Response time should not automatically be interpreted as ability:

- A very fast answer may indicate fluency, guessing, or a special interaction condition.
- A slower answer may indicate difficulty, careful reasoning, interruption, or inactivity.

Therefore, response time is more meaningful when combined with signals such as correctness, attempts, hints, and recent performance.

Negative, zero, and extreme response-time values will require separate preprocessing decisions before model training.

No final response-time transformation or threshold is selected at this stage.

## 57. Student History Coverage Using Base Interactions

The Student Learning State system will use behavioural history such as:

- Recent accuracy
- Hint usage
- Attempt behaviour
- Response time
- Improvement or decline

Raw dataset rows cannot be directly treated as independent student interactions because one `order_id` may appear across multiple skill or opportunity representations.

We therefore examine student history using one base interaction per:

`order_id`

For each student, we calculate the number of unique base interactions and examine how many students have at least:

- 1
- 5
- 10
- 20
- 30
- 50
- 100

base interactions.

This helps determine how much historical evidence is available for learning-state estimation.

In [47]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

student_base_counts = (
    base_interactions
    .groupby("user_id")
    .size()
)

print(
    "Unique base interactions:",
    len(base_interactions)
)

print(
    "Students:",
    len(student_base_counts)
)

print("\nBase interactions per student:")

print(
    student_base_counts
    .describe()
    .round(2)
)

history_thresholds = [
    1,
    5,
    10,
    20,
    30,
    50,
    100
]

print("\nStudent history coverage:")

for threshold in history_thresholds:

    count = (
        student_base_counts >= threshold
    ).sum()

    percentage = (
        student_base_counts >= threshold
    ).mean() * 100

    print(
        f"Students with >= {threshold} "
        f"base interactions: "
        f"{count} ({percentage:.2f}%)"
    )

Unique base interactions: 346860
Students: 4217

Base interactions per student:
count    4217.00
mean       82.25
std       163.68
min         1.00
25%        10.00
50%        23.00
75%        74.00
max      1383.00
dtype: float64

Student history coverage:
Students with >= 1 base interactions: 4217 (100.00%)
Students with >= 5 base interactions: 3744 (88.78%)
Students with >= 10 base interactions: 3168 (75.12%)
Students with >= 20 base interactions: 2314 (54.87%)
Students with >= 30 base interactions: 1843 (43.70%)
Students with >= 50 base interactions: 1342 (31.82%)
Students with >= 100 base interactions: 836 (19.82%)


### Interpretation

This analysis measures how much interaction history is available for each student after removing repeated representations of the same base interaction.

This is important because learning-state estimates based on very little history are less reliable than estimates based on many previous interactions.

The result will help us later design:

- Minimum-history requirements
- Cold-start handling
- Recent interaction windows
- Confidence scores for learning-state predictions

Students with limited history will not automatically be removed.

Instead, the final system should explicitly handle low-history or cold-start students.

No minimum-history threshold is selected at this stage.

## 58. Concept History Coverage per Student

The personalization system requires Concept-Based Memory in addition to general student history.

A student may have many interactions overall but very little evidence for a particular mathematical skill.

To measure concept-specific history, we construct a skill-aware interaction representation using:

`order_id + skill_id`

Repeated opportunity rows for the same interaction-skill pair are counted only once.

Rows with missing `skill_id` are excluded only from this specific analysis because they cannot be assigned to a known concept.

For each:

`student + skill`

we calculate the number of available interactions.

We then examine how many student-skill pairs have at least:

- 1 interaction
- 2 interactions
- 3 interactions
- 5 interactions
- 10 interactions
- 20 interactions

No raw data is deleted or modified.

In [48]:
skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(
        subset=["order_id", "skill_id"]
    )
)

student_skill_counts = (
    skill_interactions
    .groupby(
        ["user_id", "skill_id"]
    )
    .size()
)

print(
    "Skill-aware interactions:",
    len(skill_interactions)
)

print(
    "Student-skill pairs:",
    len(student_skill_counts)
)

print(
    "\nInteractions per student-skill pair:"
)

print(
    student_skill_counts
    .describe()
    .round(2)
)

skill_history_thresholds = [
    1,
    2,
    3,
    5,
    10,
    20
]

print("\nConcept history coverage:")

for threshold in skill_history_thresholds:

    count = (
        student_skill_counts >= threshold
    ).sum()

    percentage = (
        student_skill_counts >= threshold
    ).mean() * 100

    print(
        f"Student-skill pairs with >= "
        f"{threshold} interactions: "
        f"{count} ({percentage:.2f}%)"
    )

Skill-aware interactions: 338001
Student-skill pairs: 41982

Interactions per student-skill pair:
count    41982.00
mean         8.05
std         10.68
min          1.00
25%          2.00
50%          5.00
75%         10.00
max        290.00
dtype: float64

Concept history coverage:
Student-skill pairs with >= 1 interactions: 41982 (100.00%)
Student-skill pairs with >= 2 interactions: 33381 (79.51%)
Student-skill pairs with >= 3 interactions: 28215 (67.21%)
Student-skill pairs with >= 5 interactions: 22133 (52.72%)
Student-skill pairs with >= 10 interactions: 11389 (27.13%)
Student-skill pairs with >= 20 interactions: 3746 (8.92%)


### Interpretation

This analysis measures the amount of concept-specific evidence available for personalized learning-state estimation.

Concept-level history is expected to be sparser than overall student history.

This has an important implication for the final system:

A student's general learning state and concept-specific learning state should not necessarily use the same amount of historical evidence.

For student-skill pairs with limited history, the system may later need to combine:

- Concept-specific evidence
- Overall recent student behaviour
- A confidence or evidence-strength measure

This provides a possible strategy for handling concept-level cold start without deleting students or skills with limited history.

No minimum concept-history threshold is selected at this stage.

## 59. Concept Breadth per Student

Concept-Based Memory needs to store a student's performance across individual mathematical skills.

We therefore investigate how many distinct known skills each student has interacted with.

For this analysis:

- Rows with missing `skill_id` are excluded because they cannot be assigned to a known concept.
- Repeated representations of the same `order_id + skill_id` are counted only once.

For each student, we calculate the number of unique skills represented in their history.

We also examine how many students have experience with at least:

- 1 skill
- 5 skills
- 10 skills
- 20 skills
- 30 skills
- 50 skills

Finally, we check whether any students have no interactions associated with a known skill.

No data is modified.

In [49]:
skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(
        subset=["order_id", "skill_id"]
    )
)

skills_per_student = (
    skill_interactions
    .groupby("user_id")["skill_id"]
    .nunique()
)

print(
    "Students with known skills:",
    len(skills_per_student)
)

print("\nUnique skills per student:")

print(
    skills_per_student
    .describe()
    .round(2)
)

skill_breadth_thresholds = [
    1,
    5,
    10,
    20,
    30,
    50
]

print("\nSkill breadth coverage:")

for threshold in skill_breadth_thresholds:

    count = (
        skills_per_student >= threshold
    ).sum()

    percentage = (
        skills_per_student >= threshold
    ).mean() * 100

    print(
        f"Students with >= {threshold} "
        f"different skills: "
        f"{count} ({percentage:.2f}%)"
    )

total_students = df["user_id"].nunique()

students_without_known_skills = (
    total_students
    - len(skills_per_student)
)

print(
    "\nStudents with no known-skill interactions:",
    students_without_known_skills
)

Students with known skills: 4163

Unique skills per student:
count    4163.00
mean       10.08
std        16.14
min         1.00
25%         2.00
50%         4.00
75%        10.00
max        92.00
Name: skill_id, dtype: float64

Skill breadth coverage:
Students with >= 1 different skills: 4163 (100.00%)
Students with >= 5 different skills: 1987 (47.73%)
Students with >= 10 different skills: 1118 (26.86%)
Students with >= 20 different skills: 522 (12.54%)
Students with >= 30 different skills: 357 (8.58%)
Students with >= 50 different skills: 194 (4.66%)

Students with no known-skill interactions: 54


### Interpretation

This analysis measures the breadth of concept information available for each student.

A student with many represented skills can support a richer Concept-Based Memory profile, while a student with only a few known skills will naturally have a more limited concept profile.

This distinction is important for the final system:

- Concept memory should be created only for skills actually observed for the student.
- Missing concepts should not automatically be interpreted as weak performance.
- A student may have interaction history even when some interactions do not contain a known `skill_id`.
- Concept-level confidence should depend on the amount of evidence available for each skill.

No minimum number of skills is required at this stage.

## 60. Missing Skill Coverage Across Students

Some interactions do not contain a known `skill_id`.

These interactions may still contain useful general behavioural information such as:

- Correctness
- Attempts
- Hint usage
- Response time

However, they cannot directly contribute to Concept-Based Memory.

We therefore investigate whether missing-skill interactions are concentrated among a small number of students or distributed across the student population.

Using one row per base `order_id`, we calculate for each student:

- Total base interactions
- Number of interactions with missing `skill_id`
- Percentage of their history with missing skill information

We also examine how many students have:

- At least one missing-skill interaction
- Only missing-skill interactions
- No missing-skill interactions
- More than 25%, 50%, or 75% missing skill information

No interactions are removed.

In [50]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

student_skill_missingness = (
    base_interactions
    .groupby("user_id")
    .agg(
        total=("order_id", "size"),
        missing_skill=(
            "skill_id",
            lambda x: x.isna().sum()
        )
    )
)

student_skill_missingness["missing_pct"] = (
    student_skill_missingness["missing_skill"]
    / student_skill_missingness["total"]
    * 100
)

print(
    "Students with at least one missing-skill "
    "base interaction:",
    (
        student_skill_missingness[
            "missing_skill"
        ] > 0
    ).sum()
)

print(
    "Students with only missing-skill "
    "base interactions:",
    (
        student_skill_missingness[
            "missing_skill"
        ]
        == student_skill_missingness["total"]
    ).sum()
)

print(
    "Students with no missing-skill "
    "base interactions:",
    (
        student_skill_missingness[
            "missing_skill"
        ] == 0
    ).sum()
)

print(
    "\nMissing-skill percentage per student:"
)

print(
    student_skill_missingness[
        "missing_pct"
    ]
    .describe()
    .round(2)
)

print(
    "\nStudents with >25% missing skill:",
    (
        student_skill_missingness[
            "missing_pct"
        ] > 25
    ).sum()
)

print(
    "Students with >50% missing skill:",
    (
        student_skill_missingness[
            "missing_pct"
        ] > 50
    ).sum()
)

print(
    "Students with >75% missing skill:",
    (
        student_skill_missingness[
            "missing_pct"
        ] > 75
    ).sum()
)

Students with at least one missing-skill base interaction: 2105
Students with only missing-skill base interactions: 54
Students with no missing-skill base interactions: 2112

Missing-skill percentage per student:
count    4217.00
mean       12.96
std        19.61
min         0.00
25%         0.00
50%         0.00
75%        20.25
max       100.00
Name: missing_pct, dtype: float64

Students with >25% missing skill: 784
Students with >50% missing skill: 244
Students with >75% missing skill: 66


### Interpretation

Missing skill information affects Concept-Based Memory differently from general Student Memory.

An interaction with no known skill can still contribute to general behavioural history, but it cannot safely be assigned to a specific concept.

Therefore, the final system should distinguish between:

1. **General interaction memory**
   - Can retain interactions even when skill information is missing.

2. **Concept-Based Memory**
   - Uses only interactions with a known concept identifier.

Students with limited or completely missing concept information should not be classified as weak in unknown concepts.

Instead, the system should represent this as insufficient concept evidence.

No missing-skill imputation strategy is selected at this stage.

## 61. Multi-Skill Interactions

Earlier structural analysis showed that one base interaction can be associated with multiple mathematical skills.

This is important for Concept-Based Memory because collapsing every interaction to only one skill could discard valid concept information.

For interactions containing at least one known `skill_id`, we calculate the number of distinct skills associated with each:

`order_id`

Repeated opportunity representations of the same interaction-skill pair are removed first.

We examine:

- Number of interactions with known skill information
- Distribution of skills per interaction
- Percentage of interactions associated with multiple skills

No skill relationships are removed or collapsed.

In [51]:
known_skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(
        subset=["order_id", "skill_id"]
    )
)

skills_per_interaction = (
    known_skill_interactions
    .groupby("order_id")["skill_id"]
    .nunique()
)

print(
    "Interactions with at least one known skill:",
    len(skills_per_interaction)
)

print("\nNumber of skills per interaction:")

print(
    skills_per_interaction
    .value_counts()
    .sort_index()
)

print("\nPercentages:")

print(
    (
        skills_per_interaction
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

multi_skill_count = (
    skills_per_interaction > 1
).sum()

multi_skill_percentage = (
    skills_per_interaction > 1
).mean() * 100

print(
    "\nInteractions with >1 skill:",
    multi_skill_count
)

print(
    "Percentage multi-skill:",
    round(multi_skill_percentage, 2)
)

Interactions with at least one known skill: 283105

Number of skills per interaction:
skill_id
1    236068
2     40857
3      4501
4      1679
Name: count, dtype: int64

Percentages:
skill_id
1    83.39
2    14.43
3     1.59
4     0.59
Name: proportion, dtype: float64

Interactions with >1 skill: 47037
Percentage multi-skill: 16.61


### Interpretation

Multi-skill interactions require special treatment in the final memory architecture.

A single student interaction may contribute evidence to more than one concept.

Therefore, the preprocessing pipeline should not arbitrarily select only one skill when multiple valid skills are associated with the same interaction.

A suitable architecture may use:

1. A **base interaction table**
   - One record per `order_id`
   - Used for general student behavioural memory.

2. A **skill-aware interaction table**
   - One record per `order_id + skill_id`
   - Used for Concept-Based Memory.

This prevents multi-skill interactions from being counted multiple times in general behavioural statistics while still allowing the same interaction to update every relevant concept.

The final table design will be implemented during preprocessing.

## 62. Student-Level Accuracy Using Base Interactions

Overall student accuracy is likely to become one of the useful features for:

- Long-Term Memory
- Student Learning State prediction

However, raw dataset rows can overcount interactions because of multi-skill and opportunity representations.

Therefore, for this analysis we first create a base-interaction view using:

`one row per order_id`

We then calculate each student's overall correctness rate.

We examine:

- Distribution of student-level accuracy
- Number of students in broad accuracy ranges

No learning-state labels are created yet.

In [52]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

student_accuracy = (
    base_interactions
    .groupby("user_id")["correct"]
    .mean()
)

print("Student accuracy summary:")

print(
    student_accuracy
    .describe()
    .round(3)
)

accuracy_bins = [
    -0.001,
    0.25,
    0.50,
    0.75,
    1.00
]

accuracy_labels = [
    "<=25%",
    "25-50%",
    "50-75%",
    "75-100%"
]

student_accuracy_group = pd.cut(
    student_accuracy,
    bins=accuracy_bins,
    labels=accuracy_labels,
    include_lowest=True
)

print("\nStudents by accuracy band:")

print(
    student_accuracy_group
    .value_counts()
    .sort_index()
)

print("\nPercentages:")

print(
    (
        student_accuracy_group
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

Student accuracy summary:
count    4217.000
mean        0.623
std         0.255
min         0.000
25%         0.500
50%         0.667
75%         0.800
max         1.000
Name: correct, dtype: float64

Students by accuracy band:
correct
<=25%       419
25-50%      774
50-75%     1640
75-100%    1384
Name: count, dtype: int64

Percentages:
correct
<=25%       9.94
25-50%     18.35
50-75%     38.89
75-100%    32.82
Name: proportion, dtype: float64


### Interpretation

Student-level accuracy provides a useful summary of overall performance, but it should not be used alone to determine the student's current learning state.

For example, two students may have the same overall accuracy but differ in:

- Recent performance
- Hint usage
- Attempt behaviour
- Response time
- Concept-specific performance
- Improvement or decline

Therefore, overall accuracy is expected to be one feature among several rather than the complete learning-state definition.

No student is labelled as NEEDS_SUPPORT, DEVELOPING, or STRONG at this stage.

## 63. Student-Skill Accuracy

Overall student accuracy cannot describe performance on individual concepts.

A student may perform strongly overall while struggling with a particular mathematical skill.

We therefore calculate concept-level performance for each:

`user_id + skill_id`

The skill-aware interaction representation uses one row per:

`order_id + skill_id`

so repeated opportunity rows do not artificially increase the interaction count.

We examine:

- Accuracy across all student-skill pairs
- Number of interactions available for each pair
- Accuracy for pairs with at least 5 interactions
- Broad accuracy bands for these better-supported pairs

The 5-interaction condition is used only to examine how the distribution changes when more evidence is available. It is not a final system threshold.

In [53]:
skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(
        subset=["order_id", "skill_id"]
    )
)

student_skill_performance = (
    skill_interactions
    .groupby(["user_id", "skill_id"])
    .agg(
        interactions=("order_id", "size"),
        accuracy=("correct", "mean")
    )
)

print(
    "Student-skill pairs:",
    len(student_skill_performance)
)

print("\nConcept accuracy summary:")

print(
    student_skill_performance["accuracy"]
    .describe()
    .round(3)
)

supported_skill_performance = (
    student_skill_performance[
        student_skill_performance["interactions"] >= 5
    ]
)

print(
    "\nPairs with >=5 interactions:",
    len(supported_skill_performance)
)

print(
    "\nAccuracy summary for pairs "
    "with >=5 interactions:"
)

print(
    supported_skill_performance["accuracy"]
    .describe()
    .round(3)
)

concept_accuracy_bins = [
    -0.001,
    0.25,
    0.50,
    0.75,
    1.00
]

concept_accuracy_labels = [
    "<=25%",
    "25-50%",
    "50-75%",
    "75-100%"
]

concept_accuracy_groups = pd.cut(
    supported_skill_performance["accuracy"],
    bins=concept_accuracy_bins,
    labels=concept_accuracy_labels,
    include_lowest=True
)

print(
    "\nAccuracy bands for pairs "
    "with >=5 interactions:"
)

print(
    concept_accuracy_groups
    .value_counts()
    .sort_index()
)

print("\nPercentages:")

print(
    (
        concept_accuracy_groups
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

Student-skill pairs: 41982

Concept accuracy summary:
count    41982.000
mean         0.708
std          0.322
min          0.000
25%          0.520
50%          0.800
75%          1.000
max          1.000
Name: accuracy, dtype: float64

Pairs with >=5 interactions: 22133

Accuracy summary for pairs with >=5 interactions:
count    22133.000
mean         0.668
std          0.257
min          0.000
25%          0.529
50%          0.714
75%          0.857
max          1.000
Name: accuracy, dtype: float64

Accuracy bands for pairs with >=5 interactions:
accuracy
<=25%      1959
25-50%     3448
50-75%     7125
75-100%    9601
Name: count, dtype: int64

Percentages:
accuracy
<=25%       8.85
25-50%     15.58
50-75%     32.19
75-100%    43.38
Name: proportion, dtype: float64


### Interpretation

Concept-level accuracy provides more personalized information than overall student accuracy.

However, concept accuracy must always be interpreted together with the amount of available evidence.

For example:

- 100% accuracy from 1 interaction is weak evidence.
- 100% accuracy from 20 interactions is much stronger evidence.

Therefore, the final Concept-Based Memory should not store only a performance score.

It should also retain evidence information such as:

- Interaction count
- Correct and incorrect counts
- Accuracy
- Hint behaviour
- Attempt behaviour
- Response behaviour
- Confidence or evidence strength

The current `>=5` analysis is exploratory only and does not establish a minimum-history requirement or learning-state threshold.

## 64. Overall Performance vs Concept-Specific Performance

The personalization architecture separates:

1. General Student Memory
2. Concept-Based Memory

To investigate whether this separation is justified, we compare each student's overall accuracy with their accuracy on individual skills.

Overall accuracy is calculated from one base interaction per:

`order_id`

Concept accuracy is calculated from one skill-aware interaction per:

`order_id + skill_id`

For this comparison only, we examine student-skill pairs with at least 5 interactions so that concept accuracy has some supporting history.

For each supported concept, we calculate:

`concept accuracy - overall student accuracy`

We then examine concepts that are at least:

- 20 percentage points below the student's overall accuracy
- 20 percentage points above the student's overall accuracy

The 20-percentage-point difference is an exploratory comparison threshold, not a final learning-state rule.

In [54]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

overall_student_accuracy = (
    base_interactions
    .groupby("user_id")["correct"]
    .mean()
    .rename("overall_accuracy")
)

skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(
        subset=["order_id", "skill_id"]
    )
)

concept_performance = (
    skill_interactions
    .groupby(["user_id", "skill_id"])
    .agg(
        concept_interactions=("order_id", "size"),
        concept_accuracy=("correct", "mean")
    )
    .reset_index()
)

supported_concepts = (
    concept_performance[
        concept_performance["concept_interactions"] >= 5
    ]
    .merge(
        overall_student_accuracy,
        on="user_id",
        how="left"
    )
)

supported_concepts["accuracy_gap"] = (
    supported_concepts["concept_accuracy"]
    - supported_concepts["overall_accuracy"]
)

print(
    "Supported student-skill pairs:",
    len(supported_concepts)
)

print(
    "\nConcept minus overall accuracy gap:"
)

print(
    supported_concepts["accuracy_gap"]
    .describe()
    .round(3)
)

concept_below = (
    supported_concepts["accuracy_gap"] <= -0.20
)

concept_above = (
    supported_concepts["accuracy_gap"] >= 0.20
)

print(
    "\nConcept >=20 percentage points below overall:",
    concept_below.sum(),
    f"({concept_below.mean() * 100:.2f}%)"
)

print(
    "Concept >=20 percentage points above overall:",
    concept_above.sum(),
    f"({concept_above.mean() * 100:.2f}%)"
)

student_concept_variation = (
    supported_concepts
    .groupby("user_id")
    .agg(
        supported_concepts=("skill_id", "size"),
        has_lower_concept=(
            "accuracy_gap",
            lambda x: (x <= -0.20).any()
        ),
        has_higher_concept=(
            "accuracy_gap",
            lambda x: (x >= 0.20).any()
        )
    )
)

print(
    "\nStudents with supported concepts:",
    len(student_concept_variation)
)

print(
    "Students with >=1 concept 20pp below overall:",
    student_concept_variation[
        "has_lower_concept"
    ].sum(),
    f"({student_concept_variation['has_lower_concept'].mean() * 100:.2f}%)"
)

print(
    "Students with >=1 concept 20pp above overall:",
    student_concept_variation[
        "has_higher_concept"
    ].sum(),
    f"({student_concept_variation['has_higher_concept'].mean() * 100:.2f}%)"
)

Supported student-skill pairs: 22133

Concept minus overall accuracy gap:
count    22133.000
mean         0.006
std          0.191
min         -0.806
25%         -0.103
50%          0.013
75%          0.131
max          0.851
Name: accuracy_gap, dtype: float64

Concept >=20 percentage points below overall: 2951 (13.33%)
Concept >=20 percentage points above overall: 3057 (13.81%)

Students with supported concepts: 3393
Students with >=1 concept 20pp below overall: 1365 (40.23%)
Students with >=1 concept 20pp above overall: 1170 (34.48%)


### Interpretation

This analysis tests whether overall student performance is sufficient to represent performance across individual concepts.

If substantial concept-level differences exist, a single overall student score would hide important learning patterns.

For example, a student may:

- Perform strongly overall
- Struggle significantly with one mathematical concept
- Perform especially strongly on another concept

This would support maintaining separate General Student Memory and Concept-Based Memory.

The 20-percentage-point difference used here is only an exploratory comparison threshold.

It must not be interpreted as a final definition of struggling or strong performance.

## 65. Student-Level Hint Behaviour

Hint usage is one of the proposed signals for identifying whether a student needs additional support.

Interaction-level hint counts alone do not show whether a student frequently depends on hints across their learning history.

Using one base interaction per:

`order_id`

we calculate for each student:

- Number of base interactions
- Average hint count
- Number of interactions where at least one hint was used
- Proportion of interactions involving hint usage

The hint-usage rate is:

`hinted interactions / total base interactions`

We also examine broad hint-usage-rate bands.

These bands are descriptive only and are not learning-state thresholds.

In [55]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

student_hint_behaviour = (
    base_interactions
    .groupby("user_id")
    .agg(
        interactions=("order_id", "size"),
        avg_hint_count=("hint_count", "mean"),
        hinted_interactions=(
            "hint_count",
            lambda x: (x > 0).sum()
        )
    )
)

student_hint_behaviour["hint_usage_rate"] = (
    student_hint_behaviour["hinted_interactions"]
    / student_hint_behaviour["interactions"]
)

print("Average hint count per student:")

print(
    student_hint_behaviour["avg_hint_count"]
    .describe()
    .round(3)
)

print("\nHint-usage rate per student:")

print(
    student_hint_behaviour["hint_usage_rate"]
    .describe()
    .round(3)
)

print(
    "\nStudents who never used a hint:",
    (
        student_hint_behaviour[
            "hinted_interactions"
        ] == 0
    ).sum()
)

print(
    "Students who used a hint at least once:",
    (
        student_hint_behaviour[
            "hinted_interactions"
        ] > 0
    ).sum()
)

hint_rate_bins = [
    -0.001,
    0,
    0.10,
    0.25,
    0.50,
    0.75,
    1.00
]

hint_rate_labels = [
    "0%",
    "0-10%",
    "10-25%",
    "25-50%",
    "50-75%",
    "75-100%"
]

student_hint_groups = pd.cut(
    student_hint_behaviour["hint_usage_rate"],
    bins=hint_rate_bins,
    labels=hint_rate_labels,
    include_lowest=True
)

print("\nStudents by hint-usage rate:")

print(
    student_hint_groups
    .value_counts()
    .sort_index()
)

print("\nPercentages:")

print(
    (
        student_hint_groups
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

Average hint count per student:
count    4217.000
mean        0.583
std         0.723
min         0.000
25%         0.056
50%         0.333
75%         0.830
max         5.000
Name: avg_hint_count, dtype: float64

Hint-usage rate per student:
count    4217.000
mean        0.208
std         0.231
min         0.000
25%         0.032
50%         0.136
75%         0.300
max         1.000
Name: hint_usage_rate, dtype: float64

Students who never used a hint: 925
Students who used a hint at least once: 3292

Students by hint-usage rate:
hint_usage_rate
0%          925
0-10%       852
10-25%     1168
25-50%      849
50-75%      248
75-100%     175
Name: count, dtype: int64

Percentages:
hint_usage_rate
0%         21.94
0-10%      20.20
10-25%     27.70
25-50%     20.13
50-75%      5.88
75-100%     4.15
Name: proportion, dtype: float64


### Interpretation

Student-level hint behaviour provides information that is different from a single interaction's hint count.

For example, two students may both use three hints on one problem, but:

- One student may rarely use hints across their overall history.
- Another student may rely on hints frequently.

Therefore, the final Student Memory may benefit from storing both:

- Recent hint behaviour
- Longer-term hint usage patterns

However, hint usage must be interpreted together with correctness, attempts, problem context, and available hints.

Students who never use hints should not automatically be classified as strong, because absence of hint usage can have multiple explanations.

No hint-based learning-state threshold is selected at this stage.

## 66. Student-Level Attempt Behaviour

Attempt behaviour is another proposed signal for Student Learning State identification.

A single large attempt count does not describe a student's usual behaviour.

Therefore, using one base interaction per:

`order_id`

we calculate for each student:

- Average attempt count
- Median attempt count
- Number of interactions requiring more than one attempt
- Proportion of interactions requiring more than one attempt
- Number of interactions with more than 10 attempts

The multi-attempt rate is:

`interactions with attempt_count > 1 / total base interactions`

The `>10` condition is used only to track the extreme values identified during earlier dataset exploration.

No attempt values are modified.

In [56]:
base_interactions = df.drop_duplicates(
    subset=["order_id"]
)

student_attempt_behaviour = (
    base_interactions
    .groupby("user_id")
    .agg(
        interactions=("order_id", "size"),
        avg_attempts=("attempt_count", "mean"),
        median_attempts=("attempt_count", "median"),
        multi_attempt_interactions=(
            "attempt_count",
            lambda x: (x > 1).sum()
        ),
        extreme_attempt_interactions=(
            "attempt_count",
            lambda x: (x > 10).sum()
        )
    )
)

student_attempt_behaviour["multi_attempt_rate"] = (
    student_attempt_behaviour["multi_attempt_interactions"]
    / student_attempt_behaviour["interactions"]
)

student_attempt_behaviour["extreme_attempt_rate"] = (
    student_attempt_behaviour["extreme_attempt_interactions"]
    / student_attempt_behaviour["interactions"]
)

print("Average attempts per student:")
print(student_attempt_behaviour["avg_attempts"].describe().round(3))
print("\nMulti-attempt rate per student:")
print(student_attempt_behaviour["multi_attempt_rate"].describe().round(3))
print("\nStudents with no >1-attempt interactions:", (student_attempt_behaviour["multi_attempt_interactions"] == 0).sum())
print("Students with at least one >1-attempt interaction:", (student_attempt_behaviour["multi_attempt_interactions"] > 0).sum())
print("Students with at least one >10-attempt interaction:", (student_attempt_behaviour["extreme_attempt_interactions"] > 0).sum())

multi_attempt_bins = [-0.001, 0, 0.10, 0.25, 0.50, 0.75, 1.00]
multi_attempt_labels = ["0%", "0-10%", "10-25%", "25-50%", "50-75%", "75-100%"]
student_multi_attempt_groups = pd.cut(
    student_attempt_behaviour["multi_attempt_rate"],
    bins=multi_attempt_bins,
    labels=multi_attempt_labels,
    include_lowest=True
)
print("\nStudents by multi-attempt rate:")
print(student_multi_attempt_groups.value_counts().sort_index())
print("\nPercentages:")
print((student_multi_attempt_groups.value_counts(normalize=True).sort_index() * 100).round(2))

Average attempts per student:
count    4217.000
mean        2.369
std        28.802
min         0.000
25%         1.146
50%         1.376
75%         1.798
max      1853.000
Name: avg_attempts, dtype: float64

Multi-attempt rate per student:
count    4217.000
mean        0.217
std         0.173
min         0.000
25%         0.109
50%         0.196
75%         0.294
max         1.000
Name: multi_attempt_rate, dtype: float64

Students with no >1-attempt interactions: 614
Students with at least one >1-attempt interaction: 3603
Students with at least one >10-attempt interaction: 1166

Students by multi-attempt rate:
multi_attempt_rate
0%          614
0-10%       403
10-25%     1791
25-50%     1215
50-75%      132
75-100%      62
Name: count, dtype: int64

Percentages:
multi_attempt_rate
0%         14.56
0-10%       9.56
10-25%     42.47
25-50%     28.81
50-75%      3.13
75-100%     1.47
Name: proportion, dtype: float64


### Interpretation

Student-level attempt behaviour can distinguish between isolated difficult interactions and repeated patterns of needing multiple attempts.

For example, two students may both have one problem requiring several attempts, but their overall behaviour may be very different.

Therefore, the final Student Memory may benefit from storing both:

- Recent multi-attempt behaviour
- Longer-term multi-attempt rate

Extreme attempt counts should remain separately identifiable because earlier analysis showed values that are unlikely to behave like ordinary attempt counts.

No final clipping rule, transformation, or learning-state threshold is selected at this stage.

## 67. Student-Level Response-Time Behaviour

Response time is a proposed behavioural signal for Student Learning State identification.

Earlier dataset exploration identified:

- Negative response times
- Zero response times
- Extremely long response times

Because these values can heavily distort an arithmetic mean, this analysis focuses on each student's:

`median positive first-response time`

Only interactions where:

`ms_first_response > 0`

contribute to this median.

We separately retain information about:

- Zero response-time interactions
- Negative response-time interactions
- Responses longer than 5 minutes
- Responses longer than 1 hour

This allows typical response behaviour and unusual timing events to be studied separately.

No response-time values are modified.

In [57]:
base_interactions = df.drop_duplicates(subset=["order_id"])

positive_response_interactions = base_interactions[
    base_interactions["ms_first_response"] > 0
]

student_median_response = (
    positive_response_interactions.groupby("user_id")["ms_first_response"]
    .median().rename("median_positive_response_ms")
)

student_response_behaviour = (
    base_interactions.groupby("user_id").agg(
        interactions=("order_id", "size"),
        zero_response=("ms_first_response", lambda x: (x == 0).sum()),
        negative_response=("ms_first_response", lambda x: (x < 0).sum()),
        over_5min=("ms_first_response", lambda x: (x > 300_000).sum()),
        over_1hour=("ms_first_response", lambda x: (x > 3_600_000).sum())
    ).join(student_median_response)
)

print("Students with positive response-time evidence:", student_response_behaviour["median_positive_response_ms"].notna().sum())
print("Students with no positive response time:", student_response_behaviour["median_positive_response_ms"].isna().sum())
print("\nMedian positive response time per student (ms):")
print(student_response_behaviour["median_positive_response_ms"].describe().round(2))
print("\nStudents with at least one zero response:", (student_response_behaviour["zero_response"] > 0).sum())
print("Students with at least one negative response:", (student_response_behaviour["negative_response"] > 0).sum())
print("Students with at least one >5 minute response:", (student_response_behaviour["over_5min"] > 0).sum())
print("Students with at least one >1 hour response:", (student_response_behaviour["over_1hour"] > 0).sum())

response_median_bins = [0, 10_000, 30_000, 60_000, 300_000, float("inf")]
response_median_labels = ["<=10s", "10-30s", "30-60s", "1-5m", ">5m"]
student_response_groups = pd.cut(
    student_response_behaviour["median_positive_response_ms"],
    bins=response_median_bins, labels=response_median_labels, include_lowest=True
)
print("\nStudents by median positive response time:")
print(student_response_groups.value_counts().sort_index())
print("\nPercentages:")
print((student_response_groups.value_counts(normalize=True).sort_index() * 100).round(2))

Students with positive response-time evidence: 4217
Students with no positive response time: 0

Median positive response time per student (ms):
count      4217.00
mean      30239.49
std       37099.50
min         906.00
25%       12749.00
50%       20907.00
75%       34513.50
max      659687.00
Name: median_positive_response_ms, dtype: float64

Students with at least one zero response: 87
Students with at least one negative response: 6
Students with at least one >5 minute response: 1173
Students with at least one >1 hour response: 91

Students by median positive response time:
median_positive_response_ms
<=10s      667
10-30s    2247
30-60s     906
1-5m       384
>5m         13
Name: count, dtype: int64

Percentages:
median_positive_response_ms
<=10s     15.82
10-30s    53.28
30-60s    21.48
1-5m       9.11
>5m        0.31
Name: proportion, dtype: float64


### Interpretation

Student-level median response time provides a more robust description of typical response behaviour than raw mean response time when extreme values are present.

However, response speed must not be interpreted as learning ability by itself.

A fast student may be:

- Fluent
- Guessing
- Responding under a special interaction condition

A slower student may be:

- Struggling
- Reasoning carefully
- Interrupted
- Inactive before answering

Therefore, the final learning-state system should combine response behaviour with other evidence such as:

- Correctness
- Attempts
- Hint usage
- Recent performance
- Concept-specific performance

Unusual timing events should remain separately identifiable rather than being allowed to dominate the student's typical response-time feature.

No final response-time preprocessing rule is selected at this stage.

## 68. Searching for a Reliable Chronological Field

The Student Learning State system requires features such as:

- Recent accuracy
- Recent hint usage
- Recent attempt behaviour
- Improvement
- Decline

These features require a reliable ordering of student interactions.

Earlier analysis showed that `order_id` should not automatically be treated as chronological because interaction rows were not consistently ordered by this field.

Before constructing recent-history features, we therefore inspect all dataset columns whose names may represent:

- Time
- Date
- Timestamp
- Order
- Position
- Opportunity
- Sequence

The purpose is to determine whether the dataset contains a trustworthy field for reconstructing student interaction order.

No chronological ordering is assumed or created in this step.

In [58]:
chronology_keywords = [
    "time",
    "date",
    "timestamp",
    "order",
    "position",
    "opportunity",
    "sequence"
]

chronology_columns = [
    column
    for column in df.columns
    if any(
        keyword in column.lower()
        for keyword in chronology_keywords
    )
]

print("Columns that may contain ordering/time information:")
print(chronology_columns)
print("\nDtypes and samples:")

for column in chronology_columns:
    print(
        f"\n{column}"
        f"\n  dtype: {df[column].dtype}"
        f"\n  missing: {df[column].isna().sum()}"
        f"\n  sample: {df[column].dropna().head(5).tolist()}"
    )

Columns that may contain ordering/time information:
['order_id', 'sequence_id', 'position', 'base_sequence_id', 'overlap_time', 'opportunity', 'opportunity_original']

Dtypes and samples:

order_id
  dtype: int64
  missing: 0
  sample: [33022537, 33022709, 35450204, 35450295, 35450311]

sequence_id
  dtype: int64
  missing: 0
  sample: [5948, 5948, 5948, 5948, 5948]

position
  dtype: int64
  missing: 0
  sample: [126, 126, 22, 22, 22]

base_sequence_id
  dtype: int64
  missing: 0
  sample: [5948, 5948, 5948, 5948, 5948]

overlap_time
  dtype: int64
  missing: 0
  sample: [32454, 4922, 42000, 4859, 124564]

opportunity
  dtype: int64
  missing: 0
  sample: [1, 2, 1, 2, 3]

opportunity_original
  dtype: float64
  missing: 76314
  sample: [1.0, 2.0, 1.0, 2.0, 3.0]


### Interpretation

Recent-performance and trend features must be based on a defensible interaction ordering.

A numeric identifier should not automatically be interpreted as time, and fields such as `position`, `sequence_id`, or `opportunity` may represent structural information rather than chronological student activity.

Therefore, each candidate field must be understood before it is used to construct:

- Recent interaction windows
- Improvement features
- Decline features
- Short-Term Memory

If no reliable chronological field exists, the system will need a different evidence-based strategy rather than inventing an interaction timeline.

No recent-history feature is created at this stage.

## 69. Investigating `opportunity` as Concept Progression

The dataset does not contain an obvious timestamp or date field.

However, `opportunity` may contain ordered information about repeated practice of a skill.

We therefore investigate whether `opportunity` behaves consistently within each:

`user_id + skill_id`

Using one row per:

`order_id + skill_id`

we examine:

- Number of student-skill pairs
- Pairs containing multiple interactions
- Whether `opportunity` changes across repeated interactions
- Minimum and maximum opportunity values
- Whether opportunity values are non-decreasing in the dataset's existing row order

This analysis does **not** assume that `opportunity` is equivalent to chronological time.

Its purpose is to determine whether it may support concept-level progression analysis.

In [59]:
skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(subset=["order_id", "skill_id"])
)

opportunity_pair_stats = (
    skill_interactions.groupby(["user_id", "skill_id"])["opportunity"]
    .agg(["count", "nunique", "min", "max"])
)
print("Student-skill pairs:", len(opportunity_pair_stats))
multi_interaction_pairs = opportunity_pair_stats[opportunity_pair_stats["count"] > 1]
print("Pairs with >1 interaction:", len(multi_interaction_pairs))
changing_opportunity = multi_interaction_pairs["nunique"] > 1
print("\nPairs where opportunity has >1 unique value:", changing_opportunity.sum(), f"({changing_opportunity.mean() * 100:.2f}%)")
constant_opportunity = multi_interaction_pairs["nunique"] == 1
print("Pairs where opportunity never changes:", constant_opportunity.sum(), f"({constant_opportunity.mean() * 100:.2f}%)")
print("\nOpportunity range for multi-interaction pairs:")
display(multi_interaction_pairs[["min", "max", "nunique"]].describe().round(2))

pairs_checked = 0
non_decreasing_pairs = 0
non_monotonic_examples = []
for key, group in skill_interactions.groupby(["user_id", "skill_id"]):
    if len(group) <= 1:
        continue
    pairs_checked += 1
    values = group["opportunity"].tolist()
    if pd.Series(values).is_monotonic_increasing:
        non_decreasing_pairs += 1
    elif len(non_monotonic_examples) < 10:
        non_monotonic_examples.append((key, values[:20]))
print("\nMulti-interaction student-skill pairs checked:", pairs_checked)
print("Already non-decreasing in dataset row order:", non_decreasing_pairs)
print("Not non-decreasing:", pairs_checked - non_decreasing_pairs)
print("Percentage non-decreasing:", round(non_decreasing_pairs / pairs_checked * 100, 2))
print("\nExample non-monotonic pairs:")
for example in non_monotonic_examples:
    print(example)

Student-skill pairs: 41982
Pairs with >1 interaction: 33381

Pairs where opportunity has >1 unique value: 33381 (100.00%)
Pairs where opportunity never changes: 0 (0.00%)

Opportunity range for multi-interaction pairs:
           min       max   nunique
count  33381.0  33381.00  33381.00
mean       1.0     12.57      9.87
std        0.0     41.18     11.29
min        1.0      2.00      2.00
25%        1.0      4.00      4.00
50%        1.0      7.00      6.00
75%        1.0     13.00     12.00
max        1.0   3371.00    290.00

Multi-interaction student-skill pairs checked: 33381
Already non-decreasing in dataset row order: 33381
Not non-decreasing: 0
Percentage non-decreasing: 100.0

Example non-monotonic pairs:


### Interpretation

`opportunity` is being evaluated as a possible concept-progression variable rather than as a timestamp.

If opportunity values generally increase across repeated interactions for the same student and skill, they may provide useful ordering information for concept-level learning progression.

However, several distinctions remain important:

- Opportunity number is not automatically equivalent to elapsed time.
- Dataset row order is not automatically chronological.
- Large opportunity values may reflect prior practice history not fully represented in the current dataset.
- Multi-skill interactions may complicate progression interpretation.

Therefore, `opportunity` will only be used for recent/improvement features if further analysis supports a defensible ordering strategy.

No trend feature is created yet.

## 70. Opportunity Increment Consistency

The previous analysis showed that `opportunity` is non-decreasing for every repeated student-skill history.

We now investigate whether opportunity values increase cleanly by one:

`1 → 2 → 3 → 4 → ...`

or whether the observed progression contains:

- Repeated opportunity numbers
- Gaps larger than one
- Large jumps

For every student-skill pair with multiple interactions, consecutive opportunity values are compared.

This helps determine how precisely `opportunity` can be interpreted as an ordered concept-practice index.

No values are modified.

In [60]:
skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(subset=["order_id", "skill_id"])
)

opportunity_increment_records = []

for (user_id, skill_id), group in skill_interactions.groupby(["user_id", "skill_id"]):
    values = group["opportunity"].tolist()
    if len(values) <= 1:
        continue
    differences = pd.Series(values).diff().dropna()
    opportunity_increment_records.append({
        "user_id": user_id,
        "skill_id": skill_id,
        "interactions": len(values),
        "step_1": (differences == 1).sum(),
        "repeated": (differences == 0).sum(),
        "gaps": (differences > 1).sum(),
        "max_jump": differences.max()
    })

opportunity_increment_stats = pd.DataFrame(opportunity_increment_records)
only_clean_increment = ((opportunity_increment_stats["repeated"] == 0) & (opportunity_increment_stats["gaps"] == 0))
contains_repeated = opportunity_increment_stats["repeated"] > 0
contains_gaps = opportunity_increment_stats["gaps"] > 0

print("Multi-interaction pairs:", len(opportunity_increment_stats))
print("Pairs with only +1 increments:", only_clean_increment.sum())
print("Pairs containing repeated opportunity values:", contains_repeated.sum())
print("Pairs containing gaps >1:", contains_gaps.sum())
print("\nPercentages:")
print("Only +1:", round(only_clean_increment.mean() * 100, 2))
print("Repeated:", round(contains_repeated.mean() * 100, 2))
print("Gaps:", round(contains_gaps.mean() * 100, 2))
print("\nMaximum opportunity jump:", opportunity_increment_stats["max_jump"].max())

Multi-interaction pairs: 33381
Pairs with only +1 increments: 32812
Pairs containing repeated opportunity values: 0
Pairs containing gaps >1: 569

Percentages:
Only +1: 98.3
Repeated: 0.0
Gaps: 1.7

Maximum opportunity jump: 215.0


### Interpretation

This analysis checks the structural consistency of `opportunity` as a concept-practice progression variable.

If opportunity usually increments by one, it provides strong evidence for an ordered sequence of practice opportunities.

Repeated values or gaps do not necessarily make the field unusable, but they would require careful interpretation.

In particular, a gap may indicate that some earlier opportunities are not represented in the current interaction view or that the dataset uses additional opportunity-counting rules.

The final system should therefore use `opportunity` as an ordered concept-practice indicator only to the extent supported by these structural results.

It should not be interpreted as elapsed clock time.

## 71. Investigating Opportunity Gaps

The previous analysis showed that 98.3% of repeated student-skill histories increment cleanly by one.

However, 569 student-skill pairs contain at least one opportunity gap larger than one.

Before using `opportunity` for concept-progression features, we investigate these gaps in more detail.

For every transition where:

`next opportunity - previous opportunity > 1`

we record:

- Student
- Skill
- Previous opportunity number
- Next opportunity number
- Size of the jump

We then examine:

- Total number of gap events
- Number of affected student-skill pairs
- Number of affected students
- Number of affected skills
- Distribution of gap sizes
- Largest observed gaps

No gap is repaired or filled.

In [61]:
skill_interactions = (
    df.dropna(subset=["skill_id"])
    .drop_duplicates(subset=["order_id", "skill_id"])
)
opportunity_gap_records = []
for (user_id, skill_id), group in skill_interactions.groupby(["user_id", "skill_id"]):
    if len(group) <= 1:
        continue
    opportunities = group["opportunity"].tolist()
    differences = pd.Series(opportunities).diff()
    gap_positions = differences[differences > 1]
    for index, jump in gap_positions.items():
        opportunity_gap_records.append({
            "user_id": user_id,
            "skill_id": skill_id,
            "previous_opportunity": opportunities[index - 1],
            "next_opportunity": opportunities[index],
            "jump": jump
        })
opportunity_gaps = pd.DataFrame(opportunity_gap_records)
print("Total gap events:", len(opportunity_gaps))
print("Student-skill pairs with gaps:", opportunity_gaps[["user_id", "skill_id"]].drop_duplicates().shape[0])
print("Unique students affected:", opportunity_gaps["user_id"].nunique())
print("Unique skills affected:", opportunity_gaps["skill_id"].nunique())
print("\nGap size summary:")
print(opportunity_gaps["jump"].describe().round(2))
print("\nMost common jump sizes:")
print(opportunity_gaps["jump"].value_counts().head(15))
print("\nLargest gap events:")
display(opportunity_gaps.sort_values("jump", ascending=False).head(15))

Total gap events: 2482
Student-skill pairs with gaps: 569
Unique students affected: 111
Unique skills affected: 24

Gap size summary:
count    2482.00
mean       37.40
std        37.93
min         2.00
25%         5.00
50%        36.00
75%        46.00
max       215.00
Name: jump, dtype: float64

Most common jump sizes:
jump
46.0    323
2.0     244
4.0     189
3.0     181
62.0    142
37.0    140
20.0    139
93.0    134
26.0    132
60.0     98
97.0     98
43.0     94
5.0      90
25.0     65
44.0     48
Name: count, dtype: int64

Largest gap events:
 user_id  skill_id  previous_opportunity  next_opportunity  jump
   77118     279.0                  1486              1701 215.0
   77118     279.0                  1701              1916 215.0
   77118     279.0                   841              1056 215.0
   77118     279.0                  1056              1271 215.0
   77899     279.0                  3156              3371 215.0
   77899     279.0                  2941              31

### Interpretation

Opportunity gaps do not automatically indicate corrupted data.

They may indicate that some practice opportunities are not represented in the current dataset view or that opportunity counting continues across interactions that are not present in the extracted records.

Therefore, missing opportunity numbers should not be artificially generated or filled.

For concept progression, the observed opportunity values can still provide ordering information, but the system should distinguish between:

- Consecutively observed practice opportunities
- Progression containing unobserved opportunity gaps

This distinction may later contribute to the confidence of trend estimates.

No preprocessing decision is finalized at this stage.

## 72. Opportunity Values Within Repeated Interaction-Skill Rows

Earlier analysis identified repeated raw rows for the same:

`order_id + skill_id`

These repeated rows were not duplicate interactions. In some cases, the `opportunity` value changed across the rows.

This is important because concept progression should distinguish between:

- Multiple raw representations belonging to one interaction
- Separate student interactions over a skill

We therefore examine each known-skill `order_id + skill_id` and calculate:

- Number of raw rows
- Number of unique opportunity values
- Minimum opportunity
- Maximum opportunity
- Opportunity span within the interaction

This helps determine how much of the unusual opportunity progression originates from repeated representations inside a single base interaction.

No rows or opportunity values are modified.

In [62]:
raw_known_skill = df.dropna(subset=["skill_id"])
interaction_skill_opportunity = (
    raw_known_skill.groupby(["order_id", "skill_id"]).agg(
        rows=("opportunity", "size"),
        unique_opportunities=("opportunity", "nunique"),
        min_opportunity=("opportunity", "min"),
        max_opportunity=("opportunity", "max")
    )
)
repeated_interaction_skill = interaction_skill_opportunity[interaction_skill_opportunity["rows"] > 1].copy()
print("Known-skill order_id + skill_id groups:", len(interaction_skill_opportunity))
print("Groups with multiple raw rows:", len(repeated_interaction_skill))
print("\nRepeated groups with multiple opportunity values:", (repeated_interaction_skill["unique_opportunities"] > 1).sum())
print("Repeated groups with only one opportunity value:", (repeated_interaction_skill["unique_opportunities"] == 1).sum())
print("\nUnique opportunities per repeated interaction:")
print(repeated_interaction_skill["unique_opportunities"].describe().round(2))
repeated_interaction_skill["opportunity_span"] = repeated_interaction_skill["max_opportunity"] - repeated_interaction_skill["min_opportunity"]
print("\nOpportunity span per repeated interaction:")
print(repeated_interaction_skill["opportunity_span"].describe().round(2))
print("\nLargest repeated interaction groups:")
display(repeated_interaction_skill.sort_values("rows", ascending=False).head(15))

Known-skill order_id + skill_id groups: 338001
Groups with multiple raw rows: 3227

Repeated groups with multiple opportunity values: 3227
Repeated groups with only one opportunity value: 0

Unique opportunities per repeated interaction:
count    3227.00
mean       38.56
std        35.90
min         2.00
25%         8.50
50%        37.00
75%        47.00
max       215.00
Name: unique_opportunities, dtype: float64

Opportunity span per repeated interaction:
count    3227.00
mean       37.56
std        35.90
min         1.00
25%         7.50
50%        36.00
75%        46.00
max       214.00
Name: span, dtype: float64

Largest repeated interaction groups:
                   rows  unique_opportunities  min_opportunity  max_opportunity  span
order_id skill_id                                                                    
36386036 279.0      215                   215             1271             1485   214
36386028 279.0      215                   215             1056             1270 

### Interpretation

Repeated `order_id + skill_id` rows must not automatically be interpreted as separate student interactions.

If one interaction-skill pair contains many different opportunity values, then opportunity progression in the raw dataset partly reflects repeated representations inside the same interaction.

This means that concept-level trend construction should operate on a carefully defined interaction representation rather than directly on all raw rows.

The analysis also helps explain unusually large opportunity jumps observed after collapsing repeated interaction-skill rows.

The final progression representation will therefore preserve one record per interaction-skill pair while retaining enough opportunity information to reconstruct the appropriate concept ordering.

No final progression feature is created yet.

## 73. Continuity of Opportunity Ranges Across Interactions

The previous analysis showed that a single `order_id + skill_id` may contain many consecutive opportunity values.

Therefore, representing an interaction using only one opportunity value can create misleading progression gaps.

For each `user_id + skill_id + order_id` we now summarize the raw opportunity values as:

- Minimum opportunity
- Maximum opportunity
- Number of raw rows

Within each student-skill history, interaction ranges are ordered by their minimum opportunity.

For each consecutive pair of interaction ranges, we calculate:

`next minimum opportunity - previous maximum opportunity`

A value of:

- `1` means the ranges connect exactly
- `<= 0` indicates overlapping ranges
- `> 1` indicates an actual gap between observed ranges

This allows us to distinguish genuine missing progression from artifacts caused by collapsing repeated raw rows.

In [63]:
interaction_opportunity_ranges = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(
        min_opportunity=("opportunity", "min"),
        max_opportunity=("opportunity", "max"),
        raw_rows=("opportunity", "size")
    ).reset_index()
)
range_transition_records = []
for (user_id, skill_id), group in interaction_opportunity_ranges.groupby(["user_id", "skill_id"], sort=False):
    if len(group) <= 1:
        continue
    ordered_group = group.sort_values(["min_opportunity", "max_opportunity"])
    previous_max = ordered_group["max_opportunity"].shift(1)
    transition_delta = ordered_group["min_opportunity"] - previous_max
    for delta in transition_delta.dropna():
        range_transition_records.append(delta)
range_transition_delta = pd.Series(range_transition_records, dtype="float64")
exactly_consecutive = range_transition_delta == 1
overlapping_ranges = range_transition_delta <= 0
actual_gaps = range_transition_delta > 1
print("Transitions checked:", len(range_transition_delta))
print("Exactly consecutive (next min = previous max + 1):", exactly_consecutive.sum(), f"({exactly_consecutive.mean() * 100:.2f}%)")
print("Overlapping/repeated ranges (delta <= 0):", overlapping_ranges.sum(), f"({overlapping_ranges.mean() * 100:.2f}%)")
print("Gaps between ranges (delta > 1):", actual_gaps.sum(), f"({actual_gaps.mean() * 100:.2f}%)")
print("\nRange transition delta summary:")
print(range_transition_delta.describe().round(2))
print("\nLargest positive gaps:")
print(range_transition_delta[range_transition_delta > 1].sort_values(ascending=False).head(20).tolist())

Transitions checked: 296019
Exactly consecutive (next min = previous max + 1): 296019 (100.00%)
Overlapping/repeated ranges (delta <= 0): 0 (0.00%)
Gaps between ranges (delta > 1): 0 (0.00%)

Range transition delta summary:
count    296019.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
dtype: float64

Largest positive gaps:
[]


### Interpretation

Opportunity ranges provide a more faithful representation of concept progression than keeping only one opportunity value from each repeated interaction-skill group.

If consecutive interaction ranges usually connect with `next minimum = previous maximum + 1`, then many apparent gaps observed earlier were artifacts of collapsing repeated raw rows.

This would support representing each concept interaction with both:

- `opportunity_start`
- `opportunity_end`

rather than a single arbitrary opportunity value.

Any remaining gaps or overlapping ranges should be investigated separately before final trend features are created.

No improvement or decline feature is created yet.

## 74. Uniqueness of Concept Opportunity Ranges

The previous analysis demonstrated that consecutive concept-interaction opportunity ranges connect perfectly.

Before using these ranges to construct concept progression, we explicitly test whether each interaction occupies a unique location in the opportunity sequence.

For each `user_id + skill_id + order_id` we retain:

- `opportunity_start`
- `opportunity_end`
- Number of raw rows represented

We then test whether different interactions for the same student and skill share:

- The same opportunity start
- The same opportunity end
- The same complete opportunity range

We also verify that `opportunity_end >= opportunity_start` for every interaction.

This establishes whether opportunity ranges provide an unambiguous ordering of concept interactions.

In [64]:
concept_interaction_ranges = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(
        opportunity_start=("opportunity", "min"),
        opportunity_end=("opportunity", "max"),
        raw_rows=("opportunity", "size")
    ).reset_index()
)
duplicate_start = concept_interaction_ranges.duplicated(subset=["user_id", "skill_id", "opportunity_start"], keep=False)
duplicate_end = concept_interaction_ranges.duplicated(subset=["user_id", "skill_id", "opportunity_end"], keep=False)
duplicate_range = concept_interaction_ranges.duplicated(subset=["user_id", "skill_id", "opportunity_start", "opportunity_end"], keep=False)
invalid_range = concept_interaction_ranges["opportunity_end"] < concept_interaction_ranges["opportunity_start"]
print("Concept interactions:", len(concept_interaction_ranges))
print("\nInteractions sharing opportunity_start within student-skill:", duplicate_start.sum())
print("Interactions sharing opportunity_end within student-skill:", duplicate_end.sum())
print("Interactions sharing the same full opportunity range:", duplicate_range.sum())
print("Interactions where end < start:", invalid_range.sum())
single_opportunity = concept_interaction_ranges["opportunity_start"] == concept_interaction_ranges["opportunity_end"]
print("\nSingle-opportunity interactions:", single_opportunity.sum())
print("Multi-opportunity interactions:", (~single_opportunity).sum())
opportunity_range_width = concept_interaction_ranges["opportunity_end"] - concept_interaction_ranges["opportunity_start"] + 1
print("\nOpportunity-range width summary:")
print(opportunity_range_width.describe().round(2))

Concept interactions: 338001

Interactions sharing opportunity_start within student-skill: 0
Interactions sharing opportunity_end within student-skill: 0
Interactions sharing the same full opportunity range: 0
Interactions where end < start: 0

Single-opportunity interactions: 334774
Multi-opportunity interactions: 3227

Opportunity-range width summary:
count    338001.00
mean          1.36
std           5.06
min           1.00
25%           1.00
50%           1.00
75%           1.00
max         215.00
dtype: float64


### Interpretation

If opportunity starts and ends are unique within each student-skill history, the opportunity ranges provide an unambiguous concept-progression order.

This would allow concept interactions to be ordered using `opportunity_start` while retaining `opportunity_end` to represent interactions that span multiple raw opportunity rows.

This is preferable to using dataset row position or `order_id` as an assumed chronological field.

Importantly, this ordering represents **practice progression within a concept**, not elapsed real-world time.

If this structural test succeeds, the next analysis can begin investigating earlier-versus-recent concept performance without inventing a timestamp.

## 75. History Coverage for Concept Trend Windows

The dataset now provides a defensible ordering of interactions within each student-skill history using opportunity ranges.

The next goal is to investigate concept-level improvement and decline.

However, a trend comparison requires enough interactions to represent both:

- Earlier performance
- Recent performance

Instead of immediately selecting an arbitrary window size, we first measure how much dataset coverage would remain under several candidate window sizes.

For a window of size `w`, a student-skill pair needs at least `2 × w interactions` to compare two non-overlapping groups:

- Previous `w` interactions
- Most recent `w` interactions

We examine candidate window sizes of 2, 3, 4, 5, and 10.

This step evaluates data availability only.

No trend threshold or final window size is selected.

In [65]:
concept_interaction_ranges = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(
        opportunity_start=("opportunity", "min"),
        opportunity_end=("opportunity", "max")
    ).reset_index()
)
concept_history_lengths = concept_interaction_ranges.groupby(["user_id", "skill_id"]).size()
print("Student-skill histories:", len(concept_history_lengths))
print("\nHistory length summary:")
print(concept_history_lengths.describe().round(2))
candidate_windows = [2, 3, 4, 5, 10]
print("\nCoverage for recent-vs-earlier windows:")
for window_size in candidate_windows:
    minimum_required = 2 * window_size
    eligible_histories = (concept_history_lengths >= minimum_required).sum()
    coverage_percentage = eligible_histories / len(concept_history_lengths) * 100
    print(
        f"{window_size} recent + {window_size} earlier "
        f"(>= {minimum_required} interactions): {eligible_histories} "
        f"({coverage_percentage:.2f}%)"
    )

Student-skill histories: 41982

History length summary:
count    41982.00
mean         8.05
std         10.68
min          1.00
25%          2.00
50%          5.00
75%         10.00
max        290.00
dtype: float64

Coverage for recent-vs-earlier windows:
2 recent + 2 earlier (>= 4 interactions): 25374 (60.44%)
3 recent + 3 earlier (>= 6 interactions): 18928 (45.09%)
4 recent + 4 earlier (>= 8 interactions): 14386 (34.27%)
5 recent + 5 earlier (>= 10 interactions): 11389 (27.13%)
10 recent + 10 earlier (>= 20 interactions): 3746 (8.92%)


### Interpretation

There is a trade-off when selecting a concept trend window.

A smaller window:

- Covers more student-skill histories
- Reacts more quickly to changing performance
- May be more sensitive to random correct or incorrect answers

A larger window:

- Uses more evidence
- Produces a more stable estimate
- Excludes more students and concepts with limited history
- May react more slowly to genuine learning changes

Therefore, the final trend window should not be chosen only because one size is convenient.

Dataset coverage, statistical stability, and the intended real-time behaviour of the Student Learning State component must all be considered.

No final window size is selected in this step.

## 76. Concept Trend Behaviour Across Candidate Windows

The previous step measured how much student-skill history is available for different recent-versus-earlier window sizes.

We now examine how the resulting performance-change signal behaves.

For each eligible student-skill history, interactions are ordered using `opportunity_start`.

For a candidate window size `w`, we compare the previous `w` interactions with the most recent `w` interactions.

The trend value is `recent accuracy - previous accuracy`.

A positive value means recent accuracy is higher, a negative value means it is lower, and zero means the windows have equal accuracy.

Candidate windows examined: 2 + 2, 3 + 3, 4 + 4, and 5 + 5.

This is exploratory analysis only. Positive and negative differences are not yet final `Improving` or `Declining` learning-state definitions.

In [66]:
ordered_concept_interactions = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(
        opportunity_start=("opportunity", "min"),
        opportunity_end=("opportunity", "max"),
        correct=("correct", "first")
    ).reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
candidate_windows = [2, 3, 4, 5]
trend_results = {}
print("Trend comparison by window size:")
for window_size in candidate_windows:
    trend_gaps = []
    for _, group in ordered_concept_interactions.groupby(["user_id", "skill_id"], sort=False):
        if len(group) < 2 * window_size:
            continue
        recent_accuracy = group.tail(window_size)["correct"].mean()
        previous_accuracy = group.iloc[-2 * window_size:-window_size]["correct"].mean()
        trend_gaps.append(recent_accuracy - previous_accuracy)
    trend_series = pd.Series(trend_gaps, dtype="float64")
    trend_results[window_size] = trend_series
    improved = trend_series > 0
    unchanged = trend_series == 0
    declined = trend_series < 0
    print(f"\n--- Window {window_size}+{window_size} ---")
    print("Eligible histories:", len(trend_series))
    print("Mean gap:", round(trend_series.mean(), 3))
    print("Median gap:", round(trend_series.median(), 3))
    print("Std:", round(trend_series.std(), 3))
    print("Improved (gap > 0):", improved.sum(), f"({improved.mean() * 100:.2f}%)")
    print("No change (gap = 0):", unchanged.sum(), f"({unchanged.mean() * 100:.2f}%)")
    print("Declined (gap < 0):", declined.sum(), f"({declined.mean() * 100:.2f}%)")

Trend comparison by window size:

--- Window 2+2 ---
Eligible histories: 25374
Mean gap: 0.132
Median gap: 0.0
Std: 0.357
Improved (gap > 0): 8612 (33.94%)
No change (gap = 0): 14334 (56.49%)
Declined (gap < 0): 2428 (9.57%)

--- Window 3+3 ---
Eligible histories: 18928
Mean gap: 0.183
Median gap: 0.333
Std: 0.34
Improved (gap > 0): 9502 (50.20%)
No change (gap = 0): 7337 (38.76%)
Declined (gap < 0): 2089 (11.04%)

--- Window 4+4 ---
Eligible histories: 14386
Mean gap: 0.139
Median gap: 0.0
Std: 0.299
Improved (gap > 0): 7045 (48.97%)
No change (gap = 0): 5223 (36.31%)
Declined (gap < 0): 2118 (14.72%)

--- Window 5+5 ---
Eligible histories: 11389
Mean gap: 0.135
Median gap: 0.2
Std: 0.287
Improved (gap > 0): 6043 (53.06%)
No change (gap = 0): 3368 (29.57%)
Declined (gap < 0): 1978 (17.37%)


### Interpretation

Different trend-window sizes provide different balances between responsiveness, stability, and dataset coverage.

Small windows can react quickly to changes but may classify a single correct or incorrect response as a large performance shift.

Larger windows reduce the influence of individual responses but require substantially more concept history.

The distribution of positive, zero, and negative changes helps us understand how sensitive each candidate window is.

A final trend feature should consider history coverage, stability, magnitude of change, amount of evidence, and suitability for real-time personalization.

A positive difference alone should not automatically imply that a student is in the `Improving` state.

No final trend window or learning-state rule is selected here.

## 77. Trend-Direction Agreement Across Window Sizes

The previous analysis showed that different recent-versus-earlier window sizes produce different trend distributions.

We now test whether the **direction** of the trend is stable when the window size changes.

To make the comparison fair, we use only student-skill histories containing at least 10 concept interactions.

This allows every history to support all four candidate windows: 2 + 2, 3 + 3, 4 + 4, and 5 + 5.

For each window, the accuracy difference is converted into an exploratory direction:

- `1` = positive difference
- `0` = no difference
- `-1` = negative difference

These values are used only to compare window stability. They are **not** final learning-state labels.

We examine agreement across all four windows, pairwise agreement, and the most common direction patterns.

In [67]:
ordered_concept_interactions = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
trend_direction_records = []
for (user_id, skill_id), group in ordered_concept_interactions.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 10:
        continue
    record = {"user_id": user_id, "skill_id": skill_id}
    for window_size in [2, 3, 4, 5]:
        recent_accuracy = group.tail(window_size)["correct"].mean()
        previous_accuracy = group.iloc[-2 * window_size:-window_size]["correct"].mean()
        gap = recent_accuracy - previous_accuracy
        direction = 1 if gap > 0 else (-1 if gap < 0 else 0)
        record[f"window_{window_size}"] = direction
    trend_direction_records.append(record)
trend_direction_comparison = pd.DataFrame(trend_direction_records)
direction_columns = ["window_2", "window_3", "window_4", "window_5"]
print("Histories compared across all four windows:", len(trend_direction_comparison))
all_same_direction = trend_direction_comparison[direction_columns].nunique(axis=1) == 1
print("\nDirection agreement across all four windows:")
print("All four same direction:", all_same_direction.sum(), f"({all_same_direction.mean() * 100:.2f}%)")
print("At least one disagreement:", (~all_same_direction).sum(), f"({(~all_same_direction).mean() * 100:.2f}%)")
print("\nPairwise direction agreement:")
for first, second in [(2, 3), (3, 4), (4, 5), (2, 5)]:
    agreement = trend_direction_comparison[f"window_{first}"] == trend_direction_comparison[f"window_{second}"]
    print(f"{first}+{first} vs {second}+{second}: {agreement.sum()} ({agreement.mean() * 100:.2f}%)")
print("\nMost common direction patterns:")
print(trend_direction_comparison[direction_columns].value_counts().head(15))

Histories compared across all four windows: 11389

Direction agreement across all four windows:
All four same direction: 2756 (24.20%)
At least one disagreement: 8633 (75.80%)

Pairwise direction agreement:
2+2 vs 3+3: 6638 (58.28%)
3+3 vs 4+4: 7848 (68.91%)
4+4 vs 5+5: 7939 (69.71%)
2+2 vs 5+5: 3994 (35.07%)

Most common direction patterns:
w2  w3  w4  w5
 0   1   1   1    2402
     0   0   0    1702
         1   1     911
 1   1   1   1     845
 0   0   0   1     725
 1   1   0   0     305
             1     304
 0   1   1   0     276
 1   1   1   0     253
        -1  -1     219
 0  -1  -1  -1     214
-1  -1  -1  -1     209
 1   1   0  -1     203
 0   1   1  -1     181
     0  -1  -1     159
Name: count, dtype: int64


### Interpretation

A useful concept-trend feature should not change direction too easily simply because a slightly different window size is selected.

High agreement would suggest that the trend signal is relatively robust. Low agreement would indicate that simple recent-versus-previous accuracy is sensitive to window selection and should not be used alone to assign an `Improving` state.

This analysis is particularly important because correctness is binary. With small windows, one answer can substantially change the calculated accuracy difference.

The final trend design may therefore need to combine direction of change, magnitude of change, number of supporting interactions, recent correctness pattern, and concept-level evidence strength.

No final trend window or state rule is selected in this step.

## 78. Magnitude of Concept Trend Changes

Trend direction alone was found to be sensitive to the selected window size.

We therefore investigate the **magnitude** of recent-versus-previous accuracy changes.

For each candidate window—2 + 2, 3 + 3, 4 + 4, and 5 + 5—we calculate `trend gap = recent accuracy - previous accuracy`.

We examine the absolute trend-gap distribution, exact observed values, and frequency of each possible change.

Because correctness is binary, small windows can only produce a limited set of discrete accuracy differences.

Understanding these possible magnitudes is necessary before deciding whether a change is large enough to provide meaningful evidence of improvement or decline.

No magnitude threshold is selected in this step.

In [68]:
ordered_concept_interactions = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
print("Trend-gap magnitude distributions:")
for window_size in [2, 3, 4, 5]:
    trend_gaps = []
    for _, group in ordered_concept_interactions.groupby(["user_id", "skill_id"], sort=False):
        if len(group) < 2 * window_size:
            continue
        recent_accuracy = group.tail(window_size)["correct"].mean()
        previous_accuracy = group.iloc[-2 * window_size:-window_size]["correct"].mean()
        trend_gaps.append(recent_accuracy - previous_accuracy)
    trend_series = pd.Series(trend_gaps, dtype="float64")
    print(f"\n--- Window {window_size}+{window_size} ---")
    print("Eligible:", len(trend_series))
    print("\nAbsolute gap summary:")
    print(trend_series.abs().describe().round(3))
    print("\nGap value distribution:")
    print(trend_series.round(3).value_counts().sort_index())

Trend-gap magnitude distributions:

--- Window 2+2 ---
Eligible: 25374

Absolute gap summary:
count    25374.000
mean         0.242
std          0.294
min          0.000
25%          0.000
50%          0.000
75%          0.500
max          1.000
dtype: float64

Gap value distribution:
-1.0      352
-0.5     2076
 0.0    14334
 0.5     7735
 1.0      877

--- Window 3+3 ---
Eligible: 18928

Absolute gap summary:
count    18928.000
mean         0.277
std          0.269
min          0.000
25%          0.000
50%          0.333
75%          0.333
max          1.000
dtype: float64

Gap value distribution:
-1.000      73
-0.667     414
-0.333    1602
 0.000    7337
 0.333    6589
 0.667    2273
 1.000     640

--- Window 4+4 ---
Eligible: 14386

Absolute gap summary:
count    14386.000
mean         0.238
std          0.229
min          0.000
25%          0.000
50%          0.250
75%          0.250
max          1.000
dtype: float64

Gap value distribution:
-1.00      16
-0.75     120
-0.50    

### Interpretation

With binary correctness, window-based accuracy changes occur in discrete steps.

For example, a 2-interaction window can only produce accuracy values such as 0%, 50%, and 100%. Therefore, even one changed answer can produce a large apparent trend.

As window size increases, the possible accuracy changes become more granular and less dependent on a single response.

The magnitude distribution helps determine whether future trend detection should require more than simply `trend gap > 0`.

A reliable improvement or decline signal may need to combine magnitude of change, direction consistency, history length, recent correctness pattern, and other behavioural signals.

No final magnitude threshold or learning-state rule is selected yet.

## 79. Persistence of Recent Concept Correctness

Fixed-window accuracy differences can be sensitive to individual answers.

We therefore inspect the actual correctness pattern inside recent concept interactions.

For every student-skill history with at least 6 interactions, interactions are ordered using `opportunity_start`.

We divide the latest six interactions into previous 3 interactions and recent 3 interactions.

Each three-interaction block is represented as a binary pattern, where `0` is incorrect and `1` is correct. Possible patterns are `000`, `001`, `010`, `011`, `100`, `101`, `110`, and `111`.

For example, `000 → 111` represents three incorrect responses followed by three correct responses.

This analysis examines whether recent performance is consistently correct, consistently incorrect, or mixed rather than relying only on an aggregate accuracy difference.

No learning-state label is assigned.

In [69]:
ordered_concept_interactions = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
recent_pattern_records = []
for (user_id, skill_id), group in ordered_concept_interactions.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 6:
        continue
    previous_correctness = group.iloc[-6:-3]["correct"].tolist()
    recent_correctness = group.tail(3)["correct"].tolist()
    recent_pattern_records.append({
        "user_id": user_id,
        "skill_id": skill_id,
        "previous_pattern": "".join(map(str, previous_correctness)),
        "recent_pattern": "".join(map(str, recent_correctness)),
        "previous_accuracy": sum(previous_correctness) / 3,
        "recent_accuracy": sum(recent_correctness) / 3
    })
recent_pattern_analysis = pd.DataFrame(recent_pattern_records)
print("Eligible histories:", len(recent_pattern_analysis))
print("\nRecent 3-answer patterns:")
print(recent_pattern_analysis["recent_pattern"].value_counts())
print("\nPercentages:")
print((recent_pattern_analysis["recent_pattern"].value_counts(normalize=True) * 100).round(2))
print("\nPrevious 3-answer patterns:")
print(recent_pattern_analysis["previous_pattern"].value_counts())
print("\nTransitions from previous pattern to recent pattern:")
display(pd.crosstab(recent_pattern_analysis["previous_pattern"], recent_pattern_analysis["recent_pattern"]))

Eligible histories: 18928

Recent 3-answer patterns:
recent_pattern
111    12583
011     1570
000     1468
101      959
110      709
001      686
010      486
100      467
Name: count, dtype: int64

Percentages:
recent_pattern
111    66.48
011     8.29
000     7.76
101     5.07
110     3.75
001     3.62
010     2.57
100     2.47
Name: proportion, dtype: float64

Previous 3-answer patterns:
previous_pattern
111    5708
011    3760
000    2334
110    2114
101    1574
010    1189
001    1131
100    1118
Name: count, dtype: int64

Transitions from previous pattern to recent pattern:
recent_pattern    000  001  010  011  100  101  110   111
previous_pattern                                         
000               932  222  108  202   77   96   57   640
001                89   58   56  140   50  105   77   556
010                79   67   45  113   34   70   37   744
011                44   62   50  235   79  170   93  3027
100               114   62   53  120   40   59   52   618
101     

### Interpretation

Recent correctness patterns provide information that is hidden by an accuracy score alone.

For example, `111` is consistently correct recently, `000` is consistently incorrect, `101` is mixed, `011` contains two correct responses after an incorrect response, and `100` contains two incorrect responses after a correct response.

However, three answers are still limited evidence. A pattern such as `111` should therefore not automatically mean that the student is `Strong`, and `000` should not automatically mean `Struggling`.

These patterns may instead become supporting features alongside recent accuracy, longer-term accuracy, hint behaviour, attempt behaviour, concept performance, trend magnitude, and evidence count.

No final learning-state rule is selected in this step.

## 80. Previous-to-Recent Concept Performance Transitions

The previous analysis examined exact three-answer correctness patterns.

We now summarize each three-interaction block by the number of correct responses: 0, 1, 2, or 3 correct out of 3.

For each student-skill history containing at least six interactions, we compare `previous 3 correct count → recent 3 correct count`.

We also calculate `correct change = recent correct count - previous correct count`. Possible changes range from `-3` to `+3`.

This separates the magnitude of recent performance movement from the exact order of correct and incorrect responses.

Changes of at least two correct answers are also counted for exploratory purposes. They are not final definitions of improvement or decline.

In [70]:
ordered_concept_interactions = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
performance_transition_records = []
for (user_id, skill_id), group in ordered_concept_interactions.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 6:
        continue
    previous_correct = int(group.iloc[-6:-3]["correct"].sum())
    recent_correct = int(group.tail(3)["correct"].sum())
    performance_transition_records.append({
        "user_id": user_id, "skill_id": skill_id,
        "previous_correct": previous_correct, "recent_correct": recent_correct,
        "correct_change": recent_correct - previous_correct
    })
performance_transitions = pd.DataFrame(performance_transition_records)
print("Eligible histories:", len(performance_transitions))
print("\nPrevious correct count -> recent correct count:")
display(pd.crosstab(performance_transitions["previous_correct"], performance_transitions["recent_correct"]))
print("\nChange in number correct:")
print(performance_transitions["correct_change"].value_counts().sort_index())
print("\nPercentages:")
print((performance_transitions["correct_change"].value_counts(normalize=True).sort_index() * 100).round(2))
strong_upward = performance_transitions["correct_change"] >= 2
strong_downward = performance_transitions["correct_change"] <= -2
print("\nStrong upward transitions (increase >=2 correct answers):", strong_upward.sum(), f"({strong_upward.mean() * 100:.2f}%)")
print("Strong downward transitions (decrease >=2 correct answers):", strong_downward.sum(), f"({strong_downward.mean() * 100:.2f}%)")

Eligible histories: 18928

Previous correct count -> recent correct count:
recent_correct      0    1     2     3
previous_correct                      
0                 932  407   355   640
1                 282  465   773  1918
2                 181  534  1324  5409
3                  73  233   786  4616

Change in number correct:
correct_change
-3      73
-2     414
-1    1602
 0    7337
 1    6589
 2    2273
 3     640
Name: count, dtype: int64

Percentages:
correct_change
-3     0.39
-2     2.19
-1     8.46
 0    38.76
 1    34.81
 2    12.01
 3     3.38
Name: proportion, dtype: float64

Strong upward transitions (increase >=2 correct answers): 2913 (15.39%)
Strong downward transitions (decrease >=2 correct answers): 487 (2.57%)


### Interpretation

Summarizing recent and previous performance by correct count provides a simple measure of how strongly concept performance has moved.

A one-answer difference may be relatively weak evidence because only six interactions are being compared.

Larger upward movements such as `0/3 → 2/3`, `0/3 → 3/3`, and `1/3 → 3/3` provide stronger evidence of upward performance movement.

Likewise, `3/3 → 1/3`, `3/3 → 0/3`, and `2/3 → 0/3` provide stronger evidence of downward movement.

However, these are still short histories. The final learning-state engine should not classify a student using this transition alone. It can instead become one trend-related signal combined with performance level, attempts, hints, response behaviour, and evidence strength.

No final `Improving` or decline rule is selected in this step.

## 81. Persistence of Strong Short-Term Concept Changes

The previous analysis identified strong short-term performance movements using the 3+3 comparison.

A strong upward movement was defined for exploratory analysis as `recent correct count - previous correct count >= 2`. A strong downward movement was `<= -2`.

However, a six-interaction comparison may still reflect a temporary fluctuation.

We therefore test whether these strong short-term movements are supported by a longer 5+5 comparison. Only student-skill histories containing at least 10 interactions are used.

For each eligible history, we calculate the 3+3 correct-count change and the 5+5 accuracy difference, then examine whether strong upward or downward movements have the same direction under the longer view.

This is a persistence analysis only. No final trend rule is created.

In [71]:
ordered_concept_interactions = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
persistence_records = []
for (user_id, skill_id), group in ordered_concept_interactions.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 10:
        continue
    previous_3_correct = group.iloc[-6:-3]["correct"].sum()
    recent_3_correct = group.tail(3)["correct"].sum()
    change_3 = recent_3_correct - previous_3_correct
    previous_5_accuracy = group.iloc[-10:-5]["correct"].mean()
    recent_5_accuracy = group.tail(5)["correct"].mean()
    gap_5 = recent_5_accuracy - previous_5_accuracy
    persistence_records.append({"user_id": user_id, "skill_id": skill_id, "change_3": change_3, "gap_5": gap_5})
trend_persistence = pd.DataFrame(persistence_records)
strong_upward = trend_persistence[trend_persistence["change_3"] >= 2]
strong_downward = trend_persistence[trend_persistence["change_3"] <= -2]
print("Histories with >=10 interactions:", len(trend_persistence))
print("\nStrong 3+3 upward cases:", len(strong_upward))
print("5+5 also positive:", (strong_upward["gap_5"] > 0).sum(), f"({(strong_upward['gap_5'] > 0).mean() * 100:.2f}%)")
print("5+5 zero:", (strong_upward["gap_5"] == 0).sum(), f"({(strong_upward['gap_5'] == 0).mean() * 100:.2f}%)")
print("5+5 negative:", (strong_upward["gap_5"] < 0).sum(), f"({(strong_upward['gap_5'] < 0).mean() * 100:.2f}%)")
print("\nStrong 3+3 downward cases:", len(strong_downward))
print("5+5 also negative:", (strong_downward["gap_5"] < 0).sum(), f"({(strong_downward['gap_5'] < 0).mean() * 100:.2f}%)")
print("5+5 zero:", (strong_downward["gap_5"] == 0).sum(), f"({(strong_downward['gap_5'] == 0).mean() * 100:.2f}%)")
print("5+5 positive:", (strong_downward["gap_5"] > 0).sum(), f"({(strong_downward['gap_5'] > 0).mean() * 100:.2f}%)")

Histories with >=10 interactions: 11389

Strong 3+3 upward cases: 1552
5+5 also positive: 943 (60.76%)
5+5 zero: 344 (22.16%)
5+5 negative: 265 (17.07%)

Strong 3+3 downward cases: 311
5+5 also negative: 201 (64.63%)
5+5 zero: 65 (20.90%)
5+5 positive: 45 (14.47%)


### Interpretation

A strong short-term change becomes more convincing when a longer performance window supports the same direction.

Strong positive 3+3 change plus positive 5+5 change provides stronger evidence of improvement. Strong negative 3+3 change plus negative 5+5 change provides stronger evidence of decline. Opposite directions suggest the short-term movement may be temporary or unstable.

This supports using multiple levels of evidence rather than relying on one fixed trend window. The 3+3 comparison can provide responsiveness, while a longer window can provide support when enough history exists.

No final `Improving` rule is selected yet.

## 82. Relationship Between Concept Accuracy Change and Hint Change

Learning improvement may be reflected not only by increased correctness but also by reduced dependence on hints.

For every student-skill history containing at least six concept interactions, the latest six interactions are divided into previous 3 and recent 3 interactions.

For each block, we calculate mean correctness and mean hint count. We then calculate `accuracy change = recent accuracy - previous accuracy` and `hint change = recent average hints - previous average hints`.

This allows us to examine combinations such as accuracy increasing while hints decrease, remain unchanged, or increase, and accuracy decreasing while hints increase.

The correlation between accuracy change and hint change is also examined.

This investigates whether hint behaviour can provide supporting evidence for learning trends. No learning-state rule is created.

In [72]:
ordered_concept_behaviour = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), hint_count=("hint_count", "first"))
    .reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
accuracy_hint_records = []
for (user_id, skill_id), group in ordered_concept_behaviour.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 6:
        continue
    previous = group.iloc[-6:-3]
    recent = group.tail(3)
    previous_accuracy = previous["correct"].mean()
    recent_accuracy = recent["correct"].mean()
    previous_avg_hints = previous["hint_count"].mean()
    recent_avg_hints = recent["hint_count"].mean()
    accuracy_hint_records.append({
        "user_id": user_id, "skill_id": skill_id,
        "accuracy_change": recent_accuracy - previous_accuracy,
        "hint_change": recent_avg_hints - previous_avg_hints,
        "previous_avg_hints": previous_avg_hints, "recent_avg_hints": recent_avg_hints
    })
accuracy_hint_change = pd.DataFrame(accuracy_hint_records)
accuracy_up = accuracy_hint_change["accuracy_change"] > 0
accuracy_down = accuracy_hint_change["accuracy_change"] < 0
hints_up = accuracy_hint_change["hint_change"] > 0
hints_down = accuracy_hint_change["hint_change"] < 0
hints_same = accuracy_hint_change["hint_change"] == 0
print("Eligible histories:", len(accuracy_hint_change))
conditions = [("Accuracy up + hints down", accuracy_up & hints_down), ("Accuracy up + hints same", accuracy_up & hints_same), ("Accuracy up + hints up", accuracy_up & hints_up), ("Accuracy down + hints up", accuracy_down & hints_up), ("Accuracy down + hints same", accuracy_down & hints_same), ("Accuracy down + hints down", accuracy_down & hints_down)]
for label, condition in conditions:
    print(f"{label}:", condition.sum(), f"({condition.mean() * 100:.2f}%)")
print("\nHint-change summary:")
print(accuracy_hint_change["hint_change"].describe().round(3))
print("\nCorrelation between accuracy change and hint change:", round(accuracy_hint_change[["accuracy_change", "hint_change"]].corr().iloc[0, 1], 3))

Eligible histories: 18928

Accuracy up + hints down: 3504 (18.51%)
Accuracy up + hints same: 5868 (31.00%)
Accuracy up + hints up: 130 (0.69%)
Accuracy down + hints up: 820 (4.33%)
Accuracy down + hints same: 1156 (6.11%)
Accuracy down + hints down: 113 (0.60%)

Hint-change summary:
count    18928.000
mean        -0.183
std          0.756
min         -7.000
25%          0.000
50%          0.000
75%          0.000
max          5.333
Name: hint_change, dtype: float64

Correlation between accuracy change and hint change: -0.502


### Interpretation

Correctness improvement accompanied by reduced hint usage can provide stronger behavioural evidence than correctness improvement alone.

Likewise, decreasing correctness together with increasing hint usage may indicate stronger evidence of difficulty.

However, unchanged hint usage does not necessarily mean that no learning occurred because many interactions involve no hints at all.

Hint behaviour should therefore be treated as a supporting signal rather than a mandatory condition for improvement.

The relationship between correctness and hints will later be considered together with attempt behaviour, response-time behaviour, recent accuracy, concept accuracy, trend evidence, and available history.

No final state thresholds or rules are selected at this stage.

## 83. Relationship Between Concept Accuracy Change and Attempt Change

Correctness is not the only behavioural signal that may reflect changes in a student's learning state.

A student may become more independent if recent concept interactions require fewer attempts while correctness improves.

For every student-skill history containing at least six interactions, the latest six interactions are divided into previous 3 and recent 3 interactions.

For each block, we calculate mean correctness and mean attempt count, then calculate `accuracy change = recent accuracy - previous accuracy` and `attempt change = recent average attempts - previous average attempts`.

This allows us to examine whether accuracy increases or decreases while attempts decrease, remain unchanged, or increase. The correlation between both changes is also examined.

This is exploratory analysis only.

In [73]:
ordered_concept_attempts = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), attempt_count=("attempt_count", "first"))
    .reset_index()
    .sort_values(["user_id", "skill_id", "opportunity_start"])
)
accuracy_attempt_records = []
for (user_id, skill_id), group in ordered_concept_attempts.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 6:
        continue
    previous = group.iloc[-6:-3]
    recent = group.tail(3)
    previous_accuracy = previous["correct"].mean()
    recent_accuracy = recent["correct"].mean()
    previous_avg_attempts = previous["attempt_count"].mean()
    recent_avg_attempts = recent["attempt_count"].mean()
    accuracy_attempt_records.append({
        "user_id": user_id, "skill_id": skill_id,
        "accuracy_change": recent_accuracy - previous_accuracy,
        "attempt_change": recent_avg_attempts - previous_avg_attempts,
        "previous_avg_attempts": previous_avg_attempts, "recent_avg_attempts": recent_avg_attempts
    })
accuracy_attempt_change = pd.DataFrame(accuracy_attempt_records)
accuracy_up = accuracy_attempt_change["accuracy_change"] > 0
accuracy_down = accuracy_attempt_change["accuracy_change"] < 0
attempts_up = accuracy_attempt_change["attempt_change"] > 0
attempts_down = accuracy_attempt_change["attempt_change"] < 0
attempts_same = accuracy_attempt_change["attempt_change"] == 0
conditions = [("Accuracy up + attempts down", accuracy_up & attempts_down), ("Accuracy up + attempts same", accuracy_up & attempts_same), ("Accuracy up + attempts up", accuracy_up & attempts_up), ("Accuracy down + attempts up", accuracy_down & attempts_up), ("Accuracy down + attempts same", accuracy_down & attempts_same), ("Accuracy down + attempts down", accuracy_down & attempts_down)]
print("Eligible histories:", len(accuracy_attempt_change))
for label, condition in conditions:
    print(f"{label}:", condition.sum(), f"({condition.mean() * 100:.2f}%)")
print("\nAttempt-change summary:")
print(accuracy_attempt_change["attempt_change"].describe().round(3))
print("\nCorrelation between accuracy change and attempt change:", round(accuracy_attempt_change[["accuracy_change", "attempt_change"]].corr().iloc[0, 1], 3))

Eligible histories: 18928

Accuracy up + attempts down: 6022 (31.82%)
Accuracy up + attempts same: 2912 (15.38%)
Accuracy up + attempts up: 568 (3.00%)
Accuracy down + attempts up: 1163 (6.14%)
Accuracy down + attempts same: 618 (3.27%)
Accuracy down + attempts down: 308 (1.63%)

Attempt-change summary:
count    18928.000
mean        -0.124
std          9.772
min       -109.333
25%         -0.333
50%          0.000
75%          0.000
max       1246.333
Name: attempt_change, dtype: float64

Correlation between accuracy change and attempt change: -0.073


### Interpretation

Attempt behaviour can provide additional evidence about changes in concept performance.

An increase in correctness accompanied by fewer attempts may indicate stronger improvement than increased correctness alone. Likewise, decreasing correctness accompanied by more attempts may provide stronger evidence of difficulty.

However, attempt counts contain unusual and extreme values in this dataset, as identified earlier. Therefore, raw attempt-count changes should not automatically be trusted as a final feature.

This analysis determines whether attempt behaviour contains useful trend information before later preprocessing and robust feature construction.

No outliers are removed and no final learning-state rule is selected in this step.

## 84. Sensitivity of Attempt Trends to Extreme Attempt Counts

The previous analysis showed that concept accuracy change and attempt change have only a weak overall correlation.

However, attempt counts contain extreme values, which may strongly influence mean-based attempt changes.

We therefore repeat the latest-six-interaction 3+3 analysis with no restriction and with all six attempt counts limited to exploratory cutoffs of 10, 5, and 3.

For each condition, we examine eligible histories, correlation, accuracy increase with attempt decrease, accuracy decrease with attempt increase, and attempt-change standard deviation.

These cutoffs are exploratory only. No interactions are removed from the dataset.

In [74]:
ordered_concept_attempts = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), attempt_count=("attempt_count", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
print("Attempt-change sensitivity analysis:")
for cutoff in [None, 10, 5, 3]:
    sensitivity_records = []
    for _, group in ordered_concept_attempts.groupby(["user_id", "skill_id"], sort=False):
        if len(group) < 6:
            continue
        last_six = group.tail(6)
        if cutoff is not None and (last_six["attempt_count"] > cutoff).any():
            continue
        previous = last_six.iloc[:3]
        recent = last_six.iloc[3:]
        sensitivity_records.append({
            "accuracy_change": recent["correct"].mean() - previous["correct"].mean(),
            "attempt_change": recent["attempt_count"].mean() - previous["attempt_count"].mean()
        })
    sensitivity_df = pd.DataFrame(sensitivity_records)
    correlation = sensitivity_df[["accuracy_change", "attempt_change"]].corr().iloc[0, 1]
    label = "No cutoff" if cutoff is None else f"All last-6 attempts <= {cutoff}"
    accuracy_up_attempts_down = (sensitivity_df["accuracy_change"] > 0) & (sensitivity_df["attempt_change"] < 0)
    accuracy_down_attempts_up = (sensitivity_df["accuracy_change"] < 0) & (sensitivity_df["attempt_change"] > 0)
    print(f"\n--- {label} ---")
    print("Eligible histories:", len(sensitivity_df))
    print("Correlation:", round(correlation, 3))
    print("Accuracy up + attempts down:", accuracy_up_attempts_down.sum(), f"({accuracy_up_attempts_down.mean() * 100:.2f}%)")
    print("Accuracy down + attempts up:", accuracy_down_attempts_up.sum(), f"({accuracy_down_attempts_up.mean() * 100:.2f}%)")
    print("Attempt-change std:", round(sensitivity_df["attempt_change"].std(), 3))

Attempt-change sensitivity analysis:

--- No cutoff ---
Eligible histories: 18928
Correlation: -0.073
Accuracy up + attempts down: 6022 (31.82%)
Accuracy down + attempts up: 1163 (6.14%)
Attempt-change std: 9.772

--- All last-6 attempts <= 10 ---
Eligible histories: 18429
Correlation: -0.491
Accuracy up + attempts down: 5822 (31.59%)
Accuracy down + attempts up: 1077 (5.84%)
Attempt-change std: 0.698

--- All last-6 attempts <= 5 ---
Eligible histories: 17574
Correlation: -0.517
Accuracy up + attempts down: 5393 (30.69%)
Accuracy down + attempts up: 949 (5.40%)
Attempt-change std: 0.512

--- All last-6 attempts <= 3 ---
Eligible histories: 15752
Correlation: -0.472
Accuracy up + attempts down: 4359 (27.67%)
Accuracy down + attempts up: 712 (4.52%)
Attempt-change std: 0.343


### Interpretation

This sensitivity analysis tests whether extreme attempt counts distort the relationship between attempt behaviour and correctness change.

If correlation becomes substantially stronger after restricting extreme values, raw mean attempt change is likely being affected by outliers.

This would support using a robust attempt feature later, such as capped attempt counts, median attempts, multi-attempt rate, or an indicator for unusually high attempts.

However, these exploratory cutoffs are not preprocessing decisions. The original data remains unchanged.

A final attempt representation will be selected only after the dataset-understanding stage is complete.

## 85. Relationship Between Concept Accuracy Change and Response-Time Change

Response time may provide additional information about changes in student performance.

Increasing correctness together with faster responses may support evidence that a student is becoming more fluent with a concept.

Earlier analysis identified zero, negative, and extremely long response times. Therefore, this exploratory analysis uses only histories where all latest six concept interactions satisfy `0 < ms_first_response <= 300000` (five minutes).

This restriction studies the response-time signal and is not a final preprocessing rule.

The six interactions are divided into previous 3 and recent 3. We calculate accuracy change and the difference between their median response times. A negative response-time change means recent responses were faster.

In [75]:
ordered_concept_response = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), response_ms=("ms_first_response", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
response_change_records = []
excluded_histories = 0
for (user_id, skill_id), group in ordered_concept_response.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 6:
        continue
    last_six = group.tail(6)
    invalid_response = (last_six["response_ms"] <= 0) | (last_six["response_ms"] > 300000)
    if invalid_response.any():
        excluded_histories += 1
        continue
    previous = last_six.iloc[:3]
    recent = last_six.iloc[3:]
    previous_accuracy = previous["correct"].mean()
    recent_accuracy = recent["correct"].mean()
    previous_response = previous["response_ms"].median()
    recent_response = recent["response_ms"].median()
    response_change_records.append({"user_id": user_id, "skill_id": skill_id, "accuracy_change": recent_accuracy - previous_accuracy, "response_change_ms": recent_response - previous_response, "previous_median_response_ms": previous_response, "recent_median_response_ms": recent_response})
response_change_analysis = pd.DataFrame(response_change_records)
print("Eligible histories before response-time restriction:", 18928)
print("Histories excluded by response-time restriction:", excluded_histories)
print("Histories analysed:", len(response_change_analysis))
accuracy_up = response_change_analysis["accuracy_change"] > 0
accuracy_down = response_change_analysis["accuracy_change"] < 0
response_faster = response_change_analysis["response_change_ms"] < 0
response_slower = response_change_analysis["response_change_ms"] > 0
response_same = response_change_analysis["response_change_ms"] == 0
conditions = [("Accuracy up + response faster", accuracy_up & response_faster), ("Accuracy up + response same", accuracy_up & response_same), ("Accuracy up + response slower", accuracy_up & response_slower), ("Accuracy down + response slower", accuracy_down & response_slower), ("Accuracy down + response same", accuracy_down & response_same), ("Accuracy down + response faster", accuracy_down & response_faster)]
print()
for label, condition in conditions:
    print(f"{label}:", condition.sum(), f"({condition.mean() * 100:.2f}%)")
print("\nResponse-change summary (ms):")
print(response_change_analysis["response_change_ms"].describe().round(2))
print("\nCorrelation between accuracy change and response-time change:", round(response_change_analysis[["accuracy_change", "response_change_ms"]].corr().iloc[0, 1], 3))

Eligible histories before response-time restriction: 18928
Histories excluded by response-time restriction: 1341
Histories analysed: 17587

Accuracy up + response faster: 5182 (29.46%)
Accuracy up + response same: 4 (0.02%)
Accuracy up + response slower: 3635 (20.67%)
Accuracy down + response slower: 983 (5.59%)
Accuracy down + response same: 0 (0.00%)
Accuracy down + response faster: 934 (5.31%)

Response-change summary (ms):
count     17587.00
mean      -1820.09
std       29504.78
min     -232828.00
25%      -10724.00
50%       -1311.00
75%        6682.00
max      226800.00
Name: response_change_ms, dtype: float64

Correlation between accuracy change and response-time change: -0.042


### Interpretation

Response-time change may provide evidence about changes in concept fluency, but it should be interpreted cautiously.

Faster responses combined with improved correctness may support an improvement signal. However, slower responses do not automatically indicate poorer learning: a problem may be harder, the student may reason carefully, or the recorded duration may include unrelated behaviour.

Response time is therefore expected to be a supporting feature rather than a primary learning-state signal.

This analysis uses a temporary five-minute restriction only to study plausible response times. No response-time values are modified or removed from the original dataset, and no final preprocessing threshold is selected.

## 86. Sensitivity of Response-Time Trends to Maximum Response Limits

The previous analysis used a temporary maximum response time of five minutes.

We now test whether the relationship between accuracy change and response-time change depends strongly on that limit.

The latest six interactions of each student-skill history are examined under maximum response-time limits of 1, 2, 5, and 10 minutes. All six response times must be positive and no greater than the selected maximum.

For each limit, we calculate eligible histories, correlation, accuracy increase with faster responses, accuracy decrease with slower responses, and response-change standard deviation.

These limits are exploratory only. No response-time preprocessing threshold is selected.

In [76]:
ordered_concept_response = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), response_ms=("ms_first_response", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
response_limits = [60000, 120000, 300000, 600000]
print("Response-time sensitivity analysis:")
for response_limit in response_limits:
    sensitivity_records = []
    for _, group in ordered_concept_response.groupby(["user_id", "skill_id"], sort=False):
        if len(group) < 6:
            continue
        last_six = group.tail(6)
        invalid_response = (last_six["response_ms"] <= 0) | (last_six["response_ms"] > response_limit)
        if invalid_response.any():
            continue
        previous = last_six.iloc[:3]
        recent = last_six.iloc[3:]
        sensitivity_records.append({"accuracy_change": recent["correct"].mean() - previous["correct"].mean(), "response_change_ms": recent["response_ms"].median() - previous["response_ms"].median()})
    sensitivity_df = pd.DataFrame(sensitivity_records)
    correlation = sensitivity_df[["accuracy_change", "response_change_ms"]].corr().iloc[0, 1]
    accuracy_up_response_faster = (sensitivity_df["accuracy_change"] > 0) & (sensitivity_df["response_change_ms"] < 0)
    accuracy_down_response_slower = (sensitivity_df["accuracy_change"] < 0) & (sensitivity_df["response_change_ms"] > 0)
    minutes = response_limit / 60000
    print(f"\n--- Maximum response {minutes:.0f} minute(s) ---")
    print("Eligible histories:", len(sensitivity_df))
    print("Correlation:", round(correlation, 3))
    print("Accuracy up + response faster:", accuracy_up_response_faster.sum(), f"({accuracy_up_response_faster.mean() * 100:.2f}%)")
    print("Accuracy down + response slower:", accuracy_down_response_slower.sum(), f"({accuracy_down_response_slower.mean() * 100:.2f}%)")
    print("Response-change std:", round(sensitivity_df["response_change_ms"].std(), 2))

Response-time sensitivity analysis:

--- Maximum response 1 minute(s) ---
Eligible histories: 9112
Correlation: -0.046
Accuracy up + response faster: 2664 (29.24%)
Accuracy down + response slower: 391 (4.29%)
Response-change std: 10466.65

--- Maximum response 2 minute(s) ---
Eligible histories: 14023
Correlation: -0.057
Accuracy up + response faster: 4173 (29.76%)
Accuracy down + response slower: 731 (5.21%)
Response-change std: 18050.2

--- Maximum response 5 minute(s) ---
Eligible histories: 17587
Correlation: -0.042
Accuracy up + response faster: 5182 (29.46%)
Accuracy down + response slower: 983 (5.59%)
Response-change std: 29504.78

--- Maximum response 10 minute(s) ---
Eligible histories: 18434
Correlation: -0.046
Accuracy up + response faster: 5442 (29.52%)
Accuracy down + response slower: 1044 (5.66%)
Response-change std: 37784.92


### Interpretation

This sensitivity analysis determines whether the observed response-time relationship depends strongly on an arbitrary maximum-time restriction.

If the relationship remains weak across plausible limits, response time should probably be treated as a secondary contextual feature rather than a major determinant of learning state.

The analysis does not establish that fast responses are always better. Response speed may depend on problem difficulty, reading requirements, careful reasoning, or external behaviour.

Response time should therefore be interpreted together with correctness and other behavioural evidence.

No original response-time values are changed and no final threshold is selected.

## 87. Combined Hint and Attempt Evidence for Concept Performance Change

Previous analyses showed that both hint usage and attempt behaviour can provide supporting evidence for changes in concept accuracy.

For every student-skill history containing at least six interactions, the latest six are divided into previous 3 and recent 3. Accuracy, average hint-count, and average attempt-count changes are calculated.

Because extreme attempts distort relationships, this exploratory analysis includes only histories where all latest six interactions have `attempt_count <= 10`.

For increasing accuracy, we examine whether hints and attempts do not increase, both strictly decrease, or at least one increases. Opposite combinations are examined for decreasing accuracy.

Percentages are calculated within the accuracy-up or accuracy-down group. This measures agreement between signals and does not define learning-state rules.

In [77]:
ordered_concept_behaviour = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), hint_count=("hint_count", "first"), attempt_count=("attempt_count", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
combined_behaviour_records = []
excluded_attempt_histories = 0
for (user_id, skill_id), group in ordered_concept_behaviour.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 6:
        continue
    last_six = group.tail(6)
    if (last_six["attempt_count"] > 10).any():
        excluded_attempt_histories += 1
        continue
    previous = last_six.iloc[:3]
    recent = last_six.iloc[3:]
    combined_behaviour_records.append({"user_id": user_id, "skill_id": skill_id, "accuracy_change": recent["correct"].mean() - previous["correct"].mean(), "hint_change": recent["hint_count"].mean() - previous["hint_count"].mean(), "attempt_change": recent["attempt_count"].mean() - previous["attempt_count"].mean()})
combined_behaviour = pd.DataFrame(combined_behaviour_records)
accuracy_up = combined_behaviour["accuracy_change"] > 0
accuracy_down = combined_behaviour["accuracy_change"] < 0
support_up = (combined_behaviour["hint_change"] <= 0) & (combined_behaviour["attempt_change"] <= 0)
strict_up = (combined_behaviour["hint_change"] < 0) & (combined_behaviour["attempt_change"] < 0)
conflict_up = (combined_behaviour["hint_change"] > 0) | (combined_behaviour["attempt_change"] > 0)
support_down = (combined_behaviour["hint_change"] >= 0) & (combined_behaviour["attempt_change"] >= 0)
strict_down = (combined_behaviour["hint_change"] > 0) & (combined_behaviour["attempt_change"] > 0)
conflict_down = (combined_behaviour["hint_change"] < 0) | (combined_behaviour["attempt_change"] < 0)
print("Eligible histories before attempt restriction:", 18928)
print("Excluded because an attempt > 10 occurred in last six:", excluded_attempt_histories)
print("Histories analysed:", len(combined_behaviour))
print("\nAccuracy increased:", accuracy_up.sum())
for label, condition in [("Hints <= 0 AND attempts <= 0", support_up), ("Hints < 0 AND attempts < 0", strict_up), ("At least one behaviour increased", conflict_up)]:
    count = (accuracy_up & condition).sum()
    print(f"  {label}:", count, f"({count / accuracy_up.sum() * 100:.2f}% of accuracy-up cases)")
print("\nAccuracy decreased:", accuracy_down.sum())
for label, condition in [("Hints >= 0 AND attempts >= 0", support_down), ("Hints > 0 AND attempts > 0", strict_down), ("At least one behaviour decreased", conflict_down)]:
    count = (accuracy_down & condition).sum()
    print(f"  {label}:", count, f"({count / accuracy_down.sum() * 100:.2f}% of accuracy-down cases)")

Eligible histories before attempt restriction: 18928
Excluded because an attempt > 10 occurred in last six: 499
Histories analysed: 18429

Accuracy increased: 9264
  Hints <= 0 AND attempts <= 0: 8661 (93.49% of accuracy-up cases)
  Hints < 0 AND attempts < 0: 2647 (28.57% of accuracy-up cases)
  At least one behaviour increased: 603 (6.51% of accuracy-up cases)

Accuracy decreased: 1983
  Hints >= 0 AND attempts >= 0: 1632 (82.30% of accuracy-down cases)
  Hints > 0 AND attempts > 0: 537 (27.08% of accuracy-down cases)
  At least one behaviour decreased: 351 (17.70% of accuracy-down cases)


### Interpretation

Hint and attempt changes can be examined together to determine whether behavioural evidence supports or conflicts with an observed accuracy change.

For increasing accuracy, non-increasing hints and attempts provide support, simultaneous decreases provide stronger agreement, and increases in either measure indicate mixed evidence. Opposite patterns may support decreasing accuracy.

Behavioural agreement should not automatically determine the learning state. A student may improve while using more attempts or hints, particularly when later problems are harder.

Combined hint and attempt behaviour is therefore supporting evidence alongside correctness and concept history.

No final scoring system or state rule is created in this step.

## 88. Current Concept Performance and Recent Behaviour

The learning-state component must identify a student's current state, not only whether performance is changing.

We examine the latest three interactions for every student-skill history containing at least three interactions. Current recent performance is represented by 0, 1, 2, or 3 correct responses out of 3.

For the same interactions, we calculate average hint count, hint-usage rate, average and median attempt count, multi-attempt rate, and whether any interaction has more than 10 attempts.

Both mean and robust attempt measures are retained because extreme attempt counts can strongly distort raw averages.

This investigates how current correctness relates to recent learning behaviour. No learning-state thresholds are selected.

In [78]:
ordered_current_behaviour = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), hint_count=("hint_count", "first"), attempt_count=("attempt_count", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
current_behaviour_records = []
for (user_id, skill_id), group in ordered_current_behaviour.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 3:
        continue
    recent = group.tail(3)
    current_behaviour_records.append({"user_id": user_id, "skill_id": skill_id, "recent_correct": int(recent["correct"].sum()), "avg_hints": recent["hint_count"].mean(), "hint_usage_rate": (recent["hint_count"] > 0).mean(), "avg_attempts": recent["attempt_count"].mean(), "median_attempts": recent["attempt_count"].median(), "multi_attempt_rate": (recent["attempt_count"] > 1).mean(), "has_attempt_gt10": (recent["attempt_count"] > 10).any()})
current_behaviour = pd.DataFrame(current_behaviour_records)
print("Histories with >=3 interactions:", len(current_behaviour))
print("\nRecent correct-count distribution:")
print(current_behaviour["recent_correct"].value_counts().sort_index())
print("\nPercentages:")
print((current_behaviour["recent_correct"].value_counts(normalize=True).sort_index() * 100).round(2))
behaviour_summary = current_behaviour.groupby("recent_correct").agg(histories=("recent_correct", "size"), avg_hints=("avg_hints", "mean"), avg_hint_usage_rate=("hint_usage_rate", "mean"), raw_mean_attempts=("avg_attempts", "mean"), median_of_median_attempts=("median_attempts", "median"), avg_multi_attempt_rate=("multi_attempt_rate", "mean"), histories_with_attempt_gt10=("has_attempt_gt10", "sum"))
behaviour_summary["pct_with_attempt_gt10"] = behaviour_summary["histories_with_attempt_gt10"] / behaviour_summary["histories"] * 100
print("\nBehaviour by recent correct count:")
display(behaviour_summary.round(3))

Histories with >=3 interactions: 28215

Recent correct-count distribution:
recent_correct
0     2342
1     2569
2     5019
3    18285
Name: count, dtype: int64

Percentages:
recent_correct
0     8.30
1     9.11
2    17.79
3    64.81
Name: proportion, dtype: float64

Behaviour by recent correct count:
                histories  avg_hints  avg_hint_usage_rate  raw_mean_attempts  median_of_median_attempts  avg_multi_attempt_rate  histories_with_attempt_gt10  pct_with_attempt_gt10
recent_correct                                                                                                                                                                     
0                    2342      1.983                0.614              2.839                        1.0                   0.358                          135                  5.764
1                    2569      0.809                0.272              2.635                        1.0                   0.360                          142  

### Interpretation

Recent correctness provides direct evidence about current concept performance, while hints and attempts describe the support or effort associated with that performance.

Two students may both achieve high recent accuracy, but one may require more hints or attempts. Low recent correctness accompanied by high hint or attempt dependence may provide stronger evidence of difficulty than correctness alone.

Because attempt counts contain extreme observations, raw mean attempts should be interpreted with robust measures such as median attempts and multi-attempt rate.

This helps separate current performance level, behavioural support/effort, and—using earlier analyses—direction of change.

No final learning-state definitions are created in this step.

## 89. Agreement Between Recent 3, 5, and 10 Interaction Accuracy

The previous analysis used the latest three concept interactions to represent current concept performance. A short window is responsive but may be unstable.

We compare recent 3-, 5-, and 10-interaction accuracy for student-skill histories containing at least ten interactions.

For each history, we calculate absolute differences between recent 3 and 5, recent 3 and 10, and recent 5 and 10.

We examine correlations, exact agreement, differences of at most 0.20, and large differences of at least 0.40.

This investigates the trade-off between short-term responsiveness and longer-term stability. No final recent-performance window is selected.

In [79]:
ordered_concept_accuracy = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
recent_window_records = []
for (user_id, skill_id), group in ordered_concept_accuracy.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 10:
        continue
    accuracy_3 = group.tail(3)["correct"].mean()
    accuracy_5 = group.tail(5)["correct"].mean()
    accuracy_10 = group.tail(10)["correct"].mean()
    recent_window_records.append({"user_id": user_id, "skill_id": skill_id, "acc3": accuracy_3, "acc5": accuracy_5, "acc10": accuracy_10, "diff_3_5": abs(accuracy_3 - accuracy_5), "diff_3_10": abs(accuracy_3 - accuracy_10), "diff_5_10": abs(accuracy_5 - accuracy_10)})
recent_window_comparison = pd.DataFrame(recent_window_records)
print("Histories with >=10 interactions:", len(recent_window_comparison))
print("\nCorrelations:")
display(recent_window_comparison[["acc3", "acc5", "acc10"]].corr().round(3))
comparisons = [("diff_3_5", "Recent 3 vs recent 5"), ("diff_3_10", "Recent 3 vs recent 10"), ("diff_5_10", "Recent 5 vs recent 10")]
for column, label in comparisons:
    difference = recent_window_comparison[column]
    print(f"\n--- {label} ---")
    print("Mean absolute difference:", round(difference.mean(), 3))
    print("Median absolute difference:", round(difference.median(), 3))
    print("Exactly same:", (difference == 0).sum(), f"({(difference == 0).mean() * 100:.2f}%)")
    print("Difference <= 0.20:", (difference <= 0.20).sum(), f"({(difference <= 0.20).mean() * 100:.2f}%)")
    print("Difference >= 0.40:", (difference >= 0.40).sum(), f"({(difference >= 0.40).mean() * 100:.2f}%)")

Histories with >=10 interactions: 11389

Correlations:
        acc3   acc5  acc10
acc3   1.000  0.880  0.757
acc5   0.880  1.000  0.868
acc10  0.757  0.868  1.000

--- Recent 3 vs recent 5 ---
Mean absolute difference: 0.101
Median absolute difference: 0.067
Exactly same: 5459 (47.93%)
Difference <= 0.20: 9960 (87.45%)
Difference >= 0.40: 802 (7.04%)

--- Recent 3 vs recent 10 ---
Mean absolute difference: 0.184
Median absolute difference: 0.167
Exactly same: 1694 (14.87%)
Difference <= 0.20: 7910 (69.45%)
Difference >= 0.40: 1515 (13.30%)

--- Recent 5 vs recent 10 ---
Mean absolute difference: 0.118
Median absolute difference: 0.1
Exactly same: 3368 (29.57%)
Difference <= 0.20: 9470 (83.15%)
Difference >= 0.40: 337 (2.96%)


### Interpretation

Different recent-performance windows balance responsiveness and stability differently.

A three-interaction window reacts quickly, but a small number of answers can change it substantially. Five- and ten-interaction windows contain more evidence and may provide more stable estimates.

High agreement indicates a short window often represents broader recent behaviour, while large disagreement suggests one fixed window may lose important information.

The final learning-state engine may benefit from retaining both a short-term recent performance measure and a broader recent performance measure.

The appropriate design will be selected only after exploratory analysis is complete.

## 90. Performance-Band Agreement Between Recent 3 and Recent 5 Interactions

The previous analysis showed strong overall agreement between recent 3- and 5-interaction accuracy, but correlation alone does not show how often they describe current performance differently.

For histories containing at least five interactions, both measures are grouped into exploratory bands: Low (accuracy <= 1/3), Medium (> 1/3 and <= 2/3), and High (> 2/3).

These are used only to examine agreement and are not final learning-state definitions.

We compare recent 3 band to recent 5 band, measuring same-band cases, different-band cases, and disagreement directions.

In [80]:
ordered_concept_accuracy = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
band_agreement_records = []
for (user_id, skill_id), group in ordered_concept_accuracy.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 5:
        continue
    band_agreement_records.append({"user_id": user_id, "skill_id": skill_id, "history_length": len(group), "acc3": group.tail(3)["correct"].mean(), "acc5": group.tail(5)["correct"].mean()})
performance_band_agreement = pd.DataFrame(band_agreement_records)
def exploratory_performance_band(value):
    if value <= (1 / 3):
        return "Low"
    if value <= (2 / 3):
        return "Medium"
    return "High"
performance_band_agreement["band3"] = performance_band_agreement["acc3"].map(exploratory_performance_band)
performance_band_agreement["band5"] = performance_band_agreement["acc5"].map(exploratory_performance_band)
band_order = ["Low", "Medium", "High"]
print("Histories with >=5 interactions:", len(performance_band_agreement))
print("\nRecent 3 performance bands:")
print(performance_band_agreement["band3"].value_counts().reindex(band_order))
print("\nRecent 5 performance bands:")
print(performance_band_agreement["band5"].value_counts().reindex(band_order))
print("\nRecent 3 band -> Recent 5 band:")
display(pd.crosstab(performance_band_agreement["band3"], performance_band_agreement["band5"]).reindex(index=band_order, columns=band_order, fill_value=0))
same_band = performance_band_agreement["band3"] == performance_band_agreement["band5"]
print("\nSame performance band:", same_band.sum(), f"({same_band.mean() * 100:.2f}%)")
print("Different performance band:", (~same_band).sum(), f"({(~same_band).mean() * 100:.2f}%)")
print("\nDisagreement directions:")
print(performance_band_agreement.loc[~same_band].groupby(["band3", "band5"], observed=True).size().sort_values(ascending=False))

Histories with >=5 interactions: 22133

Recent 3 performance bands:
band3
Low        3644
Medium     3802
High      14687
Name: count, dtype: int64

Recent 5 performance bands:
band5
Low        2280
Medium     5431
High      14422
Name: count, dtype: int64

Recent 3 band -> Recent 5 band:
band5    Low  Medium   High
band3                      
Low     2280    1364      0
Medium     0    2380   1422
High       0    1687  13000

Same performance band: 17660 (79.79%)
Different performance band: 4473 (20.21%)

Disagreement directions:
band3   band5 
High    Medium    1687
Medium  High      1422
Low     Medium    1364
dtype: int64


### Interpretation

Performance-band agreement provides a practical view of the difference between short and broader recent windows.

Frequent agreement suggests the short window reasonably represents current performance. When bands disagree, direction matters: recent 3 High versus recent 5 Medium may indicate very recent improvement, while recent 3 Low versus recent 5 Medium may indicate very recent decline.

Disagreement may itself contain useful information rather than simply measurement noise. The final state engine may use it as context when deciding whether a student is stable or changing.

The Low, Medium, and High bands remain exploratory and are not final state thresholds.

## 91. Do Short-vs-Broader Performance Disagreements Reflect Recent Movement?

Recent 3- and recent 5-interaction performance bands disagree for approximately one-fifth of eligible student-skill histories. These disagreements may contain useful trend information.

For example, recent 3 High with recent 5 Medium may indicate very recent upward movement, while recent 3 Low with recent 5 Medium may indicate downward movement.

We examine the exact latest-five correctness pattern. For each disagreement, we calculate accuracy over the first two interactions, accuracy over the final three, and `recent shift = last 3 accuracy - first 2 accuracy`.

This determines whether disagreement is systematically associated with recent performance movement. No learning-state rule is created.

In [81]:
ordered_concept_accuracy = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
def exploratory_performance_band(value):
    if value <= (1 / 3): return "Low"
    if value <= (2 / 3): return "Medium"
    return "High"
disagreement_records = []
for (user_id, skill_id), group in ordered_concept_accuracy.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 5: continue
    last_five = group.tail(5)
    band_3 = exploratory_performance_band(last_five.tail(3)["correct"].mean())
    band_5 = exploratory_performance_band(last_five["correct"].mean())
    if band_3 == band_5: continue
    first_two_accuracy = last_five.head(2)["correct"].mean()
    last_three_accuracy = last_five.tail(3)["correct"].mean()
    disagreement_records.append({"user_id": user_id, "skill_id": skill_id, "band3": band_3, "band5": band_5, "last5_pattern": "".join(last_five["correct"].astype(int).astype(str)), "first2_accuracy": first_two_accuracy, "last3_accuracy": last_three_accuracy, "recent_shift": last_three_accuracy - first_two_accuracy})
recent_window_disagreements = pd.DataFrame(disagreement_records)
print("Disagreement histories:", len(recent_window_disagreements))
print("\nDisagreement type summary:")
display(recent_window_disagreements.groupby(["band3", "band5"]).agg(histories=("recent_shift", "size"), mean_shift=("recent_shift", "mean"), median_shift=("recent_shift", "median")).round(3))
print("\nMost common last-5 patterns by disagreement type:")
for (band3, band5), group in recent_window_disagreements.groupby(["band3", "band5"]):
    print(f"\n--- Recent 3 = {band3}, Recent 5 = {band5} ---")
    print(group["last5_pattern"].value_counts().head(10))
positive_shift = recent_window_disagreements["recent_shift"] > 0
zero_shift = recent_window_disagreements["recent_shift"] == 0
negative_shift = recent_window_disagreements["recent_shift"] < 0
print("\nPositive recent shift:", positive_shift.sum(), f"({positive_shift.mean() * 100:.2f}%)")
print("No recent shift:", zero_shift.sum(), f"({zero_shift.mean() * 100:.2f}%)")
print("Negative recent shift:", negative_shift.sum(), f"({negative_shift.mean() * 100:.2f}%)")

Disagreement histories: 4473

Disagreement type summary:
               histories  mean_shift  median_shift
band3  band5                                      
High   Medium       1687       1.000         1.000
Low    Medium       1364      -0.417        -0.167
Medium High         1422      -0.333        -0.333

Most common last-5 patterns by disagreement type:

--- Recent 3 = High, Recent 5 = Medium ---
last5_pattern
00111    1687
Name: count, dtype: int64

--- Recent 3 = Low, Recent 5 = Medium ---
last5_pattern
11001    180
11100    166
01100    150
01010    138
10001    136
01001    133
11000    128
11010    123
10010    115
10100     95
Name: count, dtype: int64

--- Recent 3 = Medium, Recent 5 = High ---
last5_pattern
11011    698
11101    396
11110    328
Name: count, dtype: int64

Positive recent shift: 1687 (37.72%)
No recent shift: 0 (0.00%)
Negative recent shift: 2786 (62.28%)


### Interpretation

Disagreement between recent 3- and recent 5-interaction performance may contain useful information about very recent movement.

When the short window is stronger, latest interactions may be pulling performance upward. When it is weaker, they may be pulling performance downward.

However, this uses very few interactions and should not independently determine an Improving or declining state. Short-vs-broader disagreement may instead support earlier 3+3 and 5+5 trend measures.

No final feature or threshold is selected in this step.

## 92. Concept History Evidence Tiers

A real learning-state engine must operate even when a student has only a small amount of history for a concept.

Different features require different minimum history: recent 3 needs at least 3 interactions, recent 5 needs 5, 3+3 trend needs 6, and 5+5 trend needs 10.

To understand cold start, student-skill histories are divided into mutually exclusive evidence tiers: 1, 2, 3–4, 5, 6–9, and 10+ interactions.

For every history, we examine length, overall concept accuracy, latest available accuracy up to 3, recent average hints, and recent median attempts. We also count histories unable to support recent-3, 3+3, and 5+5 calculations.

These tiers describe evidence availability only and are not learning-state categories.

In [82]:
ordered_concept_evidence = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), hint_count=("hint_count", "first"), attempt_count=("attempt_count", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
evidence_records = []
for (user_id, skill_id), group in ordered_concept_evidence.groupby(["user_id", "skill_id"], sort=False):
    history_length = len(group)
    if history_length == 1: evidence_tier = "1"
    elif history_length == 2: evidence_tier = "2"
    elif history_length <= 4: evidence_tier = "3-4"
    elif history_length == 5: evidence_tier = "5"
    elif history_length <= 9: evidence_tier = "6-9"
    else: evidence_tier = "10+"
    available_recent = group.tail(min(3, history_length))
    evidence_records.append({"user_id": user_id, "skill_id": skill_id, "history_length": history_length, "evidence_tier": evidence_tier, "overall_concept_accuracy": group["correct"].mean(), "recent_available_accuracy": available_recent["correct"].mean(), "recent_avg_hints": available_recent["hint_count"].mean(), "recent_median_attempts": available_recent["attempt_count"].median()})
concept_evidence = pd.DataFrame(evidence_records)
tier_order = ["1", "2", "3-4", "5", "6-9", "10+"]
tier_counts = concept_evidence["evidence_tier"].value_counts().reindex(tier_order)
print("Total student-skill histories:", len(concept_evidence))
print("\nEvidence-tier counts:")
print(tier_counts)
print("\nPercentages:")
print((tier_counts / len(concept_evidence) * 100).round(2))
print("\nBehaviour by evidence tier:")
evidence_summary = concept_evidence.groupby("evidence_tier").agg(histories=("skill_id", "size"), mean_history_length=("history_length", "mean"), mean_overall_accuracy=("overall_concept_accuracy", "mean"), mean_recent_available_accuracy=("recent_available_accuracy", "mean"), mean_recent_hints=("recent_avg_hints", "mean"), median_recent_attempts=("recent_median_attempts", "median")).reindex(tier_order)
display(evidence_summary.round(3))
less_than_3 = concept_evidence["history_length"] < 3
less_than_6 = concept_evidence["history_length"] < 6
less_than_10 = concept_evidence["history_length"] < 10
print("\nHistories unable to calculate recent-3 performance:", less_than_3.sum(), f"({less_than_3.mean() * 100:.2f}%)")
print("Histories unable to calculate 3+3 trend:", less_than_6.sum(), f"({less_than_6.mean() * 100:.2f}%)")
print("Histories unable to calculate 5+5 trend:", less_than_10.sum(), f"({less_than_10.mean() * 100:.2f}%)")

Total student-skill histories: 41982

Evidence-tier counts:
evidence_tier
1       8601
2       5166
3-4     6082
5       3205
6-9     7539
10+    11389
Name: count, dtype: int64

Percentages:
evidence_tier
1      20.49
2      12.31
3-4    14.49
5       7.63
6-9    17.96
10+    27.13
Name: count, dtype: float64

Behaviour by evidence tier:
               histories  mean_history_length  mean_overall_accuracy  mean_recent_available_accuracy  mean_recent_hints  median_recent_attempts
evidence_tier                                                                                                                                  
1                   8601                1.000                  0.779                           0.779              0.284                     1.0
2                   5166                2.000                  0.770                           0.770              0.311                     1.0
3-4                 6082                3.533                  0.704               

### Interpretation

Concept-history length determines which learning-state signals can be calculated reliably.

One or two interactions provide limited current evidence. Three to four can support a short recent estimate but not a previous-versus-recent trend. Six or more support 3+3, while ten or more support both short and broader trend comparisons.

The future engine should not assume equal evidence for every student-skill pair. It may need to use the strongest signals available for the current history length, especially for newly encountered concepts.

No cold-start strategy or final minimum-history requirement is selected in this step.

## 93. Overall Student History Available at First Concept Encounter

Concept-specific learning-state estimation faces a cold-start problem when a student encounters a skill for the first time. However, the student may already have history from other concepts.

This analysis examines each student's first observed interaction with every known skill and measures how many overall student interactions occurred before it.

For every student-skill pair, `prior student interactions = overall interaction position - 1`. We examine thresholds of 5, 10, 20, and 50 prior interactions.

This investigates whether general student-history evidence may be available when concept-specific history is limited. No cold-start prediction strategy is selected.

In [83]:
base_student_interactions = (
    df.groupby("order_id", sort=False)
    .agg(user_id=("user_id", "first"), correct=("correct", "first"))
    .reset_index()
)
base_student_interactions["student_interaction_number"] = base_student_interactions.groupby("user_id", sort=False).cumcount() + 1
skill_interactions = (
    df.dropna(subset=["skill_id"])
    .groupby(["order_id", "skill_id"], sort=False)
    .agg(user_id=("user_id", "first"), opportunity_start=("opportunity", "min"))
    .reset_index()
)
first_skill_encounters = (
    skill_interactions.sort_values(["user_id", "skill_id", "opportunity_start"])
    .groupby(["user_id", "skill_id"], as_index=False).first()
)
first_skill_encounters = first_skill_encounters.merge(base_student_interactions[["order_id", "student_interaction_number"]], on="order_id", how="left")
first_skill_encounters["prior_student_interactions"] = first_skill_encounters["student_interaction_number"] - 1
print("Student-skill first encounters:", len(first_skill_encounters))
print("\nPrior overall student interactions at first skill encounter:")
print(first_skill_encounters["prior_student_interactions"].describe().round(2))
zero_prior = first_skill_encounters["prior_student_interactions"] == 0
print("\nFirst skill encounter with 0 prior student interactions:", zero_prior.sum(), f"({zero_prior.mean() * 100:.2f}%)")
for threshold in [5, 10, 20, 50]:
    condition = first_skill_encounters["prior_student_interactions"] >= threshold
    print(f"With >={threshold} prior interactions:", condition.sum(), f"({condition.mean() * 100:.2f}%)")

Student-skill first encounters: 41982

Prior overall student interactions at first skill encounter:
count    41982.00
mean       114.14
std        156.51
min          0.00
25%          7.00
50%         42.00
75%        158.00
max       1050.00
Name: prior_student_interactions, dtype: float64

First skill encounter with 0 prior student interactions: 5535 (13.18%)
With >=5 prior interactions: 33274 (79.26%)
With >=10 prior interactions: 30410 (72.44%)
With >=20 prior interactions: 26428 (62.95%)
With >=50 prior interactions: 19705 (46.94%)


### Interpretation

Concept-specific history may be unavailable when a skill is first encountered, but overall student history may already provide useful background information.

General history should not replace concept evidence because a student can be strong in one concept and weak in another. It may instead provide context during concept cold start until sufficient concept-specific history becomes available.

Because the dataset has no explicit timestamp, this interaction position is based on the observed base-interaction row sequence and should be interpreted as an exploratory ordering proxy rather than verified real-world chronology.

No final cold-start mechanism is defined in this step.

## 94. Redundancy Among Recent Performance and Behaviour Signals

Earlier analyses showed strong relationships between correctness, hint usage, and repeated attempts. Before selecting features, we quantify how strongly candidate recent signals are related.

For every student-skill history with at least three interactions, the latest three provide recent accuracy, average hints, hint-usage rate, average attempts, median attempts, and multi-attempt rate.

Correlation is calculated using all eligible histories and again after excluding histories where any latest-three attempt count exceeds 10. The second view reduces influence from extreme attempts.

This investigates redundancy only. No feature is removed or selected.

In [84]:
ordered_signal_data = (
    df.dropna(subset=["skill_id"])
    .groupby(["user_id", "skill_id", "order_id"], sort=False)
    .agg(opportunity_start=("opportunity", "min"), correct=("correct", "first"), hint_count=("hint_count", "first"), attempt_count=("attempt_count", "first"))
    .reset_index().sort_values(["user_id", "skill_id", "opportunity_start"])
)
signal_records = []
for (user_id, skill_id), group in ordered_signal_data.groupby(["user_id", "skill_id"], sort=False):
    if len(group) < 3: continue
    recent = group.tail(3)
    signal_records.append({"user_id": user_id, "skill_id": skill_id, "recent_accuracy": recent["correct"].mean(), "avg_hints": recent["hint_count"].mean(), "hint_usage_rate": (recent["hint_count"] > 0).mean(), "avg_attempts": recent["attempt_count"].mean(), "median_attempts": recent["attempt_count"].median(), "multi_attempt_rate": (recent["attempt_count"] > 1).mean(), "has_attempt_gt10": (recent["attempt_count"] > 10).any()})
recent_signals = pd.DataFrame(signal_records)
signal_columns = ["recent_accuracy", "avg_hints", "hint_usage_rate", "avg_attempts", "median_attempts", "multi_attempt_rate"]
print("Histories analysed:", len(recent_signals))
print("\nCorrelation matrix - all eligible histories:")
display(recent_signals[signal_columns].corr().round(3))
robust_recent_signals = recent_signals[~recent_signals["has_attempt_gt10"]]
print("\nHistories without any recent attempt >10:", len(robust_recent_signals))
print("\nCorrelation matrix - without recent attempt >10:")
display(robust_recent_signals[signal_columns].corr().round(3))
print("\nAbsolute correlations with recent accuracy:")
display(robust_recent_signals[signal_columns].corr()["recent_accuracy"].drop("recent_accuracy").abs().sort_values(ascending=False).round(3))

Histories analysed: 28215

Correlation matrix - all eligible histories:
                    recent_accuracy  avg_hints  hint_usage_rate  avg_attempts  median_attempts  multi_attempt_rate
recent_accuracy               1.000     -0.658           -0.710        -0.077           -0.278              -0.634
avg_hints                    -0.658      1.000            0.929         0.071            0.245               0.476
hint_usage_rate              -0.710      0.929            1.000         0.064            0.251               0.518
avg_attempts                 -0.077      0.071            0.064         1.000            0.144               0.114
median_attempts              -0.278      0.245            0.251         0.144            1.000               0.501
multi_attempt_rate           -0.634      0.476            0.518         0.114            0.501               1.000

Histories without any recent attempt >10: 27843

Correlation matrix - without recent attempt >10:
                    rece

### Interpretation

Strong correlations indicate candidate signals may contain overlapping information, as expected because incorrect responses, hint use, and repeated attempts are behaviourally connected.

Correlation does not automatically make a feature unnecessary; measures may still distinguish particular incomplete or conflicting cases.

The robust comparison shows whether attempt relationships are distorted by unusually large counts. Feature selection and weighting should occur later during feature engineering and model development.

No candidate signal is removed in this step.

## 95. Consolidated Raw-Column Inventory

The dataset contains 30 raw columns. Previous steps investigated their meaning, quality, relationships, and limitations.

This step consolidates the schema into functional groups: core interaction/identity, learning behaviour, concept information, ordering/structure, learning context, administrative/environment, and answer/template content.

For every field, missingness, unique values, and a sample are inspected. This inventory does not perform final feature selection.

In [85]:
raw_column_groups = {
    "Core interaction / identity": ["order_id", "user_id", "assignment_id", "problem_id", "assistment_id"],
    "Learning behaviour": ["correct", "attempt_count", "hint_count", "hint_total", "ms_first_response", "overlap_time", "first_action", "bottom_hint"],
    "Concept information": ["skill_id", "skill_name", "opportunity", "opportunity_original"],
    "Ordering / structure": ["sequence_id", "base_sequence_id", "position", "original"],
    "Learning context": ["tutor_mode", "answer_type", "type"],
    "Administrative / environment": ["student_class_id", "teacher_id", "school_id"],
    "Answer / template content": ["template_id", "answer_id", "answer_text"]
}
print("Total raw columns:", len(df.columns))
print("Columns assigned to inventory:", sum(len(columns) for columns in raw_column_groups.values()))
for group_name, columns in raw_column_groups.items():
    print(f"\n=== {group_name} ===")
    inventory_rows = []
    for column in columns:
        non_missing = df[column].dropna()
        sample_value = non_missing.iloc[0] if len(non_missing) > 0 else None
        inventory_rows.append({"column": column, "dtype": str(df[column].dtype), "missing": df[column].isna().sum(), "missing_pct": df[column].isna().mean() * 100, "unique": df[column].nunique(dropna=True), "sample": sample_value})
    display(pd.DataFrame(inventory_rows).round({"missing_pct": 2}))

Total raw columns: 30
Columns assigned to inventory: 30

=== Core interaction / identity ===
order_id               missing=     0 unique= 346860 sample=33022537
user_id                missing=     0 unique=   4217 sample=64525
assignment_id          missing=     0 unique=   3521 sample=277618
problem_id             missing=     0 unique=  26688 sample=51424
assistment_id          missing=     0 unique=  17725 sample=33139

=== Learning behaviour ===
correct                missing=     0 unique=      2 sample=1
attempt_count          missing=     0 unique=    200 sample=1
hint_count             missing=     0 unique=      9 sample=0
hint_total             missing=     0 unique=     10 sample=3
ms_first_response      missing=     0 unique=  87721 sample=32454
overlap_time           missing=     0 unique= 101182 sample=32454
first_action           missing=     0 unique=      3 sample=0
bottom_hint            missing=445627 unique=      2 sample=0.0

=== Concept information ===
skill_id  

### Interpretation

The inventory separates fields by role rather than assuming every column should become a model feature.

Some identify interactions, some provide behavioural evidence, some support concept history or ordering, and others describe context, administration, or answer content. A field may be operationally essential without being predictive.

Final retention, transformation, exclusion, and grouping decisions will occur during preprocessing and feature engineering.

No raw column is removed in this step.

## 96. Consolidated Data-Quality Issue Inventory

Previous exploratory steps identified data-quality characteristics that may affect preprocessing or feature engineering. This step consolidates them into one reproducible inventory.

Checks include missing concepts, unusual response and overlap times, zero and extreme attempts, structural `bottom_hint` missingness, missing answer fields, and repeated raw rows caused by interaction/skill/opportunity representation.

The purpose is not to label every unusual observation as an error. No rows are removed and no values are changed.

In [86]:
quality_checks = [
    ("Missing skill_id", df["skill_id"].isna()),
    ("Missing skill_name", df["skill_name"].isna()),
    ("Both skill fields missing", df["skill_id"].isna() & df["skill_name"].isna()),
    ("skill_id present but skill_name missing", df["skill_id"].notna() & df["skill_name"].isna()),
    ("Negative ms_first_response", df["ms_first_response"] < 0),
    ("Zero ms_first_response", df["ms_first_response"] == 0),
    ("ms_first_response > 5 minutes", df["ms_first_response"] > 300000),
    ("ms_first_response > 1 hour", df["ms_first_response"] > 3600000),
    ("attempt_count = 0", df["attempt_count"] == 0),
    ("attempt_count > 10", df["attempt_count"] > 10),
    ("attempt_count > 100", df["attempt_count"] > 100),
    ("Negative overlap_time", df["overlap_time"] < 0),
    ("Zero overlap_time", df["overlap_time"] == 0),
    ("Missing bottom_hint", df["bottom_hint"].isna()),
    ("Missing opportunity_original", df["opportunity_original"].isna()),
    ("Missing answer_id", df["answer_id"].isna()),
    ("Missing answer_text", df["answer_text"].isna())
]
quality_inventory = pd.DataFrame([{"issue": issue, "rows": int(condition.sum()), "percentage": condition.mean() * 100} for issue, condition in quality_checks])
print("Raw rows:", len(df))
display(quality_inventory.round({"percentage": 3}))
print("\nExact duplicate raw rows:", df.duplicated().sum())
print("Unique order_ids:", df["order_id"].nunique())
print("Repeated raw rows beyond one row per order_id:", len(df) - df["order_id"].nunique())

Raw rows: 525534

                                  issue   rows  percentage
                       Missing skill_id  66326      12.621
                     Missing skill_name  78690      14.973
              Both skill fields missing  66326      12.621
skill_id present but skill_name missing  12364       2.353
             Negative ms_first_response      8       0.002
                 Zero ms_first_response   4678       0.890
          ms_first_response > 5 minutes   6544       1.245
             ms_first_response > 1 hour    200       0.038
                      attempt_count = 0  22153       4.215
                     attempt_count > 10   2807       0.534
                    attempt_count > 100    148       0.028
                  Negative overlap_time     11       0.002
                      Zero overlap_time   2982       0.567
                    Missing bottom_hint 445627      84.795
           Missing opportunity_original  76314      14.521
                      Missing answer_i

### Interpretation

The dataset contains different kinds of quality considerations that should not all be handled identically.

These include concept missingness, structural missingness, potentially invalid times, special zero values requiring interpretation, extreme values that distort averages, representation effects, and incomplete answer fields.

Preprocessing should use issue-specific handling rather than a generic rule such as dropping every row with missing or unusual values.

No preprocessing decision is made in this step.

## 97. Consolidated Interaction Representation

The exploratory analysis identified three representation levels: raw rows, base interactions identified by `order_id`, and concept interactions identified by `order_id + skill_id`.

A raw row is not automatically an independent interaction. Important interaction fields remain stable within `order_id`, while one base interaction may contain multiple skills. Some concept interactions also span multiple raw opportunity rows.

This step consolidates counts and consistency checks. No deduplication or transformation is performed.

In [87]:
raw_row_count = len(df)
base_interaction_count = df["order_id"].nunique()
known_skill_rows = df.dropna(subset=["skill_id"])
concept_interactions = known_skill_rows[["order_id", "skill_id"]].drop_duplicates()
known_skill_base_count = known_skill_rows["order_id"].nunique()
base_skill_status = df.groupby("order_id")["skill_id"].apply(lambda values: values.isna().all())
no_known_skill_base_count = base_skill_status.sum()
skills_per_base_interaction = concept_interactions.groupby("order_id")["skill_id"].nunique()
raw_rows_per_concept_interaction = known_skill_rows.groupby(["order_id", "skill_id"]).size()
print("Raw rows:", raw_row_count)
print("Base interactions (unique order_id):", base_interaction_count)
print("Base interactions with >=1 known skill:", known_skill_base_count)
print("Base interactions with no known skill:", no_known_skill_base_count)
print("Known-skill concept interactions:", len(concept_interactions))
print("\nKnown-skill base interactions by number of skills:")
print(skills_per_base_interaction.value_counts().sort_index())
multi_skill_base = skills_per_base_interaction > 1
print("\nBase interactions with >1 known skill:", multi_skill_base.sum(), f"({multi_skill_base.mean() * 100:.2f}%)")
print("\nConcept interactions represented by exactly 1 raw row:", (raw_rows_per_concept_interaction == 1).sum())
print("Concept interactions represented by >1 raw row:", (raw_rows_per_concept_interaction > 1).sum())
print("Maximum raw rows for one concept interaction:", raw_rows_per_concept_interaction.max())
stable_fields = ["user_id", "assignment_id", "problem_id", "correct", "attempt_count"]
stability_results = {column: (df.groupby("order_id")[column].nunique(dropna=False) <= 1).all() for column in stable_fields}
print("\nStable base-interaction fields within order_id:")
display(pd.Series(stability_results))

Raw rows: 525534
Base interactions (unique order_id): 346860
Base interactions with >=1 known skill: 283105
Base interactions with no known skill: 63755
Known-skill concept interactions: 338001

Known-skill base interactions by number of skills:
skill_id
1    236068
2     40857
3      4501
4      1679
Name: count, dtype: int64

Base interactions with >1 known skill: 47037 (16.61%)

Concept interactions represented by exactly 1 raw row: 334774
Concept interactions represented by >1 raw row: 3227
Maximum raw rows for one concept interaction: 215

Stable base-interaction fields within order_id:
user_id          True
assignment_id    True
problem_id       True
correct          True
attempt_count    True
dtype: bool


### Interpretation

The raw dataset should not be treated as a simple one-row-per-interaction table. The supported layered representation is `Raw rows -> Base interactions -> Concept interactions`.

`order_id` identifies the underlying interaction because important fields remain stable, but collapsing to one row per `order_id` would lose valid multi-skill information. Concept memory therefore requires `order_id + skill_id`.

Multiple raw rows for one concept interaction may represent consecutive opportunity ranges rather than separate answers. This distinction is essential for preprocessing, feature engineering, and student-skill history construction.

No final processed dataset is created in this step.

## 98. Evidence Summary for Learning State Identification

The exploratory analysis has now established the main types of evidence available for identifying a student's learning state.

### 1. Current Correctness Performance — Primary Evidence

Correctness provides the clearest direct measure of current performance.

Recent concept performance showed a strong relationship with other learning behaviours.

For student-skill histories with at least three interactions:

- 0/3 correct histories showed high hint usage and repeated-attempt behaviour.
- 1/3 correct histories also showed substantial support requirements.
- 2/3 correct histories showed lower hint and attempt dependence.
- 3/3 correct histories showed almost no hint usage and almost always a single attempt.

Therefore, recent correctness is strongly supported as a primary learning-state signal.

---

### 2. Hint Behaviour — Strong Supporting Evidence

Hint behaviour is strongly associated with correctness.

After robust attempt filtering, recent accuracy had correlations of approximately:

- `-0.705` with hint-usage rate
- `-0.655` with average hint count

Trend analysis also showed that increasing accuracy was frequently accompanied by decreasing or unchanged hint use.

However, hint measures are highly redundant with each other:

- Average hints and hint-usage rate had correlation approximately `0.931`.

Therefore, hint behaviour contains useful supporting evidence, but multiple highly similar hint features should not automatically receive independent weight.

---

### 3. Attempt Behaviour — Useful but Requires Robust Treatment

Attempt behaviour is also related to learning difficulty.

However, extreme attempt counts strongly distort raw averages.

The relationship between recent accuracy and average attempts changed from approximately:

- `-0.077` when extreme observations were included

to:

- `-0.442` when recent interactions containing attempts greater than 10 were excluded from the exploratory comparison.

Multi-attempt rate showed a stronger relationship with recent accuracy:

- approximately `-0.618`

Attempt-based measures are also strongly correlated with each other.

Therefore, attempt behaviour is useful supporting evidence, but robust representation is required.

No final attempt cutoff or transformation has yet been selected.

---

### 4. Response Time — Weak Contextual Evidence

Response time showed only a weak direct relationship with accuracy change.

Sensitivity analysis across maximum response limits of:

- 1 minute
- 2 minutes
- 5 minutes
- 10 minutes

produced correlations close to zero, approximately between `-0.042` and `-0.057`.

Therefore, response time should not currently be considered a primary learning-state signal.

It may still provide contextual evidence when combined with stronger behavioural measures.

Invalid, zero, and extreme response-time observations will require careful preprocessing.

---

### 5. Concept-Specific Performance — Essential Evidence

Students do not perform uniformly across concepts.

For supported student-skill histories, concept accuracy could differ substantially from the student's overall accuracy.

Among student-skill pairs with at least five interactions:

- 13.33% were at least 20 percentage points below the student's overall accuracy.
- 13.81% were at least 20 percentage points above the student's overall accuracy.

Among students with supported concepts:

- 40.23% had at least one concept 20 percentage points below their overall performance.
- 34.48% had at least one concept 20 percentage points above their overall performance.

Therefore, overall student performance cannot replace concept-specific memory.

Concept-aware learning-state estimation is necessary.

---

### 6. Trend Evidence — Valuable When Sufficient History Exists

Previous-vs-recent correctness comparisons showed meaningful improvement and decline patterns.

The 3+3 comparison provides a useful balance between coverage and trend information.

However, trend direction changes depending on window size for many histories.

Among histories supporting all tested windows, only 24.20% had identical direction across 2+2, 3+3, 4+4, and 5+5 comparisons.

Therefore, trend should not be represented as an unquestioned binary signal from one arbitrary window.

Short-term and broader evidence may later be combined.

---

### 7. Short-vs-Broader Recent Performance

Recent 3 and recent 5 accuracy were strongly related:

- Correlation = `0.880`
- 87.45% differed by no more than 0.20

However, their exploratory performance bands disagreed in 20.21% of histories.

These disagreements were always between adjacent performance bands rather than direct Low-to-High or High-to-Low contradictions.

Further analysis showed that these disagreements reflect very recent performance movement.

Therefore, short and broader recent windows may provide complementary information.

---

### 8. Evidence Availability / Cold Start

Concept histories vary substantially in length.

Among 41,982 student-skill histories:

- 20.49% contain only 1 interaction.
- 12.31% contain only 2 interactions.
- 14.49% contain 3–4 interactions.
- 7.63% contain 5 interactions.
- 17.96% contain 6–9 interactions.
- 27.13% contain at least 10 interactions.

As a result:

- 32.79% cannot support recent-3 performance.
- 54.91% cannot support a 3+3 trend.
- 72.87% cannot support a 5+5 trend.

Therefore, the future learning-state engine must be evidence-aware rather than requiring the same features for every prediction.

---

### 9. General Student History During Concept Cold Start

Concept-specific history may be limited when a skill is first encountered, but general student history is often already available.

At first concept encounter:

- 79.26% had at least 5 prior overall interactions.
- 72.44% had at least 10.
- 62.95% had at least 20.
- 46.94% had at least 50.

Only 13.18% occurred with no prior overall student interaction.

Therefore, general student history may provide background evidence during concept cold start.

However, it must not replace concept-specific performance because students can differ substantially across concepts.

---

### 10. Interaction Representation

The raw dataset contains:

- 525,534 raw rows
- 346,860 base interactions identified by `order_id`
- 283,105 base interactions with at least one known skill
- 63,755 base interactions with no known skill
- 338,001 known-skill concept interactions

Among known-skill base interactions:

- 16.61% contain more than one skill.

Therefore:

`order_id`

is appropriate for identifying the base student interaction, while:

`order_id + skill_id`

is required for concept-specific interaction representation.

Multiple raw rows belonging to the same concept interaction can represent consecutive opportunity ranges rather than separate student answers.

---

### 11. Ordering Concept Histories

Raw dataset row order and `order_id` alone are not reliable general chronological ordering mechanisms.

For student-skill histories, `opportunity` provided a consistent progression.

All multi-interaction student-skill histories were non-decreasing in opportunity order.

After accounting for repeated raw rows belonging to the same concept interaction, opportunity ranges were consecutive between concept interactions.

Therefore, `opportunity` provides the strongest observed basis for reconstructing concept-specific interaction sequence in this dataset.

---

### 12. Overall Evidence Structure

The exploratory results support a future learning-state representation containing three broad evidence dimensions:

1. **Current performance**
   - Primarily recent correctness
   - Supported by hint and attempt behaviour

2. **Direction of change**
   - Previous-vs-recent concept performance
   - Potentially supported by behavioural changes

3. **Evidence availability/context**
   - Concept-history length
   - General student history
   - Missing concept information
   - Weaker contextual measures such as response time

These are evidence categories derived from exploratory analysis.

They are not yet the final feature set, scoring system, classifier inputs, or learning-state definitions.

The next development phases must determine how these signals should be cleaned, transformed, engineered, and validated before defining the final states:

- Struggling
- Needs Support
- Improving
- Stable
- Strong

## 99. Preprocessing Decision Checklist

Exploratory Data Analysis identified several issues that require explicit decisions during the preprocessing and feature-engineering phases.

No decision below is implemented during Data Understanding.

### 1. Base Interaction Construction

The raw dataset cannot be treated as one row per independent student interaction.

Preprocessing must construct a reliable base-interaction representation using:

`order_id`

Interaction-level fields such as student, assignment, problem, correctness, and attempt count were shown to remain stable within an `order_id`.

---

### 2. Multi-Skill Interaction Preservation

A base interaction may belong to multiple skills.

Among known-skill base interactions, 16.61% are associated with more than one skill.

Therefore, preprocessing must preserve valid:

`order_id + skill_id`

associations rather than keeping only one skill for each `order_id`.

---

### 3. Repeated Raw Rows Within Concept Interactions

Some `order_id + skill_id` combinations contain multiple raw rows.

Earlier analysis showed that these rows correspond to consecutive opportunity ranges.

They must not automatically be treated as repeated independent answers.

Preprocessing must construct one logical concept interaction while preserving the correct opportunity range.

---

### 4. Missing Skill Information

12.621% of raw rows have no `skill_id`.

At the base-interaction level, 63,755 interactions have no known skill.

The future system therefore needs an explicit strategy for interactions without concept information.

These interactions may still contain useful overall student evidence even though they cannot directly update a known concept history.

---

### 5. Missing Skill Names

Some interactions contain a valid `skill_id` but no `skill_name`.

Earlier analysis found that known skill-ID/name mappings are consistent, but 12 skill IDs do not have recoverable names in the dataset.

The preprocessing phase must determine whether `skill_id` should be treated as the authoritative concept identifier.

---

### 6. Concept-History Ordering

General raw row order and `order_id` are not reliable chronological ordering mechanisms.

Within student-skill histories, `opportunity` showed consistent progression.

Preprocessing must therefore preserve opportunity-based concept ordering.

For logical concept interactions represented by multiple raw rows, the opportunity start and end values should be retained or reconstructed appropriately.

---

### 7. Attempt Count

`attempt_count` contains:

- Zero values
- Extreme values
- Values above 10
- Rare values above 100
- A maximum of 3824

Extreme attempts were shown to strongly distort raw attempt-based relationships.

The preprocessing/feature-engineering phase must evaluate robust representations such as:

- Clipping/winsorization
- Log transformation
- Capped counts
- Multi-attempt indicators
- Robust summary statistics

No final treatment has yet been selected.

---

### 8. Response Time

`ms_first_response` contains:

- Negative values
- Zero values
- Very long response times
- Rare values above one hour

Response time showed only weak direct trend evidence.

The preprocessing phase must determine:

- How invalid negative values should be represented
- How zero values should be interpreted
- Whether extreme times should be capped or transformed
- Whether response time should remain only a secondary contextual feature

No final response-time threshold has been selected.

---

### 9. Hint Behaviour

Hint information is useful but several fields represent related behaviour.

`hint_count`, hint-usage rate, and `bottom_hint` should not automatically be treated as independent evidence.

`bottom_hint` is structurally missing for most non-hinted interactions.

The preprocessing phase must preserve this structural meaning rather than treating all missing values as ordinary unknown data.

---

### 10. Overlap Time

`overlap_time` contains negative, zero, and extreme observations and is often equal to `ms_first_response`.

Its incremental value beyond response time must be evaluated before it is used as a learning-state feature.

---

### 11. Zero Attempts

Approximately 4.22% of raw rows have `attempt_count = 0`.

These rows include both correct and incorrect outcomes and cannot automatically be interpreted as missing records.

Their meaning must be considered before transformation or filtering.

---

### 12. Context Variables

Fields such as:

- `answer_type`
- `tutor_mode`
- assignment/problem information

may provide context but should not automatically become predictive learning-state features.

Their usefulness and risk of dataset-specific dependence must be evaluated during feature engineering.

---

### 13. Administrative Identifiers

Fields such as:

- `student_class_id`
- `teacher_id`
- `school_id`

should not automatically be used as predictive signals.

They may introduce environment-specific patterns that do not represent the student's learning state and may reduce generalization to new classes, teachers, or schools.

---

### 14. High-Missingness Answer Fields

`answer_id` is missing for approximately 90.74% of raw rows.

`answer_text` is also incomplete and highly variable.

These fields require strong justification before inclusion in the learning-state model.

---

### 15. Constant Field

`type` contains only one observed value:

`MasterySection`

Therefore, it contains no variation within this dataset.

Whether it should simply be excluded from predictive feature engineering will be decided during preprocessing.

---

### 16. Evidence-Aware Feature Availability

Not every student-skill history supports the same features.

The future preprocessing and feature-engineering pipeline must support variable evidence availability.

Examples:

- 1–2 interactions: very limited concept evidence
- 3–4 interactions: short recent performance available
- 5 interactions: broader recent performance available
- 6–9 interactions: 3+3 trend available
- 10+ interactions: broader trend comparisons available

Missing trend features should not automatically cause the interaction to be discarded.

---

### 17. General vs Concept-Specific History

General student history may provide useful context during concept cold start.

However, overall student performance must remain separate from concept-specific performance because substantial within-student concept differences were observed.

The preprocessing pipeline should therefore support both levels of history without mixing their meanings.

---

## Preprocessing Principle

The next phase should not apply one generic cleaning rule to the entire dataset.

Instead, preprocessing decisions must be based on the meaning of each field and on the evidence established during Data Understanding.

The original raw dataset should remain unchanged.

A separate reproducible processed-data pipeline should perform all transformations.

## 100. Candidate Information Map for the Learning State Component

The Data Understanding phase has identified several categories of information that may be required by the future Student Learning State Identification component.

This map organizes the available evidence according to its role in the system.

It is not yet the final feature set.

---

### 1. Student and Interaction Identity

Purpose:

Identify the student and the underlying interaction.

Relevant raw information:

- `user_id`
- `order_id`
- `assignment_id`
- `problem_id`
- `assistment_id`

Expected future use:

- Retrieve student history
- Group interactions
- Prevent repeated raw representations from being counted as separate base interactions
- Link state predictions back to the correct student interaction

These identifiers are operational fields rather than direct measures of learning ability.

---

### 2. Concept Identity

Purpose:

Identify which mathematical concept or skill is associated with an interaction.

Relevant information:

- `skill_id`
- `skill_name`

Primary concept identifier:

`skill_id`

Reason:

Skill ID/name mappings were consistent where names were available, while some valid skill IDs have no recoverable skill name.

Expected future use:

- Concept-Based Memory
- Student-skill history
- Concept-specific performance
- Concept-specific state estimation

Interactions with no known skill must remain usable for general student memory even though they cannot update a known concept profile.

---

### 3. Base Interaction Representation

Purpose:

Represent one underlying student interaction without inflating behavioural statistics.

Logical key:

`order_id`

Observed stable fields within an `order_id` include:

- `user_id`
- `assignment_id`
- `problem_id`
- `correct`
- `attempt_count`

Expected future use:

- General Student Memory
- Overall behavioural summaries
- Overall student performance
- Student-level history counts

---

### 4. Concept Interaction Representation

Purpose:

Preserve every valid concept association attached to an interaction.

Logical key:

`order_id + skill_id`

Required concept-progression information:

- `opportunity_start`
- `opportunity_end`

Expected future use:

- Concept-Based Memory
- Concept interaction ordering
- Recent concept performance
- Concept trend estimation
- Concept evidence counts

One base interaction may update more than one concept.

---

### 5. Current Performance Evidence

Purpose:

Describe how well the student is currently performing.

Candidate evidence:

- Recent correctness
- Recent correct count
- Recent incorrect count
- Recent concept accuracy
- Broader recent concept accuracy

Current EDA evidence indicates that correctness is the strongest direct performance signal.

Possible future short and broader windows include:

- Recent 3 interactions
- Recent 5 interactions

These windows are not yet finalized.

---

### 6. Hint Behaviour

Purpose:

Describe how much instructional support the student requires.

Candidate evidence:

- `hint_count`
- Whether a hint was used
- Hint-usage rate
- Possible normalized hint usage using `hint_count / hint_total`
- `bottom_hint` as additional contextual information if its semantics are verified

Important EDA findings:

- Hint usage is strongly negatively associated with recent accuracy.
- Average hint count and hint-usage rate are highly correlated.
- Multiple similar hint measures should not automatically be treated as independent evidence.

Expected future role:

Strong supporting evidence for current difficulty and learning-state change.

---

### 7. Attempt Behaviour

Purpose:

Describe repeated effort or difficulty when answering.

Candidate evidence:

- `attempt_count`
- Multi-attempt indicator
- Multi-attempt rate
- Robust/capped attempt measure
- Extreme-attempt indicator

Important EDA findings:

- Attempt behaviour is associated with correctness.
- Extreme values strongly distort raw means.
- Multi-attempt rate is more stable than unrestricted raw mean attempts.

Expected future role:

Supporting evidence for difficulty, support need, and improvement.

A robust representation must be selected during preprocessing and feature engineering.

---

### 8. Response Behaviour

Purpose:

Provide contextual information about interaction behaviour.

Candidate information:

- `ms_first_response`
- Median positive response time
- Unusual-time indicators
- Possibly transformed/capped response-time values

Important EDA findings:

- Response time contains negative, zero, and extreme values.
- Its direct relationship with accuracy change is weak.
- Response speed should not be interpreted as ability by itself.

Expected future role:

Secondary contextual evidence rather than a primary learning-state signal.

---

### 9. Concept Performance Evidence

Purpose:

Represent the student's performance on a specific mathematical concept.

Candidate evidence:

- Concept interaction count
- Concept correct count
- Concept incorrect count
- Concept accuracy
- Recent concept accuracy
- Longer concept-history accuracy
- Concept hint behaviour
- Concept attempt behaviour

Important EDA finding:

Overall student performance cannot replace concept-specific performance because substantial within-student differences across concepts were observed.

Expected future role:

Essential input for Concept-Based Memory and concept-specific state estimation.

---

### 10. Trend / Direction-of-Change Evidence

Purpose:

Determine whether concept performance is improving, declining, or remaining relatively stable.

Candidate evidence:

- Previous vs recent accuracy difference
- Recent correct-count change
- 3+3 trend
- Broader 5+5 supporting trend
- Short-vs-broader recent-performance disagreement
- Hint-change direction
- Attempt-change direction

Important EDA findings:

- Trend direction is sensitive to window size.
- A positive accuracy difference alone is not sufficient evidence of improvement.
- Stronger trends become more convincing when behavioural signals and broader windows support the same direction.

Expected future role:

Important for distinguishing `Improving` from performance states such as `Stable` or `Strong`.

---

### 11. Evidence Strength / History Availability

Purpose:

Represent how much evidence supports a state estimate.

Candidate evidence:

- Concept history length
- Overall student history length
- Number of interactions supporting recent features
- Availability of trend features
- Whether concept-specific history is in cold start

Observed concept-history tiers:

- 1 interaction
- 2 interactions
- 3–4 interactions
- 5 interactions
- 6–9 interactions
- 10+ interactions

Expected future role:

Control prediction confidence and determine which feature groups are available.

The system should not require identical history depth for every student-skill state estimate.

---

### 12. General Student History

Purpose:

Provide background behavioural evidence beyond the current concept.

Candidate evidence:

- Overall student accuracy
- Overall recent behaviour
- Overall hint-usage pattern
- Overall multi-attempt pattern
- Number of previous student interactions

Important EDA finding:

Most first concept encounters occur after the student already has useful general interaction history.

Expected future role:

Support concept cold start.

Important limitation:

General history must not replace concept-specific evidence.

---

### 13. Interaction Context

Purpose:

Provide information about how an interaction was presented.

Possible contextual fields:

- `answer_type`
- `tutor_mode`
- Assignment/problem context

Important EDA findings:

Different answer types show different behavioural distributions.

`tutor_mode = test` is extremely rare compared with tutor interactions.

Expected future role:

Possible contextual adjustment or metadata.

These fields should not automatically determine learning state.

---

### 14. Data-Quality Indicators

Purpose:

Allow the processed system to distinguish unusual or incomplete evidence rather than silently treating all observations the same.

Possible indicators:

- Missing skill information
- Zero attempt count
- Extreme attempt count
- Invalid response time
- Zero response time
- Extreme response time
- Missing concept name
- Limited concept-history evidence

Expected future role:

Preprocessing control, feature reliability, and prediction-confidence logic.

---

### 15. Fields Likely to Have Limited Predictive Value

Based on the current dataset exploration, some fields are unlikely to be direct learning-state signals.

Examples include:

- `type` because it is constant
- Administrative identifiers such as class, teacher, and school
- Highly incomplete answer identifiers
- Raw IDs used only for linking/grouping

These fields may still be operationally useful.

Final exclusion decisions will be made later.

---

## Candidate Evidence Architecture

The future learning-state engine can conceptually organize information into:

### A. Current Performance
How well is the student performing now?

### B. Support / Effort
How much help or repeated effort is required?

### C. Trend
Is performance moving upward, downward, or remaining stable?

### D. Concept Context
Which concept is being evaluated?

### E. General Student Context
What does the broader student history indicate?

### F. Evidence Strength
How much history supports this estimate?

### G. Data Quality
Are any supporting measurements missing, unusual, or unreliable?

These categories summarize the evidence discovered during Data Understanding.

They do not yet define the final engineered features, model inputs, weights, thresholds, or learning-state rules.

## 101. Final Data Understanding Conclusion

The Data Understanding phase for the Student Learning State Identification component is complete.

The ASSISTments Skill Builder dataset was examined at raw-row, interaction, student, and concept levels to determine whether it can support the intended learning-state system.

The analysis established the following conclusions.

---

### 1. Dataset Scale

The raw dataset contains:

- 525,534 rows
- 30 columns
- 4,217 students
- 3,521 assignments
- 26,688 problems
- 123 skill IDs
- 110 available skill names

The dataset provides sufficient interaction volume for developing and evaluating a student learning-state component.

However, history depth varies substantially between students and between student-skill pairs.

---

### 2. Raw Rows Are Not Independent Student Interactions

The dataset cannot safely be interpreted as one independent student interaction per raw row.

There are:

- 525,534 raw rows
- 346,860 unique `order_id` values
- 0 exact duplicate raw rows

Important interaction-level fields remain stable within each `order_id`, including:

- Student
- Assignment
- Problem
- Correctness
- Attempt count

Therefore, `order_id` is suitable as the base interaction identifier.

---

### 3. Multi-Skill Interactions Must Be Preserved

A single base interaction can be associated with multiple skills.

Among base interactions containing known skills:

- 236,068 contain 1 skill
- 40,857 contain 2 skills
- 4,501 contain 3 skills
- 1,679 contain 4 skills

Overall, 16.61% of known-skill base interactions contain more than one skill.

Therefore, reducing the data to one row per `order_id` would destroy valid concept information.

For concept-specific processing, the logical interaction representation must preserve:

`order_id + skill_id`

---

### 4. Concept Interaction Representation Is Understood

The dataset contains:

- 283,105 base interactions with at least one known skill
- 63,755 base interactions with no known skill
- 338,001 known-skill concept interactions

Most concept interactions are represented by one raw row.

However:

- 3,227 concept interactions contain multiple raw rows.
- The maximum observed raw-row count for one concept interaction is 215.

Detailed opportunity analysis showed that these repeated rows represent consecutive opportunity ranges rather than separate independent answers.

Therefore, concept interaction construction must consolidate these representations without losing opportunity progression.

---

### 5. Concept History Can Be Reliably Ordered

Raw dataset row order is not a reliable universal chronological ordering mechanism.

`order_id` is also not consistently increasing within students or assignments.

However, within student-skill histories, `opportunity` showed consistent progression.

All multi-interaction student-skill histories were non-decreasing by opportunity.

After accounting for repeated raw rows belonging to one logical concept interaction, opportunity ranges were consecutive between interactions.

Therefore, `opportunity` is the strongest observed ordering mechanism for concept-specific histories in this dataset.

---

### 6. Correctness Is the Strongest Current-Performance Signal

Correctness provides the clearest direct evidence of current concept performance.

Recent three-interaction histories showed a strong behavioural progression:

- Students with 0/3 recent correct responses used substantially more hints and repeated attempts.
- Students with 1/3 correct still required considerable support.
- Students with 2/3 correct required much less support.
- Students with 3/3 correct used almost no hints and almost always answered in one attempt.

Therefore, recent correctness should be treated as a primary candidate signal during feature engineering.

---

### 7. Hint Behaviour Provides Strong Supporting Evidence

Hint usage is strongly related to current performance.

After robust treatment of attempt extremes, recent accuracy showed strong negative relationships with:

- Hint-usage rate
- Average hint count

However, hint measures are themselves highly correlated.

Therefore, hint behaviour is valuable, but multiple similar hint measures should not automatically be interpreted as independent evidence.

---

### 8. Attempt Behaviour Is Useful but Contains Extreme Values

Attempt behaviour contains meaningful information about difficulty and repeated effort.

However, `attempt_count` contains substantial extreme observations, including values above 10, above 100, and a maximum of 3824.

These extremes strongly distort raw attempt averages.

When recent extreme attempts were excluded during exploratory sensitivity analysis, the relationship between attempts and accuracy became substantially clearer.

Therefore, attempt behaviour should be retained as candidate evidence but represented robustly during preprocessing and feature engineering.

---

### 9. Response Time Is a Secondary Signal

`ms_first_response` contains:

- Negative values
- Zero values
- Very large values
- Rare observations exceeding one hour

Response-time sensitivity analyses using several maximum-time restrictions consistently showed only weak relationships with accuracy change.

Therefore, response time should not currently be treated as a primary indicator of learning state.

It may still provide secondary contextual evidence after appropriate preprocessing.

---

### 10. Concept-Specific Memory Is Necessary

Student performance varies meaningfully across concepts.

Among sufficiently supported student-skill pairs:

- 13.33% had concept accuracy at least 20 percentage points below overall student accuracy.
- 13.81% had concept accuracy at least 20 percentage points above overall student accuracy.

Among students with supported concepts:

- 40.23% had at least one substantially weaker concept.
- 34.48% had at least one substantially stronger concept.

Therefore, overall student performance cannot replace concept-specific learning information.

The final component must support Concept-Based Memory.

---

### 11. Trend Information Is Useful but Window-Sensitive

Several previous-vs-recent performance windows were investigated:

- 2+2
- 3+3
- 4+4
- 5+5

Trend direction was not identical across these windows.

Among histories supporting all four comparisons, only 24.20% showed the same direction across every window.

Therefore, improvement or decline should not be determined from one arbitrary difference alone.

The future system should consider trend strength, available history, and supporting behavioural evidence.

---

### 12. Recent 3 and Recent 5 Provide Complementary Information

Recent-three and recent-five accuracy were strongly related.

For histories with sufficient data:

- Correlation was 0.880.
- 87.45% differed by no more than 0.20.
- Exploratory performance bands agreed in 79.79% of histories.

The disagreement cases represented meaningful recent movement between adjacent performance levels.

Therefore, short and broader recent windows may provide complementary evidence during feature engineering.

---

### 13. Cold Start Is a Major Design Requirement

Concept-history depth varies substantially.

Among 41,982 student-skill histories:

- 20.49% contain 1 interaction
- 12.31% contain 2 interactions
- 14.49% contain 3–4 interactions
- 7.63% contain 5 interactions
- 17.96% contain 6–9 interactions
- 27.13% contain 10 or more interactions

Consequently:

- 32.79% cannot support recent-three performance.
- 54.91% cannot support a 3+3 trend.
- 72.87% cannot support a 5+5 trend.

Therefore, the learning-state engine cannot require the same history-dependent features for every prediction.

It must be evidence-aware.

---

### 14. General Student History Can Support Concept Cold Start

Although concept-specific evidence may initially be unavailable, general student history is often already present.

At the first observed interaction with a skill:

- 79.26% had at least 5 prior overall interactions.
- 72.44% had at least 10.
- 62.95% had at least 20.
- 46.94% had at least 50.

Only 13.18% had no prior overall student interaction.

Therefore, general student history may provide background evidence during concept cold start.

However, it must remain separate from concept-specific mastery evidence.

---

### 15. Important Data-Quality Issues Were Identified

Important observations include:

- 12.621% of raw rows have missing `skill_id`.
- 14.973% have missing `skill_name`.
- 2.353% have a skill ID but no skill name.
- 4.215% have zero attempts.
- 0.534% have more than 10 attempts.
- Negative and zero response times exist.
- Extreme response times exist.
- `bottom_hint` is structurally missing for most non-hinted interactions.
- `answer_id` is missing for approximately 90.74% of raw rows.
- `type` contains only one observed value.

These issues require field-specific preprocessing rather than generic row deletion.

---

### 16. The Dataset Does Not Contain Ground-Truth Learning-State Labels

The dataset provides interaction behaviour but does not directly label students as:

- Struggling
- Needs Support
- Improving
- Stable
- Strong

Therefore, these states cannot simply be learned as ordinary supervised labels from the raw dataset.

A defensible state-definition and labeling methodology must be designed later using the behavioural evidence established during Data Understanding.

The resulting labels or state rules must be validated rather than assumed to represent ground truth.

---

### 17. Final Evidence Structure

The Data Understanding phase supports organizing future learning-state evidence into:

1. Current performance
2. Support and effort behaviour
3. Direction of change
4. Concept-specific history
5. General student history
6. Evidence availability
7. Contextual behaviour
8. Data-quality/reliability information

This structure provides the foundation for preprocessing and feature engineering.

---

## Data Understanding Phase Status

**Status: COMPLETE**

No final feature selection, preprocessing transformation, state threshold, label-generation rule, classifier, or production decision has been made during this phase.

The original raw dataset remains unchanged.

The next phase will convert the findings from this exploratory analysis into a reproducible processed-data representation.

### Next Phase

**Data Preprocessing and Interaction Construction**

The immediate objectives will be:

- Build a reproducible preprocessing pipeline.
- Construct one base-interaction representation from `order_id`.
- Construct the concept-interaction representation from `order_id + skill_id`.
- Preserve multi-skill interactions correctly.
- Consolidate repeated opportunity rows.
- Handle missing concept information explicitly.
- Define safe treatment for invalid/extreme behavioural measurements.
- Preserve raw data separately from processed data.
- Add validation checks to ensure preprocessing does not accidentally change interaction meaning.

Only after this processed representation is validated will feature engineering begin.

In [ ]:
import pandas as pd
from pathlib import Path
print('='*90+'\nPHASE 6 â€” STEP 1: MODELLING DATASET PREPARATION & VALIDATION\n'+'='*90)
path=Path('data/processed/learning_state_labelled.csv');d=pd.read_csv(path,low_memory=False);predictors=['previous_interaction_count','previous_skill_interaction_count','recent_accuracy_10','recent_accuracy_change_5_vs_10','recent_attempt_log_mean_10','recent_multi_attempt_rate_10','recent_hint_usage_rate_10','recent_hint_available_rate_10','recent_response_log_mean_10','recent_response_median_ms_10','recent_skill_accuracy_5','recent_skill_accuracy_change_3_vs_previous_2','has_overall_history','has_skill_history','has_full_skill_window_5','has_recent_hint_usage_evidence'];target='learning_state'
print('\nSOURCE DATASET\n'+'='*90);print('Path:',path);print('Shape:',d.shape);print('Rows:',len(d));print('Columns:',len(d.columns));print('\nAPPROVED PREDICTORS\n'+'='*90)
for i,x in enumerate(predictors,1):print(f'{i:2d}. {x}')
print('\nNumber of predictors:',len(predictors));missing=[x for x in predictors if x not in d];print('\nPREDICTOR EXISTENCE CHECK\n'+'='*90);print('Missing approved predictors:',missing);print('All 16 predictors exist:',not missing)
states=['NEEDS_SUPPORT','DEVELOPING','STRONG'];e=d[d[target].isin(states)].copy();c=d[d[target]=='UNAVAILABLE'].copy();print('\nROW SEPARATION\n'+'='*90);print('Total rows:',len(d));print('Eligible modelling rows:',len(e));print('Cold-start rows:',len(c));print('Eligible + cold-start = total:',len(e)+len(c)==len(d));print('\nELIGIBLE TARGET DISTRIBUTION\n'+'='*90);z=e[target].value_counts();print(pd.DataFrame({'interactions':z,'percentage':(e[target].value_counts(normalize=True)*100).round(2)}));print('\nOnly 3 modelling states present:',set(e[target].unique())==set(states));print('UNAVAILABLE excluded from modelling rows:','UNAVAILABLE' not in e[target].unique())
print('\nCOLD-START VALIDATION\n'+'='*90);print('All cold-start rows labelled UNAVAILABLE:',c[target].eq('UNAVAILABLE').all());print('All cold-start rows have no overall history:',c.has_overall_history.eq(0).all());print('All cold-start rows have no skill history:',c.has_skill_history.eq(0).all());print('All cold-start rows have previous_interaction_count = 0:',c.previous_interaction_count.eq(0).all());print('All cold-start rows have previous_skill_interaction_count = 0:',c.previous_skill_interaction_count.eq(0).all())
X=e[predictors].copy();y=e[target].copy();print('\nMODELLING MATRICES\n'+'='*90);print('X shape:',X.shape);print('y shape:',y.shape);print('X rows equal y rows:',len(X)==len(y));print('X contains exactly 16 columns:',X.shape[1]==16)
forbid=['correct','attempt_count','hint_count','hint_total','ms_first_response','overlap_time','bottom_hint','first_action'];ids=['order_id','user_id','assignment_id','problem_id','skill_id','skill_name'];ctx=['primary_performance','performance_source','evidence_level','evidence_strength','trend_context','support_context'];print('\nLEAKAGE / EXCLUSION CHECKS\n'+'='*90);print('Current correct excluded:','correct' not in X);print('No direct current behavioural fields:',not any(x in X for x in forbid));print('No identifiers used as predictors:',not any(x in X for x in ids));print('No derived label-context columns used as predictors:',not any(x in X for x in ctx));print('Target not included in predictors:',target not in X)
print('\nPREDICTOR MISSINGNESS â€” ELIGIBLE ROWS\n'+'='*90);print(pd.DataFrame({'missing':X.isna().sum(),'percentage':(X.isna().mean()*100).round(2)}).sort_values(['missing','percentage'],ascending=False));print('\nEVIDENCE LEVELS IN ELIGIBLE MODELLING DATA\n'+'='*90);print(e.evidence_level.value_counts());print('\nPercentage:');print((e.evidence_level.value_counts(normalize=True)*100).round(2));print('\nSTUDENT COVERAGE\n'+'='*90);print('Unique students in eligible modelling data:',e.user_id.nunique());print('Unique students in cold-start data:',c.user_id.nunique());print('\nEligible interactions per student:');print(e.groupby('user_id').size().describe().round(2))
checks={'Source rows = 338001':len(d)==338001,'Source columns = 30':d.shape[1]==30,'Eligible rows = 333061':len(e)==333061,'Cold-start rows = 4940':len(c)==4940,'Eligible + cold-start = source':len(e)+len(c)==len(d),'All 16 approved predictors exist':not missing,'X contains 16 predictors':X.shape[1]==16,'X and y row counts match':len(X)==len(y),'Only 3 target states in modelling data':set(y.unique())==set(states),'UNAVAILABLE excluded from modelling data':'UNAVAILABLE' not in y.unique(),'Current correct excluded':'correct' not in X,'No direct current behavioural leakage':not any(x in X for x in forbid),'No identifier predictors':not any(x in X for x in ids),'No derived label-context predictors':not any(x in X for x in ctx),'Target excluded from X':target not in X,'Cold-start has no overall history':c.has_overall_history.eq(0).all(),'Cold-start has no skill history':c.has_skill_history.eq(0).all()};v=pd.DataFrame(checks.items(),columns=['validation','passed']);print('\nFINAL STEP 1 VALIDATION\n'+'='*90);print(v.to_string(index=False));print(f'\nChecks passed: {int(v.passed.sum())} / {len(v)}');print('\nPHASE 6 â€” STEP 1 RESULT:','PASS' if v.passed.all() else 'REVIEW REQUIRED');print('\n'+'='*90+'\nPHASE 6 â€” STEP 1 COMPLETE\n'+'='*90)



PHASE 6 — STEP 1: MODELLING DATASET PREPARATION & VALIDATION

SOURCE DATASET
Path: data\processed\learning_state_labelled.csv
Shape: (338001, 30)
Rows: 338001
Columns: 30

APPROVED PREDICTORS
 1. previous_interaction_count
 2. previous_skill_interaction_count
 3. recent_accuracy_10
 4. recent_accuracy_change_5_vs_10
 5. recent_attempt_log_mean_10
 6. recent_multi_attempt_rate_10
 7. recent_hint_usage_rate_10
 8. recent_hint_available_rate_10
 9. recent_response_log_mean_10
10. recent_response_median_ms_10
11. recent_skill_accuracy_5
12. recent_skill_accuracy_change_3_vs_previous_2
13. has_overall_history
14. has_skill_history
15. has_full_skill_window_5
16. has_recent_hint_usage_evidence

Number of predictors: 16

PREDICTOR EXISTENCE CHECK
Missing approved predictors: []
All 16 predictors exist: True

ROW SEPARATION
Total rows: 338001
Eligible modelling rows: 333061
Cold-start rows: 4940
Eligible + cold-start = total: True

ELIGIBLE TARGET DISTRIBUTION
                interactions  per